In [2]:
!pip install torch torchvision --quiet
!python actr_experiment.py --quick --data-root /kaggle/working/data   # sanity check first
!python actr_experiment.py --data-root /kaggle/working/data           # full run

/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `python kaggle_actr_experiment(1).py --quick --data-root /kaggle/working/data   # sanity check first'
/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `python kaggle_actr_experiment(1).py --data-root /kaggle/working/data           # full run'


In [3]:
"""
ACTR Prototype Experiment
==========================
Tests whether shallow/perceptual-feature retrieval (ACTR's hypothesis) reduces
catastrophic forgetting more than semantic-embedding retrieval, on Split-CIFAR-10.

Three conditions, everything else held identical:
    (a) random       - baseline, sample replay memory uniformly at random
    (b) semantic      - retrieve memory items whose penultimate-layer (semantic)
                        embedding is most similar to the current batch
    (c) shallow       - retrieve memory items whose early-conv-layer (perceptual)
                        embedding is most similar to the current batch  <- ACTR's bet

Primary metric: Backward Transfer (BWT). Less negative / more positive = less forgetting.

Usage:
    python actr_experiment.py                  # runs all three conditions
    python actr_experiment.py --condition shallow --epochs 3
    python actr_experiment.py --quick           # tiny subset, fast sanity check

On Kaggle:
    1. Settings -> Accelerator -> GPU (T4 x2 or P100) -- off by default, turn it on.
    2. Either upload this file and run:  !python actr_experiment.py --data-root /kaggle/working/data
       or paste the code into cells and call run_condition(...) directly.
    3. Commit the notebook (or manually save results) before the session ends --
       /kaggle/working is wiped between sessions otherwise.

Requires: torch, torchvision  (pip install torch torchvision --break-system-packages)

Performance note: memory-bank embeddings used for similarity retrieval are cached
and only refreshed periodically (start of each task + every --refresh-every batches),
rather than recomputed on every single training step. This matters once the memory
bank grows -- recomputing embeddings for the whole bank every batch scales badly.
"""

import argparse
import copy
import random
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T


# --------------------------------------------------------------------------
# Reproducibility
# --------------------------------------------------------------------------
def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


# --------------------------------------------------------------------------
# Model: small CNN with two "tap points" we can extract features from
#   - shallow_features(): output of the first conv block (texture/edge level)
#   - semantic_features(): output of the penultimate FC layer (task-level meaning)
# --------------------------------------------------------------------------
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # -> 32 x 16 x 16
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # -> 64 x 8 x 8
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # -> 128 x 4 x 4
        )
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

        # small pooled head just for the shallow-feature vector (keeps it low-dim)
        self.shallow_pool = nn.AdaptiveAvgPool2d((4, 4))  # 32 x 4 x 4 = 512-d

    def shallow_features(self, x):
        """Early perceptual features: texture/edge statistics, pre-semantic."""
        h = self.conv1(x)
        h = self.shallow_pool(h)
        return h.flatten(1)  # (B, 512)

    def semantic_features(self, x):
        """Late, task-relevant features: 'what this image means' to the model."""
        h = self.conv1(x)
        h = self.conv2(h)
        h = self.conv3(h)
        h = h.flatten(1)
        h = F.relu(self.fc1(h))
        return h  # (B, 256)

    def forward(self, x):
        h = self.semantic_features(x)
        return self.fc2(h)


# --------------------------------------------------------------------------
# Split-CIFAR-10: 5 tasks, 2 classes each
# --------------------------------------------------------------------------
def build_split_cifar10(data_root="./data", quick=False):
    transform = T.Compose([T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])
    train_full = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=transform)
    test_full = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=transform)

    task_classes = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

    def indices_for_classes(dataset, classes):
        idx = [i for i, y in enumerate(dataset.targets) if y in classes]
        if quick:
            idx = idx[:400]  # tiny subset for a fast sanity check
        return idx

    train_tasks = [Subset(train_full, indices_for_classes(train_full, c)) for c in task_classes]
    test_tasks = [Subset(test_full, indices_for_classes(test_full, c)) for c in task_classes]
    return train_tasks, test_tasks


# --------------------------------------------------------------------------
# Episodic memory bank
# --------------------------------------------------------------------------
class MemoryBank:
    """Stores raw (image, label) pairs. Retrieval strategy is decided at query time."""

    def __init__(self, per_class_capacity=40):
        self.per_class_capacity = per_class_capacity
        self.images = []   # list of tensors (C,H,W)
        self.labels = []   # list of ints
        self._by_class = defaultdict(list)  # class -> indices into self.images

        # cached embeddings for similarity retrieval, keyed by a name (e.g. "shallow"/"semantic")
        # so we don't re-encode the whole bank on every single training batch.
        self._embed_cache = {}  # name -> (M, D) normalized tensor, aligned with self.images order

    def add_task_data(self, dataset):
        by_class = defaultdict(list)
        for i in range(len(dataset)):
            _, y = dataset[i]
            by_class[y].append(i)

        for y, idxs in by_class.items():
            chosen = random.sample(idxs, min(self.per_class_capacity, len(idxs)))
            for i in chosen:
                x, y2 = dataset[i]
                pos = len(self.images)
                self.images.append(x)
                self.labels.append(y2)
                self._by_class[y2].append(pos)

    def __len__(self):
        return len(self.images)

    def random_batch(self, batch_size, device):
        if len(self) == 0:
            return None
        n = min(batch_size, len(self))
        idxs = random.sample(range(len(self)), n)
        return self._gather(idxs, device)

    def similarity_batch(self, query_features, feature_fn, batch_size, device):
        """
        Retrieve the memory items whose feature_fn(image) is most similar
        (cosine similarity) to the mean of query_features from the current batch.
        This is the ACTR-style associative trigger: incoming stimulus decides recall.

        Recomputes the whole memory bank's embeddings every call -- correct but slow
        once the bank is large. Prefer refresh_embeddings() + similarity_batch_cached()
        for training loops (see below); this version is kept for clarity/small banks.
        """
        if len(self) == 0:
            return None
        n = min(batch_size, len(self))

        with torch.no_grad():
            mem_imgs = torch.stack(self.images).to(device)
            mem_feats = feature_fn(mem_imgs)  # (M, D)
            mem_feats = F.normalize(mem_feats, dim=1)

            query = F.normalize(query_features.mean(dim=0, keepdim=True), dim=1)  # (1, D)
            sims = (mem_feats @ query.T).squeeze(1)  # (M,)
            topk = torch.topk(sims, n).indices.tolist()

        return self._gather(topk, device)

    def refresh_embeddings(self, feature_fn, name, device, chunk_size=256):
        """
        Recompute and cache embeddings for the whole memory bank under `name`
        (e.g. "shallow" or "semantic"). Call this at the start of each task and
        periodically during training -- NOT on every batch -- since that's what
        actually saves the compute.
        """
        if len(self) == 0:
            self._embed_cache[name] = None
            return

        with torch.no_grad():
            feats = []
            for start in range(0, len(self.images), chunk_size):
                chunk = torch.stack(self.images[start:start + chunk_size]).to(device)
                feats.append(feature_fn(chunk))
            feats = torch.cat(feats, dim=0)
            self._embed_cache[name] = F.normalize(feats, dim=1)

    def similarity_batch_cached(self, query_features, name, batch_size, device):
        """
        Same idea as similarity_batch(), but uses the cached embeddings from
        refresh_embeddings(name=...) instead of recomputing the whole bank.
        Only the current batch's query features are computed fresh (cheap).
        """
        mem_feats = self._embed_cache.get(name)
        if mem_feats is None or len(self) == 0:
            return None
        n = min(batch_size, len(self))

        with torch.no_grad():
            query = F.normalize(query_features.mean(dim=0, keepdim=True), dim=1)  # (1, D)
            sims = (mem_feats @ query.T).squeeze(1)  # (M,)
            topk = torch.topk(sims, n).indices.tolist()

        return self._gather(topk, device)

    def _gather(self, idxs, device):
        imgs = torch.stack([self.images[i] for i in idxs]).to(device)
        labels = torch.tensor([self.labels[i] for i in idxs]).to(device)
        return imgs, labels


# --------------------------------------------------------------------------
# Training / evaluation for one condition
# --------------------------------------------------------------------------
def evaluate(model, test_tasks, seen_task_ids, device):
    model.eval()
    accs = {}
    with torch.no_grad():
        for t in seen_task_ids:
            loader = DataLoader(test_tasks[t], batch_size=256)
            correct, total = 0, 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                pred = model(x).argmax(dim=1)
                correct += (pred == y).sum().item()
                total += y.size(0)
            accs[t] = correct / total
    model.train()
    return accs


def run_condition(condition, train_tasks, test_tasks, device, epochs=3, batch_size=64, lr=1e-3,
                   mem_batch_size=32, refresh_every=50):
    set_seed(0)
    model = SmallCNN(num_classes=10).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    memory = MemoryBank(per_class_capacity=40)

    feature_fn = {"semantic": model.semantic_features, "shallow": model.shallow_features}.get(condition)

    num_tasks = len(train_tasks)
    # R[i][j] = accuracy on task j, measured right after finishing training on task i
    R = [[None] * num_tasks for _ in range(num_tasks)]

    for t in range(num_tasks):
        loader = DataLoader(train_tasks[t], batch_size=batch_size, shuffle=True)

        # refresh the embedding cache once at the start of the task (bank is stable
        # going in; the model itself is what changes as training proceeds below)
        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        step = 0
        for epoch in range(epochs):
            for x, y in loader:
                x, y = x.to(device), y.to(device)

                # --- decide what (if anything) to replay from memory ---
                replay = None
                if condition == "random":
                    replay = memory.random_batch(mem_batch_size, device)
                elif condition in ("semantic", "shallow"):
                    with torch.no_grad():
                        q_feats = feature_fn(x)
                    replay = memory.similarity_batch_cached(q_feats, condition, mem_batch_size, device)
                else:
                    raise ValueError(f"unknown condition: {condition}")

                if replay is not None:
                    rx, ry = replay
                    x_all = torch.cat([x, rx], dim=0)
                    y_all = torch.cat([y, ry], dim=0)
                else:
                    x_all, y_all = x, y

                optimizer.zero_grad()
                logits = model(x_all)
                loss = F.cross_entropy(logits, y_all)
                loss.backward()
                optimizer.step()

                # periodically refresh the cache: the model (and therefore its
                # features) keeps changing during training, so a stale cache
                # slowly drifts from what the current model would actually retrieve
                step += 1
                if feature_fn is not None and step % refresh_every == 0:
                    memory.refresh_embeddings(feature_fn, condition, device)

        # consolidate: add this task's data into the memory bank
        memory.add_task_data(train_tasks[t])
        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        # evaluate on all tasks seen so far
        accs = evaluate(model, test_tasks, seen_task_ids=list(range(t + 1)), device=device)
        for j, a in accs.items():
            R[t][j] = a
        print(f"[{condition}] after task {t}: " + ", ".join(f"T{j}={a:.3f}" for j, a in accs.items()))

    return R


def summarize(R, num_tasks):
    """Average accuracy at the end, and Backward Transfer (BWT)."""
    final_row = R[num_tasks - 1]
    avg_acc = np.mean([v for v in final_row if v is not None])

    # BWT = average over tasks i<T of (accuracy on i right after T) - (accuracy on i right after i)
    diffs = []
    for i in range(num_tasks - 1):
        acc_after_last = R[num_tasks - 1][i]
        acc_right_after_i = R[i][i]
        if acc_after_last is not None and acc_right_after_i is not None:
            diffs.append(acc_after_last - acc_right_after_i)
    bwt = np.mean(diffs) if diffs else float("nan")

    return avg_acc, bwt


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--condition", choices=["random", "semantic", "shallow", "all"], default="all")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--quick", action="store_true", help="tiny data subset, fast sanity check")
    parser.add_argument("--data-root", type=str, default="./data",
                         help="Where CIFAR-10 downloads to. On Kaggle use /kaggle/working/data.")
    parser.add_argument("--refresh-every", type=int, default=50,
                         help="Re-encode the memory bank's cached embeddings every N training batches.")
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_tasks, test_tasks = build_split_cifar10(data_root=args.data_root, quick=args.quick)
    num_tasks = len(train_tasks)

    conditions = ["random", "semantic", "shallow"] if args.condition == "all" else [args.condition]

    results = {}
    for cond in conditions:
        print(f"\n===== Running condition: {cond} =====")
        R = run_condition(cond, train_tasks, test_tasks, device, epochs=args.epochs,
                           refresh_every=args.refresh_every)
        avg_acc, bwt = summarize(R, num_tasks)
        results[cond] = (avg_acc, bwt)

    print("\n===== Summary =====")
    print(f"{'condition':<10} {'avg_acc':>10} {'BWT':>10}   (BWT closer to 0 = less forgetting)")
    for cond, (avg_acc, bwt) in results.items():
        print(f"{cond:<10} {avg_acc:>10.3f} {bwt:>10.3f}")


if __name__ == "__main__":
    main()

usage: colab_kernel_launcher.py [-h]
                                [--condition {random,semantic,shallow,all}]
                                [--epochs EPOCHS] [--quick]
                                [--data-root DATA_ROOT]
                                [--refresh-every REFRESH_EVERY]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-144a9104-fab0-49e3-852c-871e8e637fef.json


SystemExit: 2

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
%%writefile actr_experiment.py
"""
ACTR Prototype Experiment
==========================
Tests whether shallow/perceptual-feature retrieval (ACTR's hypothesis) reduces
catastrophic forgetting more than semantic-embedding retrieval, on Split-CIFAR-10.

Three conditions, everything else held identical:
    (a) random       - baseline, sample replay memory uniformly at random
    (b) semantic      - retrieve memory items whose penultimate-layer (semantic)
                        embedding is most similar to the current batch
    (c) shallow       - retrieve memory items whose early-conv-layer (perceptual)
                        embedding is most similar to the current batch  <- ACTR's bet

Primary metric: Backward Transfer (BWT). Less negative / more positive = less forgetting.
"""

import argparse
import random
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T


def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.shallow_pool = nn.AdaptiveAvgPool2d((4, 4))

    def shallow_features(self, x):
        h = self.conv1(x)
        h = self.shallow_pool(h)
        return h.flatten(1)

    def semantic_features(self, x):
        h = self.conv1(x)
        h = self.conv2(h)
        h = self.conv3(h)
        h = h.flatten(1)
        h = F.relu(self.fc1(h))
        return h

    def forward(self, x):
        h = self.semantic_features(x)
        return self.fc2(h)


def build_split_cifar10(data_root="./data", quick=False):
    transform = T.Compose([T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])
    train_full = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=transform)
    test_full = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=transform)

    task_classes = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

    def indices_for_classes(dataset, classes):
        idx = [i for i, y in enumerate(dataset.targets) if y in classes]
        if quick:
            idx = idx[:400]
        return idx

    train_tasks = [Subset(train_full, indices_for_classes(train_full, c)) for c in task_classes]
    test_tasks = [Subset(test_full, indices_for_classes(test_full, c)) for c in task_classes]
    return train_tasks, test_tasks


class MemoryBank:
    def __init__(self, per_class_capacity=40):
        self.per_class_capacity = per_class_capacity
        self.images = []
        self.labels = []
        self._by_class = defaultdict(list)
        self._embed_cache = {}

    def add_task_data(self, dataset):
        by_class = defaultdict(list)
        for i in range(len(dataset)):
            _, y = dataset[i]
            by_class[y].append(i)

        for y, idxs in by_class.items():
            chosen = random.sample(idxs, min(self.per_class_capacity, len(idxs)))
            for i in chosen:
                x, y2 = dataset[i]
                pos = len(self.images)
                self.images.append(x)
                self.labels.append(y2)
                self._by_class[y2].append(pos)

    def __len__(self):
        return len(self.images)

    def random_batch(self, batch_size, device):
        if len(self) == 0:
            return None
        n = min(batch_size, len(self))
        idxs = random.sample(range(len(self)), n)
        return self._gather(idxs, device)

    def refresh_embeddings(self, feature_fn, name, device, chunk_size=256):
        if len(self) == 0:
            self._embed_cache[name] = None
            return
        with torch.no_grad():
            feats = []
            for start in range(0, len(self.images), chunk_size):
                chunk = torch.stack(self.images[start:start + chunk_size]).to(device)
                feats.append(feature_fn(chunk))
            feats = torch.cat(feats, dim=0)
            self._embed_cache[name] = F.normalize(feats, dim=1)

    def similarity_batch_cached(self, query_features, name, batch_size, device):
        mem_feats = self._embed_cache.get(name)
        if mem_feats is None or len(self) == 0:
            return None
        n = min(batch_size, len(self))
        with torch.no_grad():
            query = F.normalize(query_features.mean(dim=0, keepdim=True), dim=1)
            sims = (mem_feats @ query.T).squeeze(1)
            topk = torch.topk(sims, n).indices.tolist()
        return self._gather(topk, device)

    def _gather(self, idxs, device):
        imgs = torch.stack([self.images[i] for i in idxs]).to(device)
        labels = torch.tensor([self.labels[i] for i in idxs]).to(device)
        return imgs, labels


def evaluate(model, test_tasks, seen_task_ids, device):
    model.eval()
    accs = {}
    with torch.no_grad():
        for t in seen_task_ids:
            loader = DataLoader(test_tasks[t], batch_size=256)
            correct, total = 0, 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                pred = model(x).argmax(dim=1)
                correct += (pred == y).sum().item()
                total += y.size(0)
            accs[t] = correct / total
    model.train()
    return accs


def run_condition(condition, train_tasks, test_tasks, device, epochs=3, batch_size=64, lr=1e-3,
                   mem_batch_size=32, refresh_every=50):
    set_seed(0)
    model = SmallCNN(num_classes=10).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    memory = MemoryBank(per_class_capacity=40)

    feature_fn = {"semantic": model.semantic_features, "shallow": model.shallow_features}.get(condition)

    num_tasks = len(train_tasks)
    R = [[None] * num_tasks for _ in range(num_tasks)]

    for t in range(num_tasks):
        loader = DataLoader(train_tasks[t], batch_size=batch_size, shuffle=True)

        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        step = 0
        for epoch in range(epochs):
            for x, y in loader:
                x, y = x.to(device), y.to(device)

                replay = None
                if condition == "random":
                    replay = memory.random_batch(mem_batch_size, device)
                elif condition in ("semantic", "shallow"):
                    with torch.no_grad():
                        q_feats = feature_fn(x)
                    replay = memory.similarity_batch_cached(q_feats, condition, mem_batch_size, device)
                else:
                    raise ValueError(f"unknown condition: {condition}")

                if replay is not None:
                    rx, ry = replay
                    x_all = torch.cat([x, rx], dim=0)
                    y_all = torch.cat([y, ry], dim=0)
                else:
                    x_all, y_all = x, y

                optimizer.zero_grad()
                logits = model(x_all)
                loss = F.cross_entropy(logits, y_all)
                loss.backward()
                optimizer.step()

                step += 1
                if feature_fn is not None and step % refresh_every == 0:
                    memory.refresh_embeddings(feature_fn, condition, device)

        memory.add_task_data(train_tasks[t])
        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        accs = evaluate(model, test_tasks, seen_task_ids=list(range(t + 1)), device=device)
        for j, a in accs.items():
            R[t][j] = a
        print(f"[{condition}] after task {t}: " + ", ".join(f"T{j}={a:.3f}" for j, a in accs.items()))

    return R


def summarize(R, num_tasks):
    final_row = R[num_tasks - 1]
    avg_acc = np.mean([v for v in final_row if v is not None])

    diffs = []
    for i in range(num_tasks - 1):
        acc_after_last = R[num_tasks - 1][i]
        acc_right_after_i = R[i][i]
        if acc_after_last is not None and acc_right_after_i is not None:
            diffs.append(acc_after_last - acc_right_after_i)
    bwt = np.mean(diffs) if diffs else float("nan")

    return avg_acc, bwt


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--condition", choices=["random", "semantic", "shallow", "all"], default="all")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--quick", action="store_true")
    parser.add_argument("--data-root", type=str, default="./data")
    parser.add_argument("--refresh-every", type=int, default=50)
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_tasks, test_tasks = build_split_cifar10(data_root=args.data_root, quick=args.quick)
    num_tasks = len(train_tasks)

    conditions = ["random", "semantic", "shallow"] if args.condition == "all" else [args.condition]

    results = {}
    for cond in conditions:
        print(f"\n===== Running condition: {cond} =====")
        R = run_condition(cond, train_tasks, test_tasks, device, epochs=args.epochs,
                           refresh_every=args.refresh_every)
        avg_acc, bwt = summarize(R, num_tasks)
        results[cond] = (avg_acc, bwt)

    print("\n===== Summary =====")
    print(f"{'condition':<10} {'avg_acc':>10} {'BWT':>10}   (BWT closer to 0 = less forgetting)")
    for cond, (avg_acc, bwt) in results.items():
        print(f"{cond:<10} {avg_acc:>10.3f} {bwt:>10.3f}")


if __name__ == "__main__":
    main()

Writing actr_experiment.py


In [2]:
!pip install torch torchvision --quiet

In [3]:
!python actr_experiment.py --quick --data-root /kaggle/working/data

Using device: cuda
100%|████████████████████████████████████████| 170M/170M [38:42<00:00, 73.4kB/s]

===== Running condition: random =====
[random] after task 0: T0=0.505
[random] after task 1: T0=0.505, T1=0.000
[random] after task 2: T0=0.505, T1=0.000, T2=0.000
[random] after task 3: T0=0.000, T1=0.495, T2=0.000, T3=0.000
[random] after task 4: T0=0.000, T1=0.000, T2=0.000, T3=0.000, T4=0.482

===== Running condition: semantic =====
[semantic] after task 0: T0=0.505
[semantic] after task 1: T0=0.495, T1=0.000
[semantic] after task 2: T0=0.000, T1=0.495, T2=0.000
[semantic] after task 3: T0=0.000, T1=0.495, T2=0.000, T3=0.000
[semantic] after task 4: T0=0.000, T1=0.495, T2=0.000, T3=0.000, T4=0.000

===== Running condition: shallow =====
[shallow] after task 0: T0=0.505
[shallow] after task 1: T0=0.505, T1=0.000
[shallow] after task 2: T0=0.505, T1=0.000, T2=0.000
[shallow] after task 3: T0=0.000, T1=0.495, T2=0.000, T3=0.000
[shallow] after task 4: T0=0.000, T1=0.000, T2=0.000, T3=0

In [4]:
!python actr_experiment.py --data-root /kaggle/working/data

Using device: cuda

===== Running condition: random =====
[random] after task 0: T0=0.938
[random] after task 1: T0=0.346, T1=0.778
[random] after task 2: T0=0.324, T1=0.013, T2=0.813
[random] after task 3: T0=0.481, T1=0.107, T2=0.145, T3=0.795
[random] after task 4: T0=0.053, T1=0.116, T2=0.208, T3=0.299, T4=0.858

===== Running condition: semantic =====
[semantic] after task 0: T0=0.935
[semantic] after task 1: T0=0.501, T1=0.738
[semantic] after task 2: T0=0.012, T1=0.165, T2=0.658
[semantic] after task 3: T0=0.340, T1=0.126, T2=0.062, T3=0.762
[semantic] after task 4: T0=0.118, T1=0.059, T2=0.028, T3=0.064, T4=0.847

===== Running condition: shallow =====
[shallow] after task 0: T0=0.931
[shallow] after task 1: T0=0.225, T1=0.784
[shallow] after task 2: T0=0.191, T1=0.008, T2=0.771
[shallow] after task 3: T0=0.269, T1=0.005, T2=0.006, T3=0.906
[shallow] after task 4: T0=0.043, T1=0.013, T2=0.002, T3=0.009, T4=0.882

===== Summary =====
condition     avg_acc        BWT   (BWT close

In [5]:
%%writefile actr_experiment.py
import argparse
import random
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T


def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.shallow_pool = nn.AdaptiveAvgPool2d((4, 4))

    def shallow_features(self, x):
        h = self.conv1(x)
        h = self.shallow_pool(h)
        return h.flatten(1)

    def semantic_features(self, x):
        h = self.conv1(x)
        h = self.conv2(h)
        h = self.conv3(h)
        h = h.flatten(1)
        h = F.relu(self.fc1(h))
        return h

    def forward(self, x):
        h = self.semantic_features(x)
        return self.fc2(h)


def build_split_cifar10(data_root="./data", quick=False):
    transform = T.Compose([T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])
    train_full = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=transform)
    test_full = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=transform)

    task_classes = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

    def indices_for_classes(dataset, classes):
        idx = [i for i, y in enumerate(dataset.targets) if y in classes]
        if quick:
            idx = idx[:400]
        return idx

    train_tasks = [Subset(train_full, indices_for_classes(train_full, c)) for c in task_classes]
    test_tasks = [Subset(test_full, indices_for_classes(test_full, c)) for c in task_classes]
    return train_tasks, test_tasks


class MemoryBank:
    def __init__(self, per_class_capacity=40):
        self.per_class_capacity = per_class_capacity
        self.images = []
        self.labels = []
        self._by_class = defaultdict(list)
        self._embed_cache = {}

    def add_task_data(self, dataset):
        by_class = defaultdict(list)
        for i in range(len(dataset)):
            _, y = dataset[i]
            by_class[y].append(i)

        for y, idxs in by_class.items():
            chosen = random.sample(idxs, min(self.per_class_capacity, len(idxs)))
            for i in chosen:
                x, y2 = dataset[i]
                pos = len(self.images)
                self.images.append(x)
                self.labels.append(y2)
                self._by_class[y2].append(pos)

    def __len__(self):
        return len(self.images)

    def random_batch(self, batch_size, device):
        if len(self) == 0:
            return None
        n = min(batch_size, len(self))
        idxs = random.sample(range(len(self)), n)
        return self._gather(idxs, device)

    def refresh_embeddings(self, feature_fn, name, device, chunk_size=256):
        if len(self) == 0:
            self._embed_cache[name] = None
            return
        with torch.no_grad():
            feats = []
            for start in range(0, len(self.images), chunk_size):
                chunk = torch.stack(self.images[start:start + chunk_size]).to(device)
                feats.append(feature_fn(chunk))
            feats = torch.cat(feats, dim=0)
            self._embed_cache[name] = F.normalize(feats, dim=1)

    def similarity_batch_cached(self, query_features, name, batch_size, device):
        mem_feats = self._embed_cache.get(name)
        if mem_feats is None or len(self) == 0:
            return None
        n = min(batch_size, len(self))
        with torch.no_grad():
            query = F.normalize(query_features.mean(dim=0, keepdim=True), dim=1)
            sims = (mem_feats @ query.T).squeeze(1)
            topk = torch.topk(sims, n).indices.tolist()
        return self._gather(topk, device)

    def _gather(self, idxs, device):
        imgs = torch.stack([self.images[i] for i in idxs]).to(device)
        labels = torch.tensor([self.labels[i] for i in idxs]).to(device)
        return imgs, labels


def evaluate(model, test_tasks, seen_task_ids, device):
    model.eval()
    accs = {}
    with torch.no_grad():
        for t in seen_task_ids:
            loader = DataLoader(test_tasks[t], batch_size=256)
            correct, total = 0, 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                pred = model(x).argmax(dim=1)
                correct += (pred == y).sum().item()
                total += y.size(0)
            accs[t] = correct / total
    model.train()
    return accs


def run_condition(condition, train_tasks, test_tasks, device, epochs=3, batch_size=64, lr=1e-3,
                   mem_batch_size=32, refresh_every=50, seed=0, mem_capacity=40):
    set_seed(seed)
    model = SmallCNN(num_classes=10).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    memory = MemoryBank(per_class_capacity=mem_capacity)

    feature_fn = {"semantic": model.semantic_features, "shallow": model.shallow_features}.get(condition)

    num_tasks = len(train_tasks)
    R = [[None] * num_tasks for _ in range(num_tasks)]

    for t in range(num_tasks):
        loader = DataLoader(train_tasks[t], batch_size=batch_size, shuffle=True)

        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        step = 0
        for epoch in range(epochs):
            for x, y in loader:
                x, y = x.to(device), y.to(device)

                replay = None
                if condition == "random":
                    replay = memory.random_batch(mem_batch_size, device)
                elif condition in ("semantic", "shallow"):
                    with torch.no_grad():
                        q_feats = feature_fn(x)
                    replay = memory.similarity_batch_cached(q_feats, condition, mem_batch_size, device)
                else:
                    raise ValueError(f"unknown condition: {condition}")

                if replay is not None:
                    rx, ry = replay
                    x_all = torch.cat([x, rx], dim=0)
                    y_all = torch.cat([y, ry], dim=0)
                else:
                    x_all, y_all = x, y

                optimizer.zero_grad()
                logits = model(x_all)
                loss = F.cross_entropy(logits, y_all)
                loss.backward()
                optimizer.step()

                step += 1
                if feature_fn is not None and step % refresh_every == 0:
                    memory.refresh_embeddings(feature_fn, condition, device)

        memory.add_task_data(train_tasks[t])
        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        accs = evaluate(model, test_tasks, seen_task_ids=list(range(t + 1)), device=device)
        for j, a in accs.items():
            R[t][j] = a
        print(f"[{condition}] after task {t}: " + ", ".join(f"T{j}={a:.3f}" for j, a in accs.items()))

    return R


def summarize(R, num_tasks):
    final_row = R[num_tasks - 1]
    avg_acc = np.mean([v for v in final_row if v is not None])

    diffs = []
    for i in range(num_tasks - 1):
        acc_after_last = R[num_tasks - 1][i]
        acc_right_after_i = R[i][i]
        if acc_after_last is not None and acc_right_after_i is not None:
            diffs.append(acc_after_last - acc_right_after_i)
    bwt = np.mean(diffs) if diffs else float("nan")

    return avg_acc, bwt


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--condition", choices=["random", "semantic", "shallow", "all"], default="all")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--quick", action="store_true")
    parser.add_argument("--data-root", type=str, default="./data")
    parser.add_argument("--refresh-every", type=int, default=50)
    parser.add_argument("--seeds", type=int, default=5)
    parser.add_argument("--mem-capacity", type=int, default=40)
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_tasks, test_tasks = build_split_cifar10(data_root=args.data_root, quick=args.quick)
    num_tasks = len(train_tasks)

    conditions = ["random", "semantic", "shallow"] if args.condition == "all" else [args.condition]

    results = {cond: [] for cond in conditions}
    for cond in conditions:
        for seed in range(args.seeds):
            print(f"\n===== Running condition: {cond} | seed {seed} =====")
            R = run_condition(cond, train_tasks, test_tasks, device, epochs=args.epochs,
                               refresh_every=args.refresh_every, seed=seed,
                               mem_batch_size=32, mem_capacity=args.mem_capacity)
            avg_acc, bwt = summarize(R, num_tasks)
            results[cond].append((avg_acc, bwt))
            print(f"[{cond} | seed {seed}] avg_acc={avg_acc:.3f}  BWT={bwt:.3f}")

    print("\n===== Summary (mean +/- std over seeds) =====")
    print(f"{'condition':<10} {'avg_acc':>18} {'BWT':>18}   (BWT closer to 0 = less forgetting)")
    for cond in conditions:
        accs = np.array([r[0] for r in results[cond]])
        bwts = np.array([r[1] for r in results[cond]])
        print(f"{cond:<10} {accs.mean():>8.3f} +/- {accs.std():<6.3f} "
              f"{bwts.mean():>8.3f} +/- {bwts.std():<6.3f}")


if __name__ == "__main__":
    main()

Overwriting actr_experiment.py


In [6]:
!pip install torch torchvision --quiet

In [7]:
!python actr_experiment.py --data-root /kaggle/working/data --seeds 5 --epochs 5

Using device: cuda

===== Running condition: random | seed 0 =====
[random] after task 0: T0=0.959
[random] after task 1: T0=0.323, T1=0.806
[random] after task 2: T0=0.347, T1=0.022, T2=0.815
[random] after task 3: T0=0.329, T1=0.043, T2=0.029, T3=0.908
[random] after task 4: T0=0.084, T1=0.101, T2=0.130, T3=0.266, T4=0.882
[random | seed 0] avg_acc=0.292  BWT=-0.727

===== Running condition: random | seed 1 =====
[random] after task 0: T0=0.960
[random] after task 1: T0=0.383, T1=0.815
[random] after task 2: T0=0.324, T1=0.019, T2=0.861
[random] after task 3: T0=0.420, T1=0.040, T2=0.054, T3=0.944
[random] after task 4: T0=0.059, T1=0.120, T2=0.182, T3=0.233, T4=0.865
[random | seed 1] avg_acc=0.292  BWT=-0.746

===== Running condition: random | seed 2 =====
[random] after task 0: T0=0.958
[random] after task 1: T0=0.176, T1=0.823
[random] after task 2: T0=0.364, T1=0.033, T2=0.863
[random] after task 3: T0=0.405, T1=0.073, T2=0.078, T3=0.902
[random] after task 4: T0=0.073, T1=0.139

In [8]:
%%writefile actr_experiment.py
import argparse
import random
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T


def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.shallow_pool = nn.AdaptiveAvgPool2d((4, 4))

    def shallow_features(self, x):
        h = self.conv1(x)
        h = self.shallow_pool(h)
        return h.flatten(1)

    def semantic_features(self, x):
        h = self.conv1(x)
        h = self.conv2(h)
        h = self.conv3(h)
        h = h.flatten(1)
        h = F.relu(self.fc1(h))
        return h

    def forward(self, x):
        h = self.semantic_features(x)
        return self.fc2(h)


def build_split_cifar10(data_root="./data", quick=False):
    transform = T.Compose([T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])
    train_full = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=transform)
    test_full = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=transform)

    task_classes = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

    def indices_for_classes(dataset, classes):
        idx = [i for i, y in enumerate(dataset.targets) if y in classes]
        if quick:
            idx = idx[:400]
        return idx

    train_tasks = [Subset(train_full, indices_for_classes(train_full, c)) for c in task_classes]
    test_tasks = [Subset(test_full, indices_for_classes(test_full, c)) for c in task_classes]
    return train_tasks, test_tasks


class MemoryBank:
    def __init__(self, per_class_capacity=40):
        self.per_class_capacity = per_class_capacity
        self.images = []
        self.labels = []
        self._by_class = defaultdict(list)
        self._embed_cache = {}

    def add_task_data(self, dataset):
        by_class = defaultdict(list)
        for i in range(len(dataset)):
            _, y = dataset[i]
            by_class[y].append(i)

        for y, idxs in by_class.items():
            chosen = random.sample(idxs, min(self.per_class_capacity, len(idxs)))
            for i in chosen:
                x, y2 = dataset[i]
                pos = len(self.images)
                self.images.append(x)
                self.labels.append(y2)
                self._by_class[y2].append(pos)

    def __len__(self):
        return len(self.images)

    def random_batch(self, batch_size, device):
        if len(self) == 0:
            return None
        n = min(batch_size, len(self))
        idxs = random.sample(range(len(self)), n)
        return self._gather(idxs, device)

    def refresh_embeddings(self, feature_fn, name, device, chunk_size=256):
        if len(self) == 0:
            self._embed_cache[name] = None
            return
        with torch.no_grad():
            feats = []
            for start in range(0, len(self.images), chunk_size):
                chunk = torch.stack(self.images[start:start + chunk_size]).to(device)
                feats.append(feature_fn(chunk))
            feats = torch.cat(feats, dim=0)
            self._embed_cache[name] = F.normalize(feats, dim=1)

    def similarity_batch_cached(self, query_features, name, batch_size, device, temperature=0.1):
        """
        Sample WITHOUT replacement, probability proportional to
        softmax(similarity / temperature), instead of pure top-k -- pure top-k
        collapses onto the same few memory items every batch and starves out
        the rest of the bank, confounding the shallow-vs-semantic comparison.
        """
        mem_feats = self._embed_cache.get(name)
        if mem_feats is None or len(self) == 0:
            return None
        n = min(batch_size, len(self))

        with torch.no_grad():
            query = F.normalize(query_features.mean(dim=0, keepdim=True), dim=1)
            sims = (mem_feats @ query.T).squeeze(1)
            probs = F.softmax(sims / temperature, dim=0).cpu().numpy()
            probs = probs / probs.sum()
            idxs = np.random.choice(len(probs), size=n, replace=False, p=probs)

        return self._gather(idxs.tolist(), device)

    def _gather(self, idxs, device):
        imgs = torch.stack([self.images[i] for i in idxs]).to(device)
        labels = torch.tensor([self.labels[i] for i in idxs]).to(device)
        return imgs, labels


def evaluate(model, test_tasks, seen_task_ids, device):
    model.eval()
    accs = {}
    with torch.no_grad():
        for t in seen_task_ids:
            loader = DataLoader(test_tasks[t], batch_size=256)
            correct, total = 0, 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                pred = model(x).argmax(dim=1)
                correct += (pred == y).sum().item()
                total += y.size(0)
            accs[t] = correct / total
    model.train()
    return accs


def run_condition(condition, train_tasks, test_tasks, device, epochs=3, batch_size=64, lr=1e-3,
                   mem_batch_size=32, refresh_every=50, seed=0, mem_capacity=40, temperature=0.1):
    set_seed(seed)
    model = SmallCNN(num_classes=10).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    memory = MemoryBank(per_class_capacity=mem_capacity)

    feature_fn = {"semantic": model.semantic_features, "shallow": model.shallow_features}.get(condition)

    num_tasks = len(train_tasks)
    R = [[None] * num_tasks for _ in range(num_tasks)]

    for t in range(num_tasks):
        loader = DataLoader(train_tasks[t], batch_size=batch_size, shuffle=True)

        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        step = 0
        for epoch in range(epochs):
            for x, y in loader:
                x, y = x.to(device), y.to(device)

                replay = None
                if condition == "random":
                    replay = memory.random_batch(mem_batch_size, device)
                elif condition in ("semantic", "shallow"):
                    with torch.no_grad():
                        q_feats = feature_fn(x)
                    replay = memory.similarity_batch_cached(q_feats, condition, mem_batch_size, device,
                                                             temperature=temperature)
                else:
                    raise ValueError(f"unknown condition: {condition}")

                if replay is not None:
                    rx, ry = replay
                    x_all = torch.cat([x, rx], dim=0)
                    y_all = torch.cat([y, ry], dim=0)
                else:
                    x_all, y_all = x, y

                optimizer.zero_grad()
                logits = model(x_all)
                loss = F.cross_entropy(logits, y_all)
                loss.backward()
                optimizer.step()

                step += 1
                if feature_fn is not None and step % refresh_every == 0:
                    memory.refresh_embeddings(feature_fn, condition, device)

        memory.add_task_data(train_tasks[t])
        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        accs = evaluate(model, test_tasks, seen_task_ids=list(range(t + 1)), device=device)
        for j, a in accs.items():
            R[t][j] = a
        print(f"[{condition}] after task {t}: " + ", ".join(f"T{j}={a:.3f}" for j, a in accs.items()))

    return R


def summarize(R, num_tasks):
    final_row = R[num_tasks - 1]
    avg_acc = np.mean([v for v in final_row if v is not None])

    diffs = []
    for i in range(num_tasks - 1):
        acc_after_last = R[num_tasks - 1][i]
        acc_right_after_i = R[i][i]
        if acc_after_last is not None and acc_right_after_i is not None:
            diffs.append(acc_after_last - acc_right_after_i)
    bwt = np.mean(diffs) if diffs else float("nan")

    return avg_acc, bwt


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--condition", choices=["random", "semantic", "shallow", "all"], default="all")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--quick", action="store_true")
    parser.add_argument("--data-root", type=str, default="./data")
    parser.add_argument("--refresh-every", type=int, default=50)
    parser.add_argument("--seeds", type=int, default=5)
    parser.add_argument("--mem-capacity", type=int, default=40)
    parser.add_argument("--temperature", type=float, default=0.1)
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_tasks, test_tasks = build_split_cifar10(data_root=args.data_root, quick=args.quick)
    num_tasks = len(train_tasks)

    conditions = ["random", "semantic", "shallow"] if args.condition == "all" else [args.condition]

    results = {cond: [] for cond in conditions}
    for cond in conditions:
        for seed in range(args.seeds):
            print(f"\n===== Running condition: {cond} | seed {seed} =====")
            R = run_condition(cond, train_tasks, test_tasks, device, epochs=args.epochs,
                               refresh_every=args.refresh_every, seed=seed,
                               mem_batch_size=32, mem_capacity=args.mem_capacity,
                               temperature=args.temperature)
            avg_acc, bwt = summarize(R, num_tasks)
            results[cond].append((avg_acc, bwt))
            print(f"[{cond} | seed {seed}] avg_acc={avg_acc:.3f}  BWT={bwt:.3f}")

    print("\n===== Summary (mean +/- std over seeds) =====")
    print(f"{'condition':<10} {'avg_acc':>18} {'BWT':>18}   (BWT closer to 0 = less forgetting)")
    for cond in conditions:
        accs = np.array([r[0] for r in results[cond]])
        bwts = np.array([r[1] for r in results[cond]])
        print(f"{cond:<10} {accs.mean():>8.3f} +/- {accs.std():<6.3f} "
              f"{bwts.mean():>8.3f} +/- {bwts.std():<6.3f}")


if __name__ == "__main__":
    main()

Overwriting actr_experiment.py


In [9]:
!pip install torch torchvision --quiet

In [10]:
!python actr_experiment.py --data-root /kaggle/working/data --seeds 5 --epochs 5 --temperature 0.1

Using device: cuda

===== Running condition: random | seed 0 =====
[random] after task 0: T0=0.955
[random] after task 1: T0=0.320, T1=0.765
[random] after task 2: T0=0.336, T1=0.018, T2=0.826
[random] after task 3: T0=0.456, T1=0.053, T2=0.047, T3=0.905
[random] after task 4: T0=0.052, T1=0.147, T2=0.176, T3=0.271, T4=0.899
[random | seed 0] avg_acc=0.309  BWT=-0.701

===== Running condition: random | seed 1 =====
[random] after task 0: T0=0.954
[random] after task 1: T0=0.346, T1=0.825
[random] after task 2: T0=0.232, T1=0.010, T2=0.843
[random] after task 3: T0=0.450, T1=0.074, T2=0.074, T3=0.905
[random] after task 4: T0=0.048, T1=0.104, T2=0.244, T3=0.227, T4=0.893
[random | seed 1] avg_acc=0.303  BWT=-0.726

===== Running condition: random | seed 2 =====
[random] after task 0: T0=0.961
[random] after task 1: T0=0.284, T1=0.821
[random] after task 2: T0=0.339, T1=0.018, T2=0.854
[random] after task 3: T0=0.257, T1=0.045, T2=0.053, T3=0.910
[random] after task 4: T0=0.000, T1=0.017

In [2]:
%%writefile actr_experiment.py
import argparse
import random
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T


def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.shallow_pool = nn.AdaptiveAvgPool2d((4, 4))

    def shallow_features(self, x):
        h = self.conv1(x)
        h = self.shallow_pool(h)
        return h.flatten(1)

    def semantic_features(self, x):
        h = self.conv1(x)
        h = self.conv2(h)
        h = self.conv3(h)
        h = h.flatten(1)
        h = F.relu(self.fc1(h))
        return h

    def forward(self, x):
        h = self.semantic_features(x)
        return self.fc2(h)


def build_split_cifar10(data_root="./data", quick=False):
    transform = T.Compose([T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])
    train_full = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=transform)
    test_full = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=transform)

    task_classes = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

    def indices_for_classes(dataset, classes):
        idx = [i for i, y in enumerate(dataset.targets) if y in classes]
        if quick:
            idx = idx[:400]
        return idx

    train_tasks = [Subset(train_full, indices_for_classes(train_full, c)) for c in task_classes]
    test_tasks = [Subset(test_full, indices_for_classes(test_full, c)) for c in task_classes]
    return train_tasks, test_tasks


class MemoryBank:
    def __init__(self, per_class_capacity=40):
        self.per_class_capacity = per_class_capacity
        self.images = []
        self.labels = []
        self._by_class = defaultdict(list)
        self._embed_cache = {}

    def add_task_data(self, dataset):
        by_class = defaultdict(list)
        for i in range(len(dataset)):
            _, y = dataset[i]
            by_class[y].append(i)

        for y, idxs in by_class.items():
            chosen = random.sample(idxs, min(self.per_class_capacity, len(idxs)))
            for i in chosen:
                x, y2 = dataset[i]
                pos = len(self.images)
                self.images.append(x)
                self.labels.append(y2)
                self._by_class[y2].append(pos)

    def __len__(self):
        return len(self.images)

    def random_batch(self, batch_size, device):
        if len(self) == 0:
            return None
        n = min(batch_size, len(self))
        idxs = random.sample(range(len(self)), n)
        return self._gather(idxs, device)

    def refresh_embeddings(self, feature_fn, name, device, chunk_size=256):
        if len(self) == 0:
            self._embed_cache[name] = None
            return
        with torch.no_grad():
            feats = []
            for start in range(0, len(self.images), chunk_size):
                chunk = torch.stack(self.images[start:start + chunk_size]).to(device)
                feats.append(feature_fn(chunk))
            feats = torch.cat(feats, dim=0)
            self._embed_cache[name] = F.normalize(feats, dim=1)

    def similarity_batch_cached(self, query_features, name, batch_size, device, temperature=0.1):
        mem_feats = self._embed_cache.get(name)
        if mem_feats is None or len(self) == 0:
            return None
        n = min(batch_size, len(self))
        with torch.no_grad():
            query = F.normalize(query_features.mean(dim=0, keepdim=True), dim=1)
            sims = (mem_feats @ query.T).squeeze(1)
            probs = F.softmax(sims / temperature, dim=0).cpu().numpy()
            probs = probs / probs.sum()
            idxs = np.random.choice(len(probs), size=n, replace=False, p=probs)
        return self._gather(idxs.tolist(), device)

    def _gather(self, idxs, device):
        imgs = torch.stack([self.images[i] for i in idxs]).to(device)
        labels = torch.tensor([self.labels[i] for i in idxs]).to(device)
        return imgs, labels


def evaluate(model, test_tasks, seen_task_ids, device):
    model.eval()
    accs = {}
    with torch.no_grad():
        for t in seen_task_ids:
            loader = DataLoader(test_tasks[t], batch_size=256)
            correct, total = 0, 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                pred = model(x).argmax(dim=1)
                correct += (pred == y).sum().item()
                total += y.size(0)
            accs[t] = correct / total
    model.train()
    return accs


def run_condition(condition, train_tasks, test_tasks, device, epochs=3, batch_size=64, lr=1e-3,
                   mem_batch_size=32, refresh_every=50, seed=0, mem_capacity=40, temperature=0.1,
                   grad_clip=5.0):
    set_seed(seed)
    model = SmallCNN(num_classes=10).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    memory = MemoryBank(per_class_capacity=mem_capacity)

    feature_fn = {"semantic": model.semantic_features, "shallow": model.shallow_features}.get(condition)

    num_tasks = len(train_tasks)
    R = [[None] * num_tasks for _ in range(num_tasks)]

    for t in range(num_tasks):
        loader = DataLoader(train_tasks[t], batch_size=batch_size, shuffle=True)

        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        step = 0
        for epoch in range(epochs):
            for x, y in loader:
                x, y = x.to(device), y.to(device)

                replay = None
                if condition == "random":
                    replay = memory.random_batch(mem_batch_size, device)
                elif condition in ("semantic", "shallow"):
                    with torch.no_grad():
                        q_feats = feature_fn(x)
                    replay = memory.similarity_batch_cached(q_feats, condition, mem_batch_size, device,
                                                             temperature=temperature)
                else:
                    raise ValueError(f"unknown condition: {condition}")

                if replay is not None:
                    rx, ry = replay
                    x_all = torch.cat([x, rx], dim=0)
                    y_all = torch.cat([y, ry], dim=0)
                else:
                    x_all, y_all = x, y

                optimizer.zero_grad()
                logits = model(x_all)
                loss = F.cross_entropy(logits, y_all)

                if not torch.isfinite(loss):
                    print(f"  [warning] non-finite loss ({loss.item()}) at step {step} -- skipping this batch")
                    optimizer.zero_grad()
                    step += 1
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
                optimizer.step()

                step += 1
                if feature_fn is not None and step % refresh_every == 0:
                    memory.refresh_embeddings(feature_fn, condition, device)

        memory.add_task_data(train_tasks[t])
        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        accs = evaluate(model, test_tasks, seen_task_ids=list(range(t + 1)), device=device)
        for j, a in accs.items():
            R[t][j] = a
        print(f"[{condition}] after task {t}: " + ", ".join(f"T{j}={a:.3f}" for j, a in accs.items()))

    return R


def summarize(R, num_tasks):
    final_row = R[num_tasks - 1]
    avg_acc = np.mean([v for v in final_row if v is not None])

    diffs = []
    for i in range(num_tasks - 1):
        acc_after_last = R[num_tasks - 1][i]
        acc_right_after_i = R[i][i]
        if acc_after_last is not None and acc_right_after_i is not None:
            diffs.append(acc_after_last - acc_right_after_i)
    bwt = np.mean(diffs) if diffs else float("nan")

    return avg_acc, bwt


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--condition", choices=["random", "semantic", "shallow", "all"], default="all")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--quick", action="store_true")
    parser.add_argument("--data-root", type=str, default="./data")
    parser.add_argument("--refresh-every", type=int, default=50)
    parser.add_argument("--seeds", type=int, default=5)
    parser.add_argument("--mem-capacity", type=int, default=40)
    parser.add_argument("--temperature", type=float, default=0.1)
    parser.add_argument("--grad-clip", type=float, default=5.0)
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_tasks, test_tasks = build_split_cifar10(data_root=args.data_root, quick=args.quick)
    num_tasks = len(train_tasks)

    conditions = ["random", "semantic", "shallow"] if args.condition == "all" else [args.condition]

    results = {cond: [] for cond in conditions}
    for cond in conditions:
        for seed in range(args.seeds):
            print(f"\n===== Running condition: {cond} | seed {seed} =====")
            R = run_condition(cond, train_tasks, test_tasks, device, epochs=args.epochs,
                               refresh_every=args.refresh_every, seed=seed,
                               mem_batch_size=32, mem_capacity=args.mem_capacity,
                               temperature=args.temperature, grad_clip=args.grad_clip)
            avg_acc, bwt = summarize(R, num_tasks)
            results[cond].append((avg_acc, bwt))
            print(f"[{cond} | seed {seed}] avg_acc={avg_acc:.3f}  BWT={bwt:.3f}")

    print("\n===== Summary (mean +/- std over seeds) =====")
    print(f"{'condition':<10} {'avg_acc':>18} {'BWT':>18}   (BWT closer to 0 = less forgetting)")
    for cond in conditions:
        accs = np.array([r[0] for r in results[cond]])
        bwts = np.array([r[1] for r in results[cond]])
        print(f"{cond:<10} {accs.mean():>8.3f} +/- {accs.std():<6.3f} "
              f"{bwts.mean():>8.3f} +/- {bwts.std():<6.3f}")


if __name__ == "__main__":
    main()

Writing actr_experiment.py


In [3]:
!pip install torch torchvision --quiet

In [14]:
!python actr_experiment.py --data-root /kaggle/working/data --seeds 5 --epochs 5 --temperature 0.1

Using device: cuda

===== Running condition: random | seed 0 =====
[random] after task 0: T0=0.956
[random] after task 1: T0=0.522, T1=0.827
[random] after task 2: T0=0.210, T1=0.018, T2=0.901
[random] after task 3: T0=0.406, T1=0.037, T2=0.051, T3=0.956
[random] after task 4: T0=0.113, T1=0.130, T2=0.173, T3=0.372, T4=0.950
[random | seed 0] avg_acc=0.347  BWT=-0.713

===== Running condition: random | seed 1 =====
[random] after task 0: T0=0.955
[random] after task 1: T0=0.429, T1=0.828
[random] after task 2: T0=0.296, T1=0.025, T2=0.900
[random] after task 3: T0=0.422, T1=0.040, T2=0.076, T3=0.953
[random] after task 4: T0=0.080, T1=0.111, T2=0.253, T3=0.484, T4=0.942
[random | seed 1] avg_acc=0.374  BWT=-0.677

===== Running condition: random | seed 2 =====
[random] after task 0: T0=0.963
[random] after task 1: T0=0.380, T1=0.827
[random] after task 2: T0=0.329, T1=0.045, T2=0.905
[random] after task 3: T0=0.333, T1=0.032, T2=0.034, T3=0.941
[random] after task 4: T0=0.081, T1=0.103

In [4]:
!python actr_experiment.py --data-root /kaggle/working/data --condition shallow --seeds 1 --epochs 5

Using device: cuda
100%|████████████████████████████████████████| 170M/170M [32:07<00:00, 88.4kB/s]

===== Running condition: shallow | seed 0 =====
[shallow] after task 0: T0=0.966
[shallow] after task 1: T0=0.349, T1=0.837
[shallow] after task 2: T0=0.326, T1=0.042, T2=0.894
[shallow] after task 3: T0=0.398, T1=0.041, T2=0.072, T3=0.957
[shallow] after task 4: T0=0.079, T1=0.177, T2=0.243, T3=0.329, T4=0.931
[shallow | seed 0] avg_acc=0.352  BWT=-0.706

===== Summary (mean +/- std over seeds) =====
condition             avg_acc                BWT   (BWT closer to 0 = less forgetting)
shallow       0.352 +/- 0.000    -0.706 +/- 0.000 


In [5]:
!python actr_experiment.py --data-root /kaggle/working/data --condition semantic --seeds 1 --epochs 5

Using device: cuda

===== Running condition: semantic | seed 0 =====
[semantic] after task 0: T0=0.960
[semantic] after task 1: T0=0.332, T1=0.814
[semantic] after task 2: T0=0.381, T1=0.036, T2=0.881
[semantic] after task 3: T0=0.315, T1=0.040, T2=0.066, T3=0.953
[semantic] after task 4: T0=0.074, T1=0.166, T2=0.234, T3=0.302, T4=0.938
[semantic | seed 0] avg_acc=0.343  BWT=-0.708

===== Summary (mean +/- std over seeds) =====
condition             avg_acc                BWT   (BWT closer to 0 = less forgetting)
semantic      0.343 +/- 0.000    -0.708 +/- 0.000 


In [1]:
%%writefile actr_experiment.py
import argparse
import random
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T


def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.shallow_pool = nn.AdaptiveAvgPool2d((4, 4))

    def shallow_features(self, x):
        h = self.conv1(x)
        h = self.shallow_pool(h)
        return h.flatten(1)

    def semantic_features(self, x):
        h = self.conv1(x)
        h = self.conv2(h)
        h = self.conv3(h)
        h = h.flatten(1)
        h = F.relu(self.fc1(h))
        return h

    def forward(self, x):
        h = self.semantic_features(x)
        return self.fc2(h)


def build_split_cifar10(data_root="./data", quick=False):
    transform = T.Compose([T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])
    train_full = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=transform)
    test_full = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=transform)

    task_classes = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

    def indices_for_classes(dataset, classes):
        idx = [i for i, y in enumerate(dataset.targets) if y in classes]
        if quick:
            idx = idx[:400]
        return idx

    train_tasks = [Subset(train_full, indices_for_classes(train_full, c)) for c in task_classes]
    test_tasks = [Subset(test_full, indices_for_classes(test_full, c)) for c in task_classes]
    return train_tasks, test_tasks


class MemoryBank:
    def __init__(self, per_class_capacity=40):
        self.per_class_capacity = per_class_capacity
        self.images = []
        self.labels = []
        self._by_class = defaultdict(list)
        self._embed_cache = {}
        self.retrieval_counts = []

    def _ensure_retrieval_counts(self):
        while len(self.retrieval_counts) < len(self.images):
            self.retrieval_counts.append(0)

    def add_task_data(self, dataset):
        by_class = defaultdict(list)
        for i in range(len(dataset)):
            _, y = dataset[i]
            by_class[y].append(i)

        for y, idxs in by_class.items():
            chosen = random.sample(idxs, min(self.per_class_capacity, len(idxs)))
            for i in chosen:
                x, y2 = dataset[i]
                pos = len(self.images)
                self.images.append(x)
                self.labels.append(y2)
                self._by_class[y2].append(pos)

    def __len__(self):
        return len(self.images)

    def random_batch(self, batch_size, device):
        if len(self) == 0:
            return None
        n = min(batch_size, len(self))
        idxs = random.sample(range(len(self)), n)
        return self._gather(idxs, device)

    def refresh_embeddings(self, feature_fn, name, device, chunk_size=256):
        if len(self) == 0:
            self._embed_cache[name] = None
            return
        with torch.no_grad():
            feats = []
            for start in range(0, len(self.images), chunk_size):
                chunk = torch.stack(self.images[start:start + chunk_size]).to(device)
                feats.append(feature_fn(chunk))
            feats = torch.cat(feats, dim=0)
            self._embed_cache[name] = F.normalize(feats, dim=1)

    def similarity_batch_cached(self, query_features, name, batch_size, device, temperature=0.1):
        mem_feats = self._embed_cache.get(name)
        if mem_feats is None or len(self) == 0:
            return None
        n = min(batch_size, len(self))
        with torch.no_grad():
            query = F.normalize(query_features.mean(dim=0, keepdim=True), dim=1)
            sims = (mem_feats @ query.T).squeeze(1)
            probs = F.softmax(sims / temperature, dim=0).cpu().numpy()
            probs = probs / probs.sum()
            idxs = np.random.choice(len(probs), size=n, replace=False, p=probs)

        self._ensure_retrieval_counts()
        for i in idxs.tolist():
            self.retrieval_counts[i] += 1

        return self._gather(idxs.tolist(), device)

    def coverage_report(self):
        self._ensure_retrieval_counts()
        report = {}
        for y, idxs in self._by_class.items():
            total_retrievals = sum(self.retrieval_counts[i] for i in idxs)
            report[y] = (total_retrievals, len(idxs))
        return report

    def _gather(self, idxs, device):
        imgs = torch.stack([self.images[i] for i in idxs]).to(device)
        labels = torch.tensor([self.labels[i] for i in idxs]).to(device)
        return imgs, labels


def evaluate(model, test_tasks, seen_task_ids, device):
    model.eval()
    accs = {}
    with torch.no_grad():
        for t in seen_task_ids:
            loader = DataLoader(test_tasks[t], batch_size=256)
            correct, total = 0, 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                pred = model(x).argmax(dim=1)
                correct += (pred == y).sum().item()
                total += y.size(0)
            accs[t] = correct / total
    model.train()
    return accs


def run_condition(condition, train_tasks, test_tasks, device, epochs=3, batch_size=64, lr=1e-3,
                   mem_batch_size=32, refresh_every=50, seed=0, mem_capacity=40, temperature=0.1,
                   grad_clip=5.0):
    set_seed(seed)
    model = SmallCNN(num_classes=10).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    memory = MemoryBank(per_class_capacity=mem_capacity)

    feature_fn = {"semantic": model.semantic_features, "shallow": model.shallow_features}.get(condition)

    num_tasks = len(train_tasks)
    R = [[None] * num_tasks for _ in range(num_tasks)]

    for t in range(num_tasks):
        loader = DataLoader(train_tasks[t], batch_size=batch_size, shuffle=True)

        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        step = 0
        for epoch in range(epochs):
            for x, y in loader:
                x, y = x.to(device), y.to(device)

                replay = None
                if condition == "random":
                    replay = memory.random_batch(mem_batch_size, device)
                elif condition in ("semantic", "shallow"):
                    with torch.no_grad():
                        q_feats = feature_fn(x)
                    replay = memory.similarity_batch_cached(q_feats, condition, mem_batch_size, device,
                                                             temperature=temperature)
                else:
                    raise ValueError(f"unknown condition: {condition}")

                if replay is not None:
                    rx, ry = replay
                    x_all = torch.cat([x, rx], dim=0)
                    y_all = torch.cat([y, ry], dim=0)
                else:
                    x_all, y_all = x, y

                optimizer.zero_grad()
                logits = model(x_all)
                loss = F.cross_entropy(logits, y_all)

                if not torch.isfinite(loss):
                    print(f"  [warning] non-finite loss ({loss.item()}) at step {step} -- skipping this batch")
                    optimizer.zero_grad()
                    step += 1
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
                optimizer.step()

                step += 1
                if feature_fn is not None and step % refresh_every == 0:
                    memory.refresh_embeddings(feature_fn, condition, device)

        memory.add_task_data(train_tasks[t])
        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        accs = evaluate(model, test_tasks, seen_task_ids=list(range(t + 1)), device=device)
        for j, a in accs.items():
            R[t][j] = a
        print(f"[{condition}] after task {t}: " + ", ".join(f"T{j}={a:.3f}" for j, a in accs.items()))

    if feature_fn is not None:
        report = memory.coverage_report()
        starved = [y for y, (retrievals, n_items) in report.items() if n_items > 0 and retrievals == 0]
        cov_str = ", ".join(f"cls{y}={retrievals}/{n_items}" for y, (retrievals, n_items) in sorted(report.items()))
        print(f"[{condition}] retrieval coverage (times_retrieved/items_stored per class): {cov_str}")
        if starved:
            print(f"[{condition}] WARNING: classes never retrieved even once: {starved}")

    return R


def summarize(R, num_tasks):
    final_row = R[num_tasks - 1]
    avg_acc = np.mean([v for v in final_row if v is not None])

    diffs = []
    for i in range(num_tasks - 1):
        acc_after_last = R[num_tasks - 1][i]
        acc_right_after_i = R[i][i]
        if acc_after_last is not None and acc_right_after_i is not None:
            diffs.append(acc_after_last - acc_right_after_i)
    bwt = np.mean(diffs) if diffs else float("nan")

    return avg_acc, bwt


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--condition", choices=["random", "semantic", "shallow", "all"], default="all")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--quick", action="store_true")
    parser.add_argument("--data-root", type=str, default="./data")
    parser.add_argument("--refresh-every", type=int, default=50)
    parser.add_argument("--seeds", type=int, default=5)
    parser.add_argument("--mem-capacity", type=int, default=40)
    parser.add_argument("--temperature", type=float, default=0.1)
    parser.add_argument("--grad-clip", type=float, default=5.0)
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_tasks, test_tasks = build_split_cifar10(data_root=args.data_root, quick=args.quick)
    num_tasks = len(train_tasks)

    conditions = ["random", "semantic", "shallow"] if args.condition == "all" else [args.condition]

    results = {cond: [] for cond in conditions}
    for cond in conditions:
        for seed in range(args.seeds):
            print(f"\n===== Running condition: {cond} | seed {seed} =====")
            R = run_condition(cond, train_tasks, test_tasks, device, epochs=args.epochs,
                               refresh_every=args.refresh_every, seed=seed,
                               mem_batch_size=32, mem_capacity=args.mem_capacity,
                               temperature=args.temperature, grad_clip=args.grad_clip)
            avg_acc, bwt = summarize(R, num_tasks)
            results[cond].append((avg_acc, bwt))
            print(f"[{cond} | seed {seed}] avg_acc={avg_acc:.3f}  BWT={bwt:.3f}")

    print("\n===== Summary (mean +/- std over seeds) =====")
    print(f"{'condition':<10} {'avg_acc':>18} {'BWT':>18}   (BWT closer to 0 = less forgetting)")
    for cond in conditions:
        accs = np.array([r[0] for r in results[cond]])
        bwts = np.array([r[1] for r in results[cond]])
        print(f"{cond:<10} {accs.mean():>8.3f} +/- {accs.std():<6.3f} "
              f"{bwts.mean():>8.3f} +/- {bwts.std():<6.3f}")


if __name__ == "__main__":
    main()

Writing actr_experiment.py


In [2]:
!python actr_experiment.py --data-root /kaggle/working/data --condition shallow --seeds 1 --epochs 5

Using device: cuda
100%|████████████████████████████████████████| 170M/170M [29:45<00:00, 95.5kB/s]

===== Running condition: shallow | seed 0 =====
[shallow] after task 0: T0=0.958
[shallow] after task 1: T0=0.317, T1=0.840
[shallow] after task 2: T0=0.256, T1=0.030, T2=0.895
[shallow] after task 3: T0=0.465, T1=0.046, T2=0.072, T3=0.953
[shallow] after task 4: T0=0.090, T1=0.121, T2=0.234, T3=0.258, T4=0.943
[shallow] retrieval coverage (times_retrieved/items_stored per class): cls0=18008/40, cls1=33953/40, cls2=11578/40, cls3=16296/40, cls4=7320/40, cls5=6843/40, cls6=3203/40, cls7=3279/40, cls8=0/40, cls9=0/40
[shallow] WARNING: classes never retrieved even once: [9, 8]
[shallow | seed 0] avg_acc=0.329  BWT=-0.735

===== Summary (mean +/- std over seeds) =====
condition             avg_acc                BWT   (BWT closer to 0 = less forgetting)
shallow       0.329 +/- 0.000    -0.735 +/- 0.000 


In [3]:
!python actr_experiment.py --data-root /kaggle/working/data --condition shallow --seeds 1 --epochs 5 --floor-frac 0.2

usage: actr_experiment.py [-h] [--condition {random,semantic,shallow,all}]
                          [--epochs EPOCHS] [--quick] [--data-root DATA_ROOT]
                          [--refresh-every REFRESH_EVERY] [--seeds SEEDS]
                          [--mem-capacity MEM_CAPACITY]
                          [--temperature TEMPERATURE] [--grad-clip GRAD_CLIP]
actr_experiment.py: error: unrecognized arguments: --floor-frac 0.2


In [1]:
%%writefile actr_experiment.py
import argparse
import random
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T


def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.shallow_pool = nn.AdaptiveAvgPool2d((4, 4))

    def shallow_features(self, x):
        h = self.conv1(x)
        h = self.shallow_pool(h)
        return h.flatten(1)

    def semantic_features(self, x):
        h = self.conv1(x)
        h = self.conv2(h)
        h = self.conv3(h)
        h = h.flatten(1)
        h = F.relu(self.fc1(h))
        return h

    def forward(self, x):
        h = self.semantic_features(x)
        return self.fc2(h)


def build_split_cifar10(data_root="./data", quick=False):
    transform = T.Compose([T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])
    train_full = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=transform)
    test_full = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=transform)

    task_classes = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

    def indices_for_classes(dataset, classes):
        idx = [i for i, y in enumerate(dataset.targets) if y in classes]
        if quick:
            idx = idx[:400]
        return idx

    train_tasks = [Subset(train_full, indices_for_classes(train_full, c)) for c in task_classes]
    test_tasks = [Subset(test_full, indices_for_classes(test_full, c)) for c in task_classes]
    return train_tasks, test_tasks


class MemoryBank:
    def __init__(self, per_class_capacity=40):
        self.per_class_capacity = per_class_capacity
        self.images = []
        self.labels = []
        self._by_class = defaultdict(list)
        self._embed_cache = {}
        self.retrieval_counts = []

    def _ensure_retrieval_counts(self):
        while len(self.retrieval_counts) < len(self.images):
            self.retrieval_counts.append(0)

    def add_task_data(self, dataset):
        by_class = defaultdict(list)
        for i in range(len(dataset)):
            _, y = dataset[i]
            by_class[y].append(i)

        for y, idxs in by_class.items():
            chosen = random.sample(idxs, min(self.per_class_capacity, len(idxs)))
            for i in chosen:
                x, y2 = dataset[i]
                pos = len(self.images)
                self.images.append(x)
                self.labels.append(y2)
                self._by_class[y2].append(pos)

    def __len__(self):
        return len(self.images)

    def random_batch(self, batch_size, device):
        if len(self) == 0:
            return None
        n = min(batch_size, len(self))
        idxs = random.sample(range(len(self)), n)
        return self._gather(idxs, device)

    def refresh_embeddings(self, feature_fn, name, device, chunk_size=256):
        if len(self) == 0:
            self._embed_cache[name] = None
            return
        with torch.no_grad():
            feats = []
            for start in range(0, len(self.images), chunk_size):
                chunk = torch.stack(self.images[start:start + chunk_size]).to(device)
                feats.append(feature_fn(chunk))
            feats = torch.cat(feats, dim=0)
            self._embed_cache[name] = F.normalize(feats, dim=1)

    def similarity_batch_cached(self, query_features, name, batch_size, device, temperature=0.1,
                                 floor_frac=0.2):
        if len(self) == 0:
            return None
        n = min(batch_size, len(self))
        floor_n = min(n, int(round(n * floor_frac)))

        selected = set()

        if floor_n > 0:
            classes = list(self._by_class.keys())
            random.shuffle(classes)
            ci = 0
            guard = 0
            while len(selected) < floor_n and guard < floor_n * 50 + 100:
                cls = classes[ci % len(classes)]
                candidates = [i for i in self._by_class[cls] if i not in selected]
                if candidates:
                    selected.add(random.choice(candidates))
                ci += 1
                guard += 1

        remaining_n = n - len(selected)
        mem_feats = self._embed_cache.get(name)
        if remaining_n > 0 and mem_feats is not None:
            with torch.no_grad():
                query = F.normalize(query_features.mean(dim=0, keepdim=True), dim=1)
                sims = (mem_feats @ query.T).squeeze(1).cpu().numpy()

            mask = np.ones(len(sims), dtype=bool)
            for i in selected:
                mask[i] = False
            avail_idx = np.where(mask)[0]

            if len(avail_idx) > 0:
                avail_sims = sims[avail_idx]
                probs = np.exp((avail_sims - avail_sims.max()) / temperature)
                probs = probs / probs.sum()
                take = min(remaining_n, len(avail_idx))
                chosen = np.random.choice(len(avail_idx), size=take, replace=False, p=probs)
                selected.update(avail_idx[chosen].tolist())
        elif remaining_n > 0:
            candidates = [i for i in range(len(self)) if i not in selected]
            take = min(remaining_n, len(candidates))
            selected.update(random.sample(candidates, take))

        idxs = list(selected)
        self._ensure_retrieval_counts()
        for i in idxs:
            self.retrieval_counts[i] += 1

        return self._gather(idxs, device)

    def coverage_report(self):
        self._ensure_retrieval_counts()
        report = {}
        for y, idxs in self._by_class.items():
            total_retrievals = sum(self.retrieval_counts[i] for i in idxs)
            report[y] = (total_retrievals, len(idxs))
        return report

    def _gather(self, idxs, device):
        imgs = torch.stack([self.images[i] for i in idxs]).to(device)
        labels = torch.tensor([self.labels[i] for i in idxs]).to(device)
        return imgs, labels


def evaluate(model, test_tasks, seen_task_ids, device):
    model.eval()
    accs = {}
    with torch.no_grad():
        for t in seen_task_ids:
            loader = DataLoader(test_tasks[t], batch_size=256)
            correct, total = 0, 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                pred = model(x).argmax(dim=1)
                correct += (pred == y).sum().item()
                total += y.size(0)
            accs[t] = correct / total
    model.train()
    return accs


def run_condition(condition, train_tasks, test_tasks, device, epochs=3, batch_size=64, lr=1e-3,
                   mem_batch_size=32, refresh_every=50, seed=0, mem_capacity=40, temperature=0.1,
                   grad_clip=5.0, floor_frac=0.2):
    set_seed(seed)
    model = SmallCNN(num_classes=10).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    memory = MemoryBank(per_class_capacity=mem_capacity)

    feature_fn = {"semantic": model.semantic_features, "shallow": model.shallow_features}.get(condition)

    num_tasks = len(train_tasks)
    R = [[None] * num_tasks for _ in range(num_tasks)]

    for t in range(num_tasks):
        loader = DataLoader(train_tasks[t], batch_size=batch_size, shuffle=True)

        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        step = 0
        for epoch in range(epochs):
            for x, y in loader:
                x, y = x.to(device), y.to(device)

                replay = None
                if condition == "random":
                    replay = memory.random_batch(mem_batch_size, device)
                elif condition in ("semantic", "shallow"):
                    with torch.no_grad():
                        q_feats = feature_fn(x)
                    replay = memory.similarity_batch_cached(q_feats, condition, mem_batch_size, device,
                                                             temperature=temperature, floor_frac=floor_frac)
                else:
                    raise ValueError(f"unknown condition: {condition}")

                if replay is not None:
                    rx, ry = replay
                    x_all = torch.cat([x, rx], dim=0)
                    y_all = torch.cat([y, ry], dim=0)
                else:
                    x_all, y_all = x, y

                optimizer.zero_grad()
                logits = model(x_all)
                loss = F.cross_entropy(logits, y_all)

                if not torch.isfinite(loss):
                    print(f"  [warning] non-finite loss ({loss.item()}) at step {step} -- skipping this batch")
                    optimizer.zero_grad()
                    step += 1
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
                optimizer.step()

                step += 1
                if feature_fn is not None and step % refresh_every == 0:
                    memory.refresh_embeddings(feature_fn, condition, device)

        memory.add_task_data(train_tasks[t])
        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        accs = evaluate(model, test_tasks, seen_task_ids=list(range(t + 1)), device=device)
        for j, a in accs.items():
            R[t][j] = a
        print(f"[{condition}] after task {t}: " + ", ".join(f"T{j}={a:.3f}" for j, a in accs.items()))

    if feature_fn is not None:
        report = memory.coverage_report()
        starved = [y for y, (retrievals, n_items) in report.items() if n_items > 0 and retrievals == 0]
        cov_str = ", ".join(f"cls{y}={retrievals}/{n_items}" for y, (retrievals, n_items) in sorted(report.items()))
        print(f"[{condition}] retrieval coverage (times_retrieved/items_stored per class): {cov_str}")
        if starved:
            print(f"[{condition}] WARNING: classes never retrieved even once: {starved}")

    return R


def summarize(R, num_tasks):
    final_row = R[num_tasks - 1]
    avg_acc = np.mean([v for v in final_row if v is not None])

    diffs = []
    for i in range(num_tasks - 1):
        acc_after_last = R[num_tasks - 1][i]
        acc_right_after_i = R[i][i]
        if acc_after_last is not None and acc_right_after_i is not None:
            diffs.append(acc_after_last - acc_right_after_i)
    bwt = np.mean(diffs) if diffs else float("nan")

    return avg_acc, bwt


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--condition", choices=["random", "semantic", "shallow", "all"], default="all")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--quick", action="store_true")
    parser.add_argument("--data-root", type=str, default="./data")
    parser.add_argument("--refresh-every", type=int, default=50)
    parser.add_argument("--seeds", type=int, default=5)
    parser.add_argument("--mem-capacity", type=int, default=40)
    parser.add_argument("--temperature", type=float, default=0.1)
    parser.add_argument("--grad-clip", type=float, default=5.0)
    parser.add_argument("--floor-frac", type=float, default=0.2)
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_tasks, test_tasks = build_split_cifar10(data_root=args.data_root, quick=args.quick)
    num_tasks = len(train_tasks)

    conditions = ["random", "semantic", "shallow"] if args.condition == "all" else [args.condition]

    results = {cond: [] for cond in conditions}
    for cond in conditions:
        for seed in range(args.seeds):
            print(f"\n===== Running condition: {cond} | seed {seed} =====")
            R = run_condition(cond, train_tasks, test_tasks, device, epochs=args.epochs,
                               refresh_every=args.refresh_every, seed=seed,
                               mem_batch_size=32, mem_capacity=args.mem_capacity,
                               temperature=args.temperature, grad_clip=args.grad_clip,
                               floor_frac=args.floor_frac)
            avg_acc, bwt = summarize(R, num_tasks)
            results[cond].append((avg_acc, bwt))
            print(f"[{cond} | seed {seed}] avg_acc={avg_acc:.3f}  BWT={bwt:.3f}")

    print("\n===== Summary (mean +/- std over seeds) =====")
    print(f"{'condition':<10} {'avg_acc':>18} {'BWT':>18}   (BWT closer to 0 = less forgetting)")
    for cond in conditions:
        accs = np.array([r[0] for r in results[cond]])
        bwts = np.array([r[1] for r in results[cond]])
        print(f"{cond:<10} {accs.mean():>8.3f} +/- {accs.std():<6.3f} "
              f"{bwts.mean():>8.3f} +/- {bwts.std():<6.3f}")


if __name__ == "__main__":
    main()

Writing actr_experiment.py


In [2]:
!python actr_experiment.py --data-root /kaggle/working/data --condition shallow --seeds 1 --epochs 5 --floor-frac 0.2

Using device: cuda
100%|█████████████████████████████████████████| 170M/170M [26:50<00:00, 106kB/s]

===== Running condition: shallow | seed 0 =====
[shallow] after task 0: T0=0.964
[shallow] after task 1: T0=0.443, T1=0.832
[shallow] after task 2: T0=0.325, T1=0.015, T2=0.896
[shallow] after task 3: T0=0.404, T1=0.049, T2=0.088, T3=0.951
[shallow] after task 4: T0=0.083, T1=0.099, T2=0.309, T3=0.297, T4=0.939
[shallow] retrieval coverage (times_retrieved/items_stored per class): cls0=19724/40, cls1=32266/40, cls2=11861/40, cls3=15543/40, cls4=7484/40, cls5=7139/40, cls6=3296/40, cls7=3167/40, cls8=0/40, cls9=0/40
[shallow] WARNING: classes never retrieved even once: [9, 8]
[shallow | seed 0] avg_acc=0.346  BWT=-0.714

===== Summary (mean +/- std over seeds) =====
condition             avg_acc                BWT   (BWT closer to 0 = less forgetting)
shallow       0.346 +/- 0.000    -0.714 +/- 0.000 


In [3]:
%%writefile actr_experiment.py
import argparse
import random
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T


def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.shallow_pool = nn.AdaptiveAvgPool2d((4, 4))

    def shallow_features(self, x):
        h = self.conv1(x)
        h = self.shallow_pool(h)
        return h.flatten(1)

    def semantic_features(self, x):
        h = self.conv1(x)
        h = self.conv2(h)
        h = self.conv3(h)
        h = h.flatten(1)
        h = F.relu(self.fc1(h))
        return h

    def forward(self, x):
        h = self.semantic_features(x)
        return self.fc2(h)


def build_split_cifar10(data_root="./data", quick=False):
    transform = T.Compose([T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])
    train_full = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=transform)
    test_full = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=transform)

    task_classes = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

    def indices_for_classes(dataset, classes):
        idx = [i for i, y in enumerate(dataset.targets) if y in classes]
        if quick:
            idx = idx[:400]
        return idx

    train_tasks = [Subset(train_full, indices_for_classes(train_full, c)) for c in task_classes]
    test_tasks = [Subset(test_full, indices_for_classes(test_full, c)) for c in task_classes]
    return train_tasks, test_tasks


class MemoryBank:
    def __init__(self, per_class_capacity=40):
        self.per_class_capacity = per_class_capacity
        self.images = []
        self.labels = []
        self._by_class = defaultdict(list)
        self._embed_cache = {}
        self.retrieval_counts = []

    def _ensure_retrieval_counts(self):
        while len(self.retrieval_counts) < len(self.images):
            self.retrieval_counts.append(0)

    def add_task_data(self, dataset):
        by_class = defaultdict(list)
        for i in range(len(dataset)):
            _, y = dataset[i]
            by_class[y].append(i)

        for y, idxs in by_class.items():
            chosen = random.sample(idxs, min(self.per_class_capacity, len(idxs)))
            for i in chosen:
                x, y2 = dataset[i]
                pos = len(self.images)
                self.images.append(x)
                self.labels.append(y2)
                self._by_class[y2].append(pos)

    def __len__(self):
        return len(self.images)

    def random_batch(self, batch_size, device):
        if len(self) == 0:
            return None
        n = min(batch_size, len(self))
        idxs = random.sample(range(len(self)), n)
        return self._gather(idxs, device)

    def refresh_embeddings(self, feature_fn, name, device, chunk_size=256):
        if len(self) == 0:
            self._embed_cache[name] = None
            return
        with torch.no_grad():
            feats = []
            for start in range(0, len(self.images), chunk_size):
                chunk = torch.stack(self.images[start:start + chunk_size]).to(device)
                feats.append(feature_fn(chunk))
            feats = torch.cat(feats, dim=0)
            self._embed_cache[name] = F.normalize(feats, dim=1)

    def similarity_batch_cached(self, query_features, name, batch_size, device, temperature=0.1,
                                 floor_frac=0.2):
        if len(self) == 0:
            return None
        n = min(batch_size, len(self))
        floor_n = min(n, int(round(n * floor_frac)))

        selected = set()

        if floor_n > 0:
            classes = list(self._by_class.keys())
            random.shuffle(classes)
            ci = 0
            guard = 0
            while len(selected) < floor_n and guard < floor_n * 50 + 100:
                cls = classes[ci % len(classes)]
                candidates = [i for i in self._by_class[cls] if i not in selected]
                if candidates:
                    selected.add(random.choice(candidates))
                ci += 1
                guard += 1

        remaining_n = n - len(selected)
        mem_feats = self._embed_cache.get(name)
        if remaining_n > 0 and mem_feats is not None:
            with torch.no_grad():
                query = F.normalize(query_features.mean(dim=0, keepdim=True), dim=1)
                sims = (mem_feats @ query.T).squeeze(1).cpu().numpy()

            mask = np.ones(len(sims), dtype=bool)
            for i in selected:
                mask[i] = False
            avail_idx = np.where(mask)[0]

            if len(avail_idx) > 0:
                avail_sims = sims[avail_idx]
                probs = np.exp((avail_sims - avail_sims.max()) / temperature)
                probs = probs / probs.sum()
                take = min(remaining_n, len(avail_idx))
                chosen = np.random.choice(len(avail_idx), size=take, replace=False, p=probs)
                selected.update(avail_idx[chosen].tolist())
        elif remaining_n > 0:
            candidates = [i for i in range(len(self)) if i not in selected]
            take = min(remaining_n, len(candidates))
            selected.update(random.sample(candidates, take))

        idxs = list(selected)
        self._ensure_retrieval_counts()
        for i in idxs:
            self.retrieval_counts[i] += 1

        return self._gather(idxs, device)

    def coverage_report(self):
        self._ensure_retrieval_counts()
        report = {}
        for y, idxs in self._by_class.items():
            total_retrievals = sum(self.retrieval_counts[i] for i in idxs)
            report[y] = (total_retrievals, len(idxs))
        return report

    def _gather(self, idxs, device):
        imgs = torch.stack([self.images[i] for i in idxs]).to(device)
        labels = torch.tensor([self.labels[i] for i in idxs]).to(device)
        return imgs, labels


def evaluate(model, test_tasks, seen_task_ids, device):
    model.eval()
    accs = {}
    with torch.no_grad():
        for t in seen_task_ids:
            loader = DataLoader(test_tasks[t], batch_size=256)
            correct, total = 0, 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                pred = model(x).argmax(dim=1)
                correct += (pred == y).sum().item()
                total += y.size(0)
            accs[t] = correct / total
    model.train()
    return accs


def run_condition(condition, train_tasks, test_tasks, device, epochs=3, batch_size=64, lr=1e-3,
                   mem_batch_size=32, refresh_every=50, seed=0, mem_capacity=40, temperature=0.1,
                   grad_clip=5.0, floor_frac=0.2):
    set_seed(seed)
    model = SmallCNN(num_classes=10).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    memory = MemoryBank(per_class_capacity=mem_capacity)

    feature_fn = {"semantic": model.semantic_features, "shallow": model.shallow_features}.get(condition)

    num_tasks = len(train_tasks)
    R = [[None] * num_tasks for _ in range(num_tasks)]

    for t in range(num_tasks):
        loader = DataLoader(train_tasks[t], batch_size=batch_size, shuffle=True)

        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        step = 0
        for epoch in range(epochs):
            for x, y in loader:
                x, y = x.to(device), y.to(device)

                replay = None
                if condition == "random":
                    replay = memory.random_batch(mem_batch_size, device)
                elif condition in ("semantic", "shallow"):
                    with torch.no_grad():
                        q_feats = feature_fn(x)
                    replay = memory.similarity_batch_cached(q_feats, condition, mem_batch_size, device,
                                                             temperature=temperature, floor_frac=floor_frac)
                else:
                    raise ValueError(f"unknown condition: {condition}")

                if replay is not None:
                    rx, ry = replay
                    x_all = torch.cat([x, rx], dim=0)
                    y_all = torch.cat([y, ry], dim=0)
                else:
                    x_all, y_all = x, y

                optimizer.zero_grad()
                logits = model(x_all)
                loss = F.cross_entropy(logits, y_all)

                if not torch.isfinite(loss):
                    print(f"  [warning] non-finite loss ({loss.item()}) at step {step} -- skipping this batch")
                    optimizer.zero_grad()
                    step += 1
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
                optimizer.step()

                step += 1
                if feature_fn is not None and step % refresh_every == 0:
                    memory.refresh_embeddings(feature_fn, condition, device)

        memory.add_task_data(train_tasks[t])
        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        accs = evaluate(model, test_tasks, seen_task_ids=list(range(t + 1)), device=device)
        for j, a in accs.items():
            R[t][j] = a
        print(f"[{condition}] after task {t}: " + ", ".join(f"T{j}={a:.3f}" for j, a in accs.items()))

    if feature_fn is not None:
        report = memory.coverage_report()
        starved = [y for y, (retrievals, n_items) in report.items() if n_items > 0 and retrievals == 0]
        cov_str = ", ".join(f"cls{y}={retrievals}/{n_items}" for y, (retrievals, n_items) in sorted(report.items()))
        print(f"[{condition}] retrieval coverage (times_retrieved/items_stored per class): {cov_str}")
        if starved:
            print(f"[{condition}] WARNING: classes never retrieved even once: {starved}")

    return R


def summarize(R, num_tasks):
    final_row = R[num_tasks - 1]
    avg_acc = np.mean([v for v in final_row if v is not None])

    diffs = []
    for i in range(num_tasks - 1):
        acc_after_last = R[num_tasks - 1][i]
        acc_right_after_i = R[i][i]
        if acc_after_last is not None and acc_right_after_i is not None:
            diffs.append(acc_after_last - acc_right_after_i)
    bwt = np.mean(diffs) if diffs else float("nan")

    return avg_acc, bwt


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--condition", choices=["random", "semantic", "shallow", "all"], default="all")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--quick", action="store_true")
    parser.add_argument("--data-root", type=str, default="./data")
    parser.add_argument("--refresh-every", type=int, default=50)
    parser.add_argument("--seeds", type=int, default=5)
    parser.add_argument("--mem-capacity", type=int, default=40)
    parser.add_argument("--temperature", type=float, default=0.1)
    parser.add_argument("--grad-clip", type=float, default=5.0)
    parser.add_argument("--floor-frac", type=float, default=0.2)
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_tasks, test_tasks = build_split_cifar10(data_root=args.data_root, quick=args.quick)
    num_tasks = len(train_tasks)

    conditions = ["random", "semantic", "shallow"] if args.condition == "all" else [args.condition]

    results = {cond: [] for cond in conditions}
    for cond in conditions:
        for seed in range(args.seeds):
            print(f"\n===== Running condition: {cond} | seed {seed} =====")
            R = run_condition(cond, train_tasks, test_tasks, device, epochs=args.epochs,
                               refresh_every=args.refresh_every, seed=seed,
                               mem_batch_size=32, mem_capacity=args.mem_capacity,
                               temperature=args.temperature, grad_clip=args.grad_clip,
                               floor_frac=args.floor_frac)
            avg_acc, bwt = summarize(R, num_tasks)
            results[cond].append((avg_acc, bwt))
            print(f"[{cond} | seed {seed}] avg_acc={avg_acc:.3f}  BWT={bwt:.3f}")

    print("\n===== Summary (mean +/- std over seeds) =====")
    print(f"{'condition':<10} {'avg_acc':>18} {'BWT':>18}   (BWT closer to 0 = less forgetting)")
    for cond in conditions:
        accs = np.array([r[0] for r in results[cond]])
        bwts = np.array([r[1] for r in results[cond]])
        print(f"{cond:<10} {accs.mean():>8.3f} +/- {accs.std():<6.3f} "
              f"{bwts.mean():>8.3f} +/- {bwts.std():<6.3f}")


if __name__ == "__main__":
    main()

Overwriting actr_experiment.py


In [4]:
!pip install torch torchvision --quiet

In [5]:
!python actr_experiment.py --data-root /kaggle/working/data --seeds 5 --epochs 5 --floor-frac 0.2

Using device: cuda

===== Running condition: random | seed 0 =====
[random] after task 0: T0=0.959
[random] after task 1: T0=0.380, T1=0.779
[random] after task 2: T0=0.348, T1=0.011, T2=0.896
[random] after task 3: T0=0.344, T1=0.025, T2=0.030, T3=0.953
[random] after task 4: T0=0.060, T1=0.060, T2=0.136, T3=0.291, T4=0.944
[random | seed 0] avg_acc=0.298  BWT=-0.760

===== Running condition: random | seed 1 =====
[random] after task 0: T0=0.960
[random] after task 1: T0=0.401, T1=0.841
[random] after task 2: T0=0.416, T1=0.026, T2=0.891
[random] after task 3: T0=0.317, T1=0.019, T2=0.056, T3=0.944
[random] after task 4: T0=0.103, T1=0.129, T2=0.205, T3=0.435, T4=0.940
[random | seed 1] avg_acc=0.363  BWT=-0.691

===== Running condition: random | seed 2 =====
[random] after task 0: T0=0.957
[random] after task 1: T0=0.413, T1=0.820
[random] after task 2: T0=0.452, T1=0.034, T2=0.909
[random] after task 3: T0=0.316, T1=0.031, T2=0.035, T3=0.964
[random] after task 4: T0=0.053, T1=0.067

In [6]:
%%writefile actr_experiment.py
import argparse
import random
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T


def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.shallow_pool = nn.AdaptiveAvgPool2d((4, 4))

    def shallow_features(self, x):
        h = self.conv1(x)
        h = self.shallow_pool(h)
        return h.flatten(1)

    def semantic_features(self, x):
        h = self.conv1(x)
        h = self.conv2(h)
        h = self.conv3(h)
        h = h.flatten(1)
        h = F.relu(self.fc1(h))
        return h

    def forward(self, x):
        h = self.semantic_features(x)
        return self.fc2(h)


def build_split_dataset(dataset_name="cifar10", data_root="./data", classes_per_task=None, quick=False):
    transform = T.Compose([T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])

    if dataset_name == "cifar10":
        num_classes = 10
        default_cpt = 2
        train_full = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=transform)
        test_full = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=transform)
    elif dataset_name == "cifar100":
        num_classes = 100
        default_cpt = 10
        train_full = torchvision.datasets.CIFAR100(root=data_root, train=True, download=True, transform=transform)
        test_full = torchvision.datasets.CIFAR100(root=data_root, train=False, download=True, transform=transform)
    else:
        raise ValueError(f"unknown dataset: {dataset_name}")

    cpt = classes_per_task or default_cpt
    task_classes = [tuple(range(i, min(i + cpt, num_classes))) for i in range(0, num_classes, cpt)]

    def indices_for_classes(dataset, classes):
        idx = [i for i, y in enumerate(dataset.targets) if y in classes]
        if quick:
            idx = idx[:400]
        return idx

    train_tasks = [Subset(train_full, indices_for_classes(train_full, c)) for c in task_classes]
    test_tasks = [Subset(test_full, indices_for_classes(test_full, c)) for c in task_classes]
    return train_tasks, test_tasks, num_classes


class MemoryBank:
    def __init__(self, per_class_capacity=40):
        self.per_class_capacity = per_class_capacity
        self.images = []
        self.labels = []
        self._by_class = defaultdict(list)
        self._embed_cache = {}
        self.retrieval_counts = []

    def _ensure_retrieval_counts(self):
        while len(self.retrieval_counts) < len(self.images):
            self.retrieval_counts.append(0)

    def add_task_data(self, dataset):
        by_class = defaultdict(list)
        for i in range(len(dataset)):
            _, y = dataset[i]
            by_class[y].append(i)

        for y, idxs in by_class.items():
            chosen = random.sample(idxs, min(self.per_class_capacity, len(idxs)))
            for i in chosen:
                x, y2 = dataset[i]
                pos = len(self.images)
                self.images.append(x)
                self.labels.append(y2)
                self._by_class[y2].append(pos)

    def __len__(self):
        return len(self.images)

    def random_batch(self, batch_size, device):
        if len(self) == 0:
            return None
        n = min(batch_size, len(self))
        idxs = random.sample(range(len(self)), n)
        return self._gather(idxs, device)

    def refresh_embeddings(self, feature_fn, name, device, chunk_size=256):
        if len(self) == 0:
            self._embed_cache[name] = None
            return
        with torch.no_grad():
            feats = []
            for start in range(0, len(self.images), chunk_size):
                chunk = torch.stack(self.images[start:start + chunk_size]).to(device)
                feats.append(feature_fn(chunk))
            feats = torch.cat(feats, dim=0)
            self._embed_cache[name] = F.normalize(feats, dim=1)

    def similarity_batch_cached(self, query_features, name, batch_size, device, temperature=0.1,
                                 floor_frac=0.2):
        if len(self) == 0:
            return None
        n = min(batch_size, len(self))
        floor_n = min(n, int(round(n * floor_frac)))

        selected = set()

        if floor_n > 0:
            classes = list(self._by_class.keys())
            random.shuffle(classes)
            ci = 0
            guard = 0
            while len(selected) < floor_n and guard < floor_n * 50 + 100:
                cls = classes[ci % len(classes)]
                candidates = [i for i in self._by_class[cls] if i not in selected]
                if candidates:
                    selected.add(random.choice(candidates))
                ci += 1
                guard += 1

        remaining_n = n - len(selected)
        mem_feats = self._embed_cache.get(name)
        if remaining_n > 0 and mem_feats is not None:
            with torch.no_grad():
                query = F.normalize(query_features.mean(dim=0, keepdim=True), dim=1)
                sims = (mem_feats @ query.T).squeeze(1).cpu().numpy()

            mask = np.ones(len(sims), dtype=bool)
            for i in selected:
                mask[i] = False
            avail_idx = np.where(mask)[0]

            if len(avail_idx) > 0:
                avail_sims = sims[avail_idx]
                probs = np.exp((avail_sims - avail_sims.max()) / temperature)
                probs = probs / probs.sum()
                take = min(remaining_n, len(avail_idx))
                chosen = np.random.choice(len(avail_idx), size=take, replace=False, p=probs)
                selected.update(avail_idx[chosen].tolist())
        elif remaining_n > 0:
            candidates = [i for i in range(len(self)) if i not in selected]
            take = min(remaining_n, len(candidates))
            selected.update(random.sample(candidates, take))

        idxs = list(selected)
        self._ensure_retrieval_counts()
        for i in idxs:
            self.retrieval_counts[i] += 1

        return self._gather(idxs, device)

    def coverage_report(self):
        self._ensure_retrieval_counts()
        report = {}
        for y, idxs in self._by_class.items():
            total_retrievals = sum(self.retrieval_counts[i] for i in idxs)
            report[y] = (total_retrievals, len(idxs))
        return report

    def _gather(self, idxs, device):
        imgs = torch.stack([self.images[i] for i in idxs]).to(device)
        labels = torch.tensor([self.labels[i] for i in idxs]).to(device)
        return imgs, labels


def evaluate(model, test_tasks, seen_task_ids, device):
    model.eval()
    accs = {}
    with torch.no_grad():
        for t in seen_task_ids:
            loader = DataLoader(test_tasks[t], batch_size=256)
            correct, total = 0, 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                pred = model(x).argmax(dim=1)
                correct += (pred == y).sum().item()
                total += y.size(0)
            accs[t] = correct / total
    model.train()
    return accs


def run_condition(condition, train_tasks, test_tasks, device, num_classes=10, epochs=3, batch_size=64, lr=1e-3,
                   mem_batch_size=32, refresh_every=50, seed=0, mem_capacity=40, temperature=0.1,
                   grad_clip=5.0, floor_frac=0.2):
    set_seed(seed)
    model = SmallCNN(num_classes=num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    memory = MemoryBank(per_class_capacity=mem_capacity)

    feature_fn = {"semantic": model.semantic_features, "shallow": model.shallow_features}.get(condition)

    num_tasks = len(train_tasks)
    R = [[None] * num_tasks for _ in range(num_tasks)]

    for t in range(num_tasks):
        loader = DataLoader(train_tasks[t], batch_size=batch_size, shuffle=True)

        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        step = 0
        for epoch in range(epochs):
            for x, y in loader:
                x, y = x.to(device), y.to(device)

                replay = None
                if condition == "random":
                    replay = memory.random_batch(mem_batch_size, device)
                elif condition in ("semantic", "shallow"):
                    with torch.no_grad():
                        q_feats = feature_fn(x)
                    replay = memory.similarity_batch_cached(q_feats, condition, mem_batch_size, device,
                                                             temperature=temperature, floor_frac=floor_frac)
                else:
                    raise ValueError(f"unknown condition: {condition}")

                if replay is not None:
                    rx, ry = replay
                    x_all = torch.cat([x, rx], dim=0)
                    y_all = torch.cat([y, ry], dim=0)
                else:
                    x_all, y_all = x, y

                optimizer.zero_grad()
                logits = model(x_all)
                loss = F.cross_entropy(logits, y_all)

                if not torch.isfinite(loss):
                    print(f"  [warning] non-finite loss ({loss.item()}) at step {step} -- skipping this batch")
                    optimizer.zero_grad()
                    step += 1
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
                optimizer.step()

                step += 1
                if feature_fn is not None and step % refresh_every == 0:
                    memory.refresh_embeddings(feature_fn, condition, device)

        memory.add_task_data(train_tasks[t])
        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        accs = evaluate(model, test_tasks, seen_task_ids=list(range(t + 1)), device=device)
        for j, a in accs.items():
            R[t][j] = a
        print(f"[{condition}] after task {t}: " + ", ".join(f"T{j}={a:.3f}" for j, a in accs.items()))

    if feature_fn is not None:
        report = memory.coverage_report()
        starved = [y for y, (retrievals, n_items) in report.items() if n_items > 0 and retrievals == 0]
        cov_str = ", ".join(f"cls{y}={retrievals}/{n_items}" for y, (retrievals, n_items) in sorted(report.items()))
        print(f"[{condition}] retrieval coverage (times_retrieved/items_stored per class): {cov_str}")
        if starved:
            print(f"[{condition}] WARNING: classes never retrieved even once: {starved}")

    return R


def summarize(R, num_tasks):
    final_row = R[num_tasks - 1]
    avg_acc = np.mean([v for v in final_row if v is not None])

    diffs = []
    for i in range(num_tasks - 1):
        acc_after_last = R[num_tasks - 1][i]
        acc_right_after_i = R[i][i]
        if acc_after_last is not None and acc_right_after_i is not None:
            diffs.append(acc_after_last - acc_right_after_i)
    bwt = np.mean(diffs) if diffs else float("nan")

    return avg_acc, bwt


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--condition", choices=["random", "semantic", "shallow", "all"], default="all")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--quick", action="store_true")
    parser.add_argument("--data-root", type=str, default="./data")
    parser.add_argument("--refresh-every", type=int, default=50)
    parser.add_argument("--seeds", type=int, default=5)
    parser.add_argument("--mem-capacity", type=int, default=40)
    parser.add_argument("--temperature", type=float, default=0.1)
    parser.add_argument("--grad-clip", type=float, default=5.0)
    parser.add_argument("--floor-frac", type=float, default=0.2)
    parser.add_argument("--dataset", choices=["cifar10", "cifar100"], default="cifar10")
    parser.add_argument("--classes-per-task", type=int, default=None)
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_tasks, test_tasks, num_classes = build_split_dataset(
        dataset_name=args.dataset, data_root=args.data_root,
        classes_per_task=args.classes_per_task, quick=args.quick)
    num_tasks = len(train_tasks)
    print(f"Dataset: {args.dataset} | {num_tasks} tasks | {num_classes} total classes")

    conditions = ["random", "semantic", "shallow"] if args.condition == "all" else [args.condition]

    results = {cond: [] for cond in conditions}
    for cond in conditions:
        for seed in range(args.seeds):
            print(f"\n===== Running condition: {cond} | seed {seed} =====")
            R = run_condition(cond, train_tasks, test_tasks, device, num_classes=num_classes, epochs=args.epochs,
                               refresh_every=args.refresh_every, seed=seed,
                               mem_batch_size=32, mem_capacity=args.mem_capacity,
                               temperature=args.temperature, grad_clip=args.grad_clip,
                               floor_frac=args.floor_frac)
            avg_acc, bwt = summarize(R, num_tasks)
            results[cond].append((avg_acc, bwt))
            print(f"[{cond} | seed {seed}] avg_acc={avg_acc:.3f}  BWT={bwt:.3f}")

    print("\n===== Summary (mean +/- std over seeds) =====")
    print(f"{'condition':<10} {'avg_acc':>18} {'BWT':>18}   (BWT closer to 0 = less forgetting)")
    for cond in conditions:
        accs = np.array([r[0] for r in results[cond]])
        bwts = np.array([r[1] for r in results[cond]])
        print(f"{cond:<10} {accs.mean():>8.3f} +/- {accs.std():<6.3f} "
              f"{bwts.mean():>8.3f} +/- {bwts.std():<6.3f}")


if __name__ == "__main__":
    main()

Overwriting actr_experiment.py


In [7]:
!pip install torch torchvision --quiet

In [8]:
!python actr_experiment.py --data-root /kaggle/working/data --dataset cifar100 --quick --condition shallow --seeds 1 --epochs 2

Using device: cuda
100%|████████████████████████████████████████| 169M/169M [30:36<00:00, 92.0kB/s]
Dataset: cifar100 | 10 tasks | 100 total classes

===== Running condition: shallow | seed 0 =====
[shallow] after task 0: T0=0.105
[shallow] after task 1: T0=0.098, T1=0.000
[shallow] after task 2: T0=0.098, T1=0.000, T2=0.000
[shallow] after task 3: T0=0.102, T1=0.000, T2=0.000, T3=0.000
[shallow] after task 4: T0=0.000, T1=0.000, T2=0.110, T3=0.000, T4=0.000
[shallow] after task 5: T0=0.000, T1=0.000, T2=0.007, T3=0.095, T4=0.000, T5=0.000
[shallow] after task 6: T0=0.085, T1=0.000, T2=0.000, T3=0.000, T4=0.000, T5=0.000, T6=0.000
[shallow] after task 7: T0=0.000, T1=0.000, T2=0.000, T3=0.000, T4=0.000, T5=0.000, T6=0.092, T7=0.000
[shallow] after task 8: T0=0.000, T1=0.000, T2=0.000, T3=0.000, T4=0.000, T5=0.000, T6=0.000, T7=0.080, T8=0.000
[shallow] after task 9: T0=0.000, T1=0.000, T2=0.000, T3=0.000, T4=0.000, T5=0.000, T6=0.000, T7=0.000, T8=0.110, T9=0.000
[shallow] retrieval co

In [ ]:
!python actr_experiment.py --data-root /kaggle/working/data --dataset cifar100 --seeds 3 --epochs 5 --floor-frac 0.2

In [2]:
%%writefile actr_experiment.py
import argparse
import random
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T


def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.shallow_pool = nn.AdaptiveAvgPool2d((4, 4))

    def shallow_features(self, x):
        h = self.conv1(x)
        h = self.shallow_pool(h)
        return h.flatten(1)

    def semantic_features(self, x):
        h = self.conv1(x)
        h = self.conv2(h)
        h = self.conv3(h)
        h = h.flatten(1)
        h = F.relu(self.fc1(h))
        return h

    def forward(self, x):
        h = self.semantic_features(x)
        return self.fc2(h)


def build_split_dataset(dataset_name="cifar10", data_root="./data", classes_per_task=None, quick=False):
    transform = T.Compose([T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])

    if dataset_name == "cifar10":
        num_classes = 10
        default_cpt = 2
        train_full = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=transform)
        test_full = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=transform)
    elif dataset_name == "cifar100":
        num_classes = 100
        default_cpt = 10
        train_full = torchvision.datasets.CIFAR100(root=data_root, train=True, download=True, transform=transform)
        test_full = torchvision.datasets.CIFAR100(root=data_root, train=False, download=True, transform=transform)
    else:
        raise ValueError(f"unknown dataset: {dataset_name}")

    cpt = classes_per_task or default_cpt
    task_classes = [tuple(range(i, min(i + cpt, num_classes))) for i in range(0, num_classes, cpt)]

    def indices_for_classes(dataset, classes):
        idx = [i for i, y in enumerate(dataset.targets) if y in classes]
        if quick:
            idx = idx[:400]
        return idx

    train_tasks = [Subset(train_full, indices_for_classes(train_full, c)) for c in task_classes]
    test_tasks = [Subset(test_full, indices_for_classes(test_full, c)) for c in task_classes]
    return train_tasks, test_tasks, num_classes


class MemoryBank:
    def __init__(self, per_class_capacity=40):
        self.per_class_capacity = per_class_capacity
        self.images = []
        self.labels = []
        self._by_class = defaultdict(list)
        self._embed_cache = {}
        self.retrieval_counts = []

    def _ensure_retrieval_counts(self):
        while len(self.retrieval_counts) < len(self.images):
            self.retrieval_counts.append(0)

    def add_task_data(self, dataset):
        by_class = defaultdict(list)
        for i in range(len(dataset)):
            _, y = dataset[i]
            by_class[y].append(i)

        for y, idxs in by_class.items():
            chosen = random.sample(idxs, min(self.per_class_capacity, len(idxs)))
            for i in chosen:
                x, y2 = dataset[i]
                pos = len(self.images)
                self.images.append(x)
                self.labels.append(y2)
                self._by_class[y2].append(pos)

    def __len__(self):
        return len(self.images)

    def random_batch(self, batch_size, device):
        if len(self) == 0:
            return None
        n = min(batch_size, len(self))
        idxs = random.sample(range(len(self)), n)
        return self._gather(idxs, device)

    def refresh_embeddings(self, feature_fn, name, device, chunk_size=256):
        if len(self) == 0:
            self._embed_cache[name] = None
            return
        with torch.no_grad():
            feats = []
            for start in range(0, len(self.images), chunk_size):
                chunk = torch.stack(self.images[start:start + chunk_size]).to(device)
                feats.append(feature_fn(chunk))
            feats = torch.cat(feats, dim=0)
            self._embed_cache[name] = F.normalize(feats, dim=1)

    def similarity_batch_cached(self, query_features, name, batch_size, device, temperature=0.1,
                                 floor_frac=0.2):
        if len(self) == 0:
            return None
        n = min(batch_size, len(self))
        floor_n = min(n, int(round(n * floor_frac)))

        selected = set()

        if floor_n > 0:
            classes = list(self._by_class.keys())
            random.shuffle(classes)
            ci = 0
            guard = 0
            while len(selected) < floor_n and guard < floor_n * 50 + 100:
                cls = classes[ci % len(classes)]
                candidates = [i for i in self._by_class[cls] if i not in selected]
                if candidates:
                    selected.add(random.choice(candidates))
                ci += 1
                guard += 1

        remaining_n = n - len(selected)
        mem_feats = self._embed_cache.get(name)
        if remaining_n > 0 and mem_feats is not None:
            with torch.no_grad():
                query = F.normalize(query_features.mean(dim=0, keepdim=True), dim=1)
                sims = (mem_feats @ query.T).squeeze(1).cpu().numpy()

            mask = np.ones(len(sims), dtype=bool)
            for i in selected:
                mask[i] = False
            avail_idx = np.where(mask)[0]

            if len(avail_idx) > 0:
                avail_sims = sims[avail_idx]
                probs = np.exp((avail_sims - avail_sims.max()) / temperature)
                probs = probs / probs.sum()
                take = min(remaining_n, len(avail_idx))
                chosen = np.random.choice(len(avail_idx), size=take, replace=False, p=probs)
                selected.update(avail_idx[chosen].tolist())
        elif remaining_n > 0:
            candidates = [i for i in range(len(self)) if i not in selected]
            take = min(remaining_n, len(candidates))
            selected.update(random.sample(candidates, take))

        idxs = list(selected)
        self._ensure_retrieval_counts()
        for i in idxs:
            self.retrieval_counts[i] += 1

        return self._gather(idxs, device)

    def coverage_report(self):
        self._ensure_retrieval_counts()
        report = {}
        for y, idxs in self._by_class.items():
            total_retrievals = sum(self.retrieval_counts[i] for i in idxs)
            report[y] = (total_retrievals, len(idxs))
        return report

    def _gather(self, idxs, device):
        imgs = torch.stack([self.images[i] for i in idxs]).to(device)
        labels = torch.tensor([self.labels[i] for i in idxs]).to(device)
        return imgs, labels


def evaluate(model, test_tasks, seen_task_ids, device):
    model.eval()
    accs = {}
    with torch.no_grad():
        for t in seen_task_ids:
            loader = DataLoader(test_tasks[t], batch_size=256)
            correct, total = 0, 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                pred = model(x).argmax(dim=1)
                correct += (pred == y).sum().item()
                total += y.size(0)
            accs[t] = correct / total
    model.train()
    return accs


def run_condition(condition, train_tasks, test_tasks, device, num_classes=10, epochs=3, batch_size=64, lr=5e-4,
                   mem_batch_size=32, refresh_every=50, seed=0, mem_capacity=40, temperature=0.1,
                   grad_clip=5.0, floor_frac=0.2):
    set_seed(seed)
    model = SmallCNN(num_classes=num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    memory = MemoryBank(per_class_capacity=mem_capacity)

    feature_fn = {"semantic": model.semantic_features, "shallow": model.shallow_features}.get(condition)

    num_tasks = len(train_tasks)
    R = [[None] * num_tasks for _ in range(num_tasks)]

    for t in range(num_tasks):
        loader = DataLoader(train_tasks[t], batch_size=batch_size, shuffle=True)

        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        step = 0
        for epoch in range(epochs):
            epoch_loss_sum = 0.0
            epoch_loss_count = 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)

                replay = None
                if condition == "random":
                    replay = memory.random_batch(mem_batch_size, device)
                elif condition in ("semantic", "shallow"):
                    with torch.no_grad():
                        q_feats = feature_fn(x)
                    replay = memory.similarity_batch_cached(q_feats, condition, mem_batch_size, device,
                                                             temperature=temperature, floor_frac=floor_frac)
                else:
                    raise ValueError(f"unknown condition: {condition}")

                if replay is not None:
                    rx, ry = replay
                    x_all = torch.cat([x, rx], dim=0)
                    y_all = torch.cat([y, ry], dim=0)
                else:
                    x_all, y_all = x, y

                optimizer.zero_grad()
                logits = model(x_all)
                loss = F.cross_entropy(logits, y_all)

                if not torch.isfinite(loss):
                    print(f"  [warning] non-finite loss ({loss.item()}) at step {step} -- skipping this batch")
                    optimizer.zero_grad()
                    step += 1
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
                optimizer.step()

                epoch_loss_sum += loss.item()
                epoch_loss_count += 1

                step += 1
                if feature_fn is not None and step % refresh_every == 0:
                    memory.refresh_embeddings(feature_fn, condition, device)

            avg_epoch_loss = epoch_loss_sum / max(epoch_loss_count, 1)
            print(f"  [{condition}] task {t} epoch {epoch}: avg_loss={avg_epoch_loss:.4f}")

        memory.add_task_data(train_tasks[t])
        if feature_fn is not None:
            memory.refresh_embeddings(feature_fn, condition, device)

        accs = evaluate(model, test_tasks, seen_task_ids=list(range(t + 1)), device=device)
        for j, a in accs.items():
            R[t][j] = a
        print(f"[{condition}] after task {t}: " + ", ".join(f"T{j}={a:.3f}" for j, a in accs.items()))

    if feature_fn is not None:
        report = memory.coverage_report()
        starved = [y for y, (retrievals, n_items) in report.items() if n_items > 0 and retrievals == 0]
        cov_str = ", ".join(f"cls{y}={retrievals}/{n_items}" for y, (retrievals, n_items) in sorted(report.items()))
        print(f"[{condition}] retrieval coverage (times_retrieved/items_stored per class): {cov_str}")
        if starved:
            print(f"[{condition}] WARNING: classes never retrieved even once: {starved}")

    return R


def summarize(R, num_tasks):
    final_row = R[num_tasks - 1]
    avg_acc = np.mean([v for v in final_row if v is not None])

    diffs = []
    for i in range(num_tasks - 1):
        acc_after_last = R[num_tasks - 1][i]
        acc_right_after_i = R[i][i]
        if acc_after_last is not None and acc_right_after_i is not None:
            diffs.append(acc_after_last - acc_right_after_i)
    bwt = np.mean(diffs) if diffs else float("nan")

    return avg_acc, bwt


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--condition", choices=["random", "semantic", "shallow", "all"], default="all")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--quick", action="store_true")
    parser.add_argument("--data-root", type=str, default="./data")
    parser.add_argument("--refresh-every", type=int, default=50)
    parser.add_argument("--seeds", type=int, default=5)
    parser.add_argument("--mem-capacity", type=int, default=40)
    parser.add_argument("--temperature", type=float, default=0.1)
    parser.add_argument("--grad-clip", type=float, default=5.0)
    parser.add_argument("--lr", type=float, default=5e-4)
    parser.add_argument("--floor-frac", type=float, default=0.2)
    parser.add_argument("--dataset", choices=["cifar10", "cifar100"], default="cifar10")
    parser.add_argument("--classes-per-task", type=int, default=None)
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_tasks, test_tasks, num_classes = build_split_dataset(
        dataset_name=args.dataset, data_root=args.data_root,
        classes_per_task=args.classes_per_task, quick=args.quick)
    num_tasks = len(train_tasks)
    print(f"Dataset: {args.dataset} | {num_tasks} tasks | {num_classes} total classes")

    conditions = ["random", "semantic", "shallow"] if args.condition == "all" else [args.condition]

    results = {cond: [] for cond in conditions}
    for cond in conditions:
        for seed in range(args.seeds):
            print(f"\n===== Running condition: {cond} | seed {seed} =====")
            R = run_condition(cond, train_tasks, test_tasks, device, num_classes=num_classes, epochs=args.epochs,
                               refresh_every=args.refresh_every, seed=seed, lr=args.lr,
                               mem_batch_size=32, mem_capacity=args.mem_capacity,
                               temperature=args.temperature, grad_clip=args.grad_clip,
                               floor_frac=args.floor_frac)
            avg_acc, bwt = summarize(R, num_tasks)
            results[cond].append((avg_acc, bwt))
            print(f"[{cond} | seed {seed}] avg_acc={avg_acc:.3f}  BWT={bwt:.3f}")

    print("\n===== Summary (mean +/- std over seeds) =====")
    print(f"{'condition':<10} {'avg_acc':>18} {'BWT':>18}   (BWT closer to 0 = less forgetting)")
    for cond in conditions:
        accs = np.array([r[0] for r in results[cond]])
        bwts = np.array([r[1] for r in results[cond]])
        print(f"{cond:<10} {accs.mean():>8.3f} +/- {accs.std():<6.3f} "
              f"{bwts.mean():>8.3f} +/- {bwts.std():<6.3f}")


if __name__ == "__main__":
    main()

Writing actr_experiment.py


In [2]:
!python actr_experiment.py --data-root /kaggle/working/data --dataset cifar100 --condition random --seeds 1 --epochs 3

Using device: cuda
100%|████████████████████████████████████████| 169M/169M [02:11<00:00, 1.29MB/s]
Dataset: cifar100 | 10 tasks | 100 total classes

===== Running condition: random | seed 0 =====
  [random] task 0 epoch 0: avg_loss=1.6983
  [random] task 0 epoch 1: avg_loss=1.0608
  [random] task 0 epoch 2: avg_loss=0.8128
[random] after task 0: T0=0.701
  [random] task 1 epoch 0: avg_loss=2.4398
  [random] task 1 epoch 1: avg_loss=1.0218
  [random] task 1 epoch 2: avg_loss=0.7127
[random] after task 1: T0=0.404, T1=0.641
  [random] task 2 epoch 0: avg_loss=2.7820
  [random] task 2 epoch 1: avg_loss=1.0944
  [random] task 2 epoch 2: avg_loss=0.7783
[random] after task 2: T0=0.291, T1=0.365, T2=0.710
  [random] task 3 epoch 0: avg_loss=3.3026
  [random] task 3 epoch 1: avg_loss=1.3390
  [random] task 3 epoch 2: avg_loss=1.0100
[random] after task 3: T0=0.252, T1=0.175, T2=0.376, T3=0.637
  [random] task 4 epoch 0: avg_loss=3.8288
  [random] task 4 epoch 1: avg_loss=1.5147
  [random] ta

In [3]:
!python actr_experiment.py --data-root /kaggle/working/data --dataset cifar100 --seeds 3 --epochs 5 --floor-frac 0.2

Using device: cuda
Dataset: cifar100 | 10 tasks | 100 total classes

===== Running condition: random | seed 0 =====
  [random] task 0 epoch 0: avg_loss=1.6972
  [random] task 0 epoch 1: avg_loss=1.0631
  [random] task 0 epoch 2: avg_loss=0.8084
  [random] task 0 epoch 3: avg_loss=0.6421
  [random] task 0 epoch 4: avg_loss=0.4972
[random] after task 0: T0=0.726
  [random] task 1 epoch 0: avg_loss=2.2362
  [random] task 1 epoch 1: avg_loss=0.9064
  [random] task 1 epoch 2: avg_loss=0.6397
  [random] task 1 epoch 3: avg_loss=0.4776
  [random] task 1 epoch 4: avg_loss=0.3538
[random] after task 1: T0=0.371, T1=0.668
  [random] task 2 epoch 0: avg_loss=2.6843
  [random] task 2 epoch 1: avg_loss=0.8596
  [random] task 2 epoch 2: avg_loss=0.5911
  [random] task 2 epoch 3: avg_loss=0.4170
  [random] task 2 epoch 4: avg_loss=0.3023
[random] after task 2: T0=0.262, T1=0.259, T2=0.752
  [random] task 3 epoch 0: avg_loss=3.3948
  [random] task 3 epoch 1: avg_loss=1.1186
  [random] task 3 epoch 2: 

In [4]:
!python actr_experiment.py --data-root /kaggle/working/data --dataset cifar100 --seeds 5 --epochs 5 --floor-frac 0.2

Using device: cuda
Dataset: cifar100 | 10 tasks | 100 total classes

===== Running condition: random | seed 0 =====
  [random] task 0 epoch 0: avg_loss=1.6972
  [random] task 0 epoch 1: avg_loss=1.0631
  [random] task 0 epoch 2: avg_loss=0.8145
  [random] task 0 epoch 3: avg_loss=0.6416
  [random] task 0 epoch 4: avg_loss=0.4928
[random] after task 0: T0=0.716
  [random] task 1 epoch 0: avg_loss=2.2162
  [random] task 1 epoch 1: avg_loss=0.8964
  [random] task 1 epoch 2: avg_loss=0.6292
  [random] task 1 epoch 3: avg_loss=0.4689
  [random] task 1 epoch 4: avg_loss=0.3533
[random] after task 1: T0=0.354, T1=0.665
  [random] task 2 epoch 0: avg_loss=2.6432
  [random] task 2 epoch 1: avg_loss=0.8681
  [random] task 2 epoch 2: avg_loss=0.6024
  [random] task 2 epoch 3: avg_loss=0.4362
  [random] task 2 epoch 4: avg_loss=0.3170
[random] after task 2: T0=0.263, T1=0.290, T2=0.776
  [random] task 3 epoch 0: avg_loss=3.4810
  [random] task 3 epoch 1: avg_loss=1.1686
  [random] task 3 epoch 2: 

In [3]:
!python actr_experiment.py --data-root /kaggle/working/data --dataset cifar100 --condition all --seeds 5 --epochs 5 --floor-frac 0.0

Using device: cuda
100%|████████████████████████████████████████| 169M/169M [02:07<00:00, 1.32MB/s]
Dataset: cifar100 | 10 tasks | 100 total classes

===== Running condition: random | seed 0 =====
  [random] task 0 epoch 0: avg_loss=1.6971
  [random] task 0 epoch 1: avg_loss=1.0548
  [random] task 0 epoch 2: avg_loss=0.8092
  [random] task 0 epoch 3: avg_loss=0.6397
  [random] task 0 epoch 4: avg_loss=0.4942
[random] after task 0: T0=0.729
  [random] task 1 epoch 0: avg_loss=2.1930
  [random] task 1 epoch 1: avg_loss=0.8848
  [random] task 1 epoch 2: avg_loss=0.6190
  [random] task 1 epoch 3: avg_loss=0.4685
  [random] task 1 epoch 4: avg_loss=0.3483
[random] after task 1: T0=0.353, T1=0.668
  [random] task 2 epoch 0: avg_loss=2.6091
  [random] task 2 epoch 1: avg_loss=0.8371
  [random] task 2 epoch 2: avg_loss=0.5814
  [random] task 2 epoch 3: avg_loss=0.4063
  [random] task 2 epoch 4: avg_loss=0.2961
[random] after task 2: T0=0.251, T1=0.259, T2=0.756
  [random] task 3 epoch 0: avg_l

In [1]:
!python actr_experiment.py --data-root /kaggle/working/data --dataset cifar100 --condition all --seeds 1 --epochs 5 --floor-frac 0.0 

python3: can't open file '/kaggle/working/actr_experiment.py': [Errno 2] No such file or directory


In [2]:
!find /kaggle/working -name "actr_experiment.py" -type f

In [5]:
%%writefile actr_diagnostic_experiment.py
#!/usr/bin/env python3
"""Kaggle-ready continual-learning experiment with retrieval diagnostics."""

import importlib.util
import subprocess
import sys


def install_missing_packages():
    packages = {
        "numpy": "numpy",
        "torch": "torch",
        "torchvision": "torchvision",
    }
    missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *missing]
        )


install_missing_packages()

import argparse
import csv
import json
import math
import os
import random
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms


@dataclass
class ReplayEvent:
    task: int
    step: int
    condition: str
    memory_size: int
    selected: int
    query_selected_similarity: float
    selected_pair_similarity: float
    selected_class_entropy: float
    selected_unique_classes: int
    score_std: float
    softmax_effective_fraction: float


class TaskDataset(Dataset):
    def __init__(self, base_dataset, indices: Sequence[int], transform):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform
        self.targets = [int(base_dataset.targets[i]) for i in self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        image, label = self.base_dataset[self.indices[index]]
        return self.transform(image), int(label)


class SmallCifarCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.projector = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
        )
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x, return_features: bool = False):
        shallow_map = self.block1(x)
        shallow = F.adaptive_avg_pool2d(shallow_map, 1).flatten(1)
        semantic = self.projector(self.block3(self.block2(shallow_map)))
        logits = self.classifier(semantic)
        if return_features:
            return logits, shallow, semantic
        return logits


def normalized_entropy(counts: Sequence[int]) -> float:
    values = np.asarray(counts, dtype=np.float64)
    values = values[values > 0]
    if len(values) <= 1:
        return 0.0
    probabilities = values / values.sum()
    return float(-(probabilities * np.log(probabilities)).sum() / np.log(len(values)))


def gini(values: Sequence[float]) -> float:
    array = np.asarray(values, dtype=np.float64)
    if array.size == 0 or np.allclose(array.sum(), 0.0):
        return 0.0
    array = np.sort(np.maximum(array, 0.0))
    index = np.arange(1, len(array) + 1)
    return float(
        (2.0 * np.sum(index * array) / (len(array) * array.sum()))
        - (len(array) + 1.0) / len(array)
    )


def distribution_jsd(left: Sequence[float], right: Sequence[float]) -> float:
    p = np.asarray(left, dtype=np.float64)
    q = np.asarray(right, dtype=np.float64)
    if p.sum() == 0 or q.sum() == 0:
        return 0.0
    p /= p.sum()
    q /= q.sum()
    middle = 0.5 * (p + q)

    def kl_divergence(a, b):
        mask = a > 0
        return float(np.sum(a[mask] * np.log2(a[mask] / b[mask])))

    return 0.5 * kl_divergence(p, middle) + 0.5 * kl_divergence(q, middle)


class ReplayMemory:
    def __init__(
        self,
        per_class_capacity: int,
        temperature: float,
        floor_frac: float,
        seed: int,
        feature_transform: str = "standardize",
        score_normalization: str = "zscore",
    ):
        self.per_class_capacity = per_class_capacity
        self.temperature = temperature
        self.floor_frac = floor_frac
        self.feature_transform = feature_transform
        self.score_normalization = score_normalization
        self.generator = torch.Generator().manual_seed(seed)
        self.images: List[torch.Tensor] = []
        self.labels: List[int] = []
        self.task_ids: List[int] = []
        self.sample_ids: List[str] = []
        self.retrieval_counts: List[int] = []
        self.expected_retrievals: List[float] = []
        self.last_retrieved_step: List[int] = []
        self.shallow_features: Optional[torch.Tensor] = None
        self.semantic_features: Optional[torch.Tensor] = None
        self.shallow_stats: Optional[Dict[str, torch.Tensor]] = None
        self.semantic_stats: Optional[Dict[str, torch.Tensor]] = None
        self.raw_shallow_cosine_mean = float("nan")
        self.raw_semantic_cosine_mean = float("nan")
        self.transformed_shallow_cosine_mean = float("nan")
        self.transformed_semantic_cosine_mean = float("nan")
        self.events: List[ReplayEvent] = []
        self.gradient_cosines: List[float] = []
        self.gradient_current_norms: List[float] = []
        self.gradient_replay_norms: List[float] = []

    def __len__(self):
        return len(self.labels)

    def add_task_data(self, dataset: TaskDataset, task_id: int):
        class_to_local_indices = defaultdict(list)
        for local_index, label in enumerate(dataset.targets):
            class_to_local_indices[label].append(local_index)

        for label in sorted(class_to_local_indices):
            candidates = class_to_local_indices[label]
            permutation = torch.randperm(len(candidates), generator=self.generator).tolist()
            chosen = [candidates[i] for i in permutation[: self.per_class_capacity]]
            for local_index in chosen:
                image, item_label = dataset[local_index]
                source_index = dataset.indices[local_index]
                self.images.append(image.cpu())
                self.labels.append(item_label)
                self.task_ids.append(task_id)
                self.sample_ids.append(f"task{task_id}:source{source_index}")
                self.retrieval_counts.append(0)
                self.expected_retrievals.append(0.0)
                self.last_retrieved_step.append(-1)

        self.shallow_features = None
        self.semantic_features = None
        self.shallow_stats = None
        self.semantic_stats = None

    def _fit_transform_stats(self, raw: torch.Tensor) -> Dict[str, torch.Tensor]:
        mean = raw.mean(dim=0, keepdim=True)
        std = raw.std(dim=0, keepdim=True).clamp_min(1e-6)
        return {"mean": mean, "std": std}

    def _apply_transform(
        self, raw: torch.Tensor, stats: Optional[Dict[str, torch.Tensor]]
    ) -> torch.Tensor:
        if self.feature_transform != "none" and stats is not None:
            raw = raw - stats["mean"]
            if self.feature_transform == "standardize":
                raw = raw / stats["std"]
        return F.normalize(raw, dim=1)

    @torch.no_grad()
    def refresh_features(self, model: nn.Module, device: torch.device, batch_size: int = 256):
        if not self.images:
            return
        was_training = model.training
        model.eval()
        shallow_parts = []
        semantic_parts = []
        for start in range(0, len(self.images), batch_size):
            batch = torch.stack(self.images[start : start + batch_size]).to(device)
            _, shallow, semantic = model(batch, return_features=True)
            shallow_parts.append(shallow.cpu())
            semantic_parts.append(semantic.cpu())
        raw_shallow = torch.cat(shallow_parts)
        raw_semantic = torch.cat(semantic_parts)

        # Post-ReLU pooled activations are non-negative, so raw cosine similarity
        # saturates near 1 and carries almost no ranking signal. Centering (and
        # optionally scaling) restores usable geometry before normalization.
        self.shallow_stats = self._fit_transform_stats(raw_shallow)
        self.semantic_stats = self._fit_transform_stats(raw_semantic)
        self.raw_shallow_cosine_mean = float(
            self._mean_pairwise_cosine(F.normalize(raw_shallow, dim=1))
        )
        self.raw_semantic_cosine_mean = float(
            self._mean_pairwise_cosine(F.normalize(raw_semantic, dim=1))
        )
        self.shallow_features = self._apply_transform(raw_shallow, self.shallow_stats)
        self.semantic_features = self._apply_transform(raw_semantic, self.semantic_stats)
        self.transformed_shallow_cosine_mean = float(
            self._mean_pairwise_cosine(self.shallow_features)
        )
        self.transformed_semantic_cosine_mean = float(
            self._mean_pairwise_cosine(self.semantic_features)
        )
        model.train(was_training)

    @staticmethod
    def _mean_pairwise_cosine(features: torch.Tensor, sample_limit: int = 512) -> float:
        if len(features) < 2:
            return float("nan")
        # Memory is stored in task order, so an evenly spaced stride keeps the
        # estimate representative instead of biasing it toward the earliest task.
        if len(features) > sample_limit:
            positions = torch.linspace(0, len(features) - 1, sample_limit).long()
            subset = features[positions]
        else:
            subset = features
        matrix = subset @ subset.T
        upper = torch.triu_indices(len(subset), len(subset), offset=1)
        return float(matrix[upper[0], upper[1]].mean())

    def _update_expected_counts(self, replay_size: int):
        if not self.labels:
            return
        probability = min(replay_size, len(self.labels)) / len(self.labels)
        for index in range(len(self.labels)):
            self.expected_retrievals[index] += probability

    def _coverage_floor_indices(self, count: int) -> List[int]:
        if count <= 0:
            return []
        by_class = defaultdict(list)
        for index, label in enumerate(self.labels):
            by_class[label].append(index)
        classes = sorted(by_class)
        class_order = torch.randperm(len(classes), generator=self.generator).tolist()
        selected = []
        cursor = 0
        while len(selected) < count:
            label = classes[class_order[cursor % len(classes)]]
            candidates = by_class[label]
            chosen = candidates[
                torch.randint(len(candidates), (1,), generator=self.generator).item()
            ]
            if chosen not in selected:
                selected.append(chosen)
            cursor += 1
            if cursor > count * len(classes) * 4:
                break
        return selected

    def _random_indices(self, replay_size: int) -> List[int]:
        count = min(replay_size, len(self.labels))
        return torch.randperm(len(self.labels), generator=self.generator)[:count].tolist()

    @torch.no_grad()
    def select(
        self,
        condition: str,
        query_images: torch.Tensor,
        model: nn.Module,
        replay_size: int,
        device: torch.device,
        task_id: int,
        global_step: int,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        count = min(replay_size, len(self.labels))
        if count == 0:
            raise RuntimeError("Replay selection requested from an empty memory.")

        self._update_expected_counts(count)
        query_selected_similarity = float("nan")
        score_std = float("nan")
        softmax_effective_fraction = float("nan")

        if condition == "random":
            selected = self._random_indices(count)
            diagnostic_features = self.semantic_features
        else:
            features = (
                self.shallow_features if condition == "shallow" else self.semantic_features
            )
            stats = (
                self.shallow_stats if condition == "shallow" else self.semantic_stats
            )
            if features is None:
                raise RuntimeError("Memory features must be refreshed before similarity replay.")

            was_training = model.training
            model.eval()
            _, query_shallow, query_semantic = model(
                query_images.to(device), return_features=True
            )
            model.train(was_training)
            query_raw = (query_shallow if condition == "shallow" else query_semantic).cpu()
            query_features = self._apply_transform(query_raw, stats)

            # Max-over-query implements cue-driven retrieval without collapsing
            # a heterogeneous current batch into a potentially meaningless centroid.
            similarities = query_features @ features.T
            scores = similarities.max(dim=0).values
            score_std = float(scores.std())
            floor_count = min(count, int(round(count * self.floor_frac)))
            floor_indices = self._coverage_floor_indices(floor_count)
            available_mask = torch.ones(len(self.labels), dtype=torch.bool)
            if floor_indices:
                available_mask[floor_indices] = False
            remaining = count - len(floor_indices)
            available_indices = torch.where(available_mask)[0]
            if remaining:
                available_scores = scores[available_indices]
                # Absolute cosine ranges differ wildly between representations, so a
                # fixed temperature is not comparable across conditions. Z-scoring
                # makes the temperature control selectivity, not feature scale.
                if self.score_normalization == "zscore":
                    available_scores = (
                        available_scores - available_scores.mean()
                    ) / available_scores.std().clamp_min(1e-6)
                probabilities = torch.softmax(
                    available_scores / max(self.temperature, 1e-8), dim=0
                )
                softmax_effective_fraction = float(
                    torch.exp(
                        -(probabilities * torch.log(probabilities.clamp_min(1e-12))).sum()
                    )
                    / len(probabilities)
                )
                sampled_positions = torch.multinomial(
                    probabilities,
                    remaining,
                    replacement=False,
                    generator=self.generator,
                )
                similarity_indices = available_indices[sampled_positions].tolist()
            else:
                similarity_indices = []
            selected = floor_indices + similarity_indices
            query_selected_similarity = float(similarities[:, selected].max(dim=0).values.mean())
            diagnostic_features = features

        for index in selected:
            self.retrieval_counts[index] += 1
            self.last_retrieved_step[index] = global_step

        selected_labels = [self.labels[index] for index in selected]
        label_counts = Counter(selected_labels)
        pair_similarity = float("nan")
        if diagnostic_features is not None and len(selected) > 1:
            chosen_features = F.normalize(diagnostic_features[selected], dim=1)
            similarity_matrix = chosen_features @ chosen_features.T
            upper = torch.triu_indices(len(selected), len(selected), offset=1)
            pair_similarity = float(similarity_matrix[upper[0], upper[1]].mean())

        self.events.append(
            ReplayEvent(
                task=task_id,
                step=global_step,
                condition=condition,
                memory_size=len(self.labels),
                selected=len(selected),
                query_selected_similarity=query_selected_similarity,
                selected_pair_similarity=pair_similarity,
                selected_class_entropy=normalized_entropy(list(label_counts.values())),
                selected_unique_classes=len(label_counts),
                score_std=score_std,
                softmax_effective_fraction=softmax_effective_fraction,
            )
        )
        images = torch.stack([self.images[index] for index in selected]).to(device)
        labels = torch.tensor(
            [self.labels[index] for index in selected], dtype=torch.long, device=device
        )
        return images, labels

    def record_gradient_diagnostics(
        self,
        current_loss: torch.Tensor,
        replay_loss: torch.Tensor,
        model: nn.Module,
    ):
        parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
        current_gradients = torch.autograd.grad(
            current_loss, parameters, retain_graph=True, allow_unused=True
        )
        replay_gradients = torch.autograd.grad(
            replay_loss, parameters, retain_graph=True, allow_unused=True
        )
        current_parts = []
        replay_parts = []
        for current, replay, parameter in zip(
            current_gradients, replay_gradients, parameters
        ):
            current_parts.append(
                current.detach().flatten()
                if current is not None
                else torch.zeros_like(parameter).flatten()
            )
            replay_parts.append(
                replay.detach().flatten()
                if replay is not None
                else torch.zeros_like(parameter).flatten()
            )
        current_vector = torch.cat(current_parts)
        replay_vector = torch.cat(replay_parts)
        current_norm = current_vector.norm()
        replay_norm = replay_vector.norm()
        cosine = F.cosine_similarity(current_vector, replay_vector, dim=0)
        self.gradient_cosines.append(float(cosine))
        self.gradient_current_norms.append(float(current_norm))
        self.gradient_replay_norms.append(float(replay_norm))

    def diagnostics(self, eligible_classes: Sequence[int]) -> Dict:
        eligible = set(eligible_classes)
        observed_by_class = Counter()
        expected_by_class = defaultdict(float)
        item_counts = []
        retrieved_items = 0
        eligible_items = 0
        task_counts = Counter()

        for index, label in enumerate(self.labels):
            if label not in eligible:
                continue
            observed = self.retrieval_counts[index]
            observed_by_class[label] += observed
            expected_by_class[label] += self.expected_retrievals[index]
            item_counts.append(observed)
            eligible_items += 1
            retrieved_items += int(observed > 0)
            task_counts[self.task_ids[index]] += observed

        classes = sorted(eligible)
        observed = np.asarray([observed_by_class[c] for c in classes], dtype=np.float64)
        expected = np.asarray([expected_by_class[c] for c in classes], dtype=np.float64)
        ratio = np.divide(
            observed,
            expected,
            out=np.zeros_like(observed),
            where=expected > 0,
        )
        nonzero_expected = expected > 0
        zero_classes = [
            class_id
            for class_id, obs, exp in zip(classes, observed, expected)
            if exp > 0 and obs == 0
        ]
        ranked_classes = sorted(
            ((class_id, int(observed_by_class[class_id])) for class_id in classes),
            key=lambda item: (item[1], item[0]),
        )
        event_values = {
            field: [
                getattr(event, field)
                for event in self.events
                if not math.isnan(getattr(event, field))
            ]
            for field in (
                "query_selected_similarity",
                "selected_pair_similarity",
                "selected_class_entropy",
                "selected_unique_classes",
                "score_std",
                "softmax_effective_fraction",
            )
        }

        def mean_or_nan(values):
            return float(np.mean(values)) if values else float("nan")

        return {
            "eligible_classes": classes,
            "zero_retrieval_classes": zero_classes,
            "zero_retrieval_class_count": len(zero_classes),
            "least_retrieved_classes": [
                {"class": class_id, "count": count}
                for class_id, count in ranked_classes[:10]
            ],
            "most_retrieved_classes": [
                {"class": class_id, "count": count}
                for class_id, count in reversed(ranked_classes[-10:])
            ],
            "class_retrieval_counts": {
                str(class_id): int(observed_by_class[class_id]) for class_id in classes
            },
            "class_expected_uniform_counts": {
                str(class_id): round(float(expected_by_class[class_id]), 4)
                for class_id in classes
            },
            "class_observed_expected_ratio": {
                str(class_id): round(float(value), 4)
                for class_id, value in zip(classes, ratio)
            },
            "retrieval_count_mean": float(observed.mean()) if len(observed) else 0.0,
            "retrieval_count_std": float(observed.std()) if len(observed) else 0.0,
            "retrieval_count_cv": float(observed.std() / observed.mean())
            if len(observed) and observed.mean() > 0
            else 0.0,
            "retrieval_count_gini": gini(observed),
            "exposure_normalized_ratio_cv": float(ratio[nonzero_expected].std())
            if nonzero_expected.any()
            else 0.0,
            "observed_vs_exposure_expected_jsd": distribution_jsd(observed, expected),
            "sample_coverage_fraction": retrieved_items / max(eligible_items, 1),
            "sample_retrieval_gini": gini(item_counts),
            "retrievals_by_source_task": {
                str(task): int(count) for task, count in sorted(task_counts.items())
            },
            "mean_query_selected_similarity": mean_or_nan(
                event_values["query_selected_similarity"]
            ),
            "mean_selected_pair_similarity": mean_or_nan(
                event_values["selected_pair_similarity"]
            ),
            "mean_batch_class_entropy": mean_or_nan(
                event_values["selected_class_entropy"]
            ),
            "mean_unique_classes_per_replay": mean_or_nan(
                event_values["selected_unique_classes"]
            ),
            "mean_score_std": mean_or_nan(event_values["score_std"]),
            "mean_softmax_effective_fraction": mean_or_nan(
                event_values["softmax_effective_fraction"]
            ),
            "raw_shallow_cosine_mean": self.raw_shallow_cosine_mean,
            "raw_semantic_cosine_mean": self.raw_semantic_cosine_mean,
            "transformed_shallow_cosine_mean": self.transformed_shallow_cosine_mean,
            "transformed_semantic_cosine_mean": self.transformed_semantic_cosine_mean,
            "gradient_cosine_mean": mean_or_nan(self.gradient_cosines),
            "gradient_cosine_std": float(np.std(self.gradient_cosines))
            if self.gradient_cosines
            else float("nan"),
            "gradient_conflict_fraction": float(
                np.mean(np.asarray(self.gradient_cosines) < 0)
            )
            if self.gradient_cosines
            else float("nan"),
            "gradient_current_norm_mean": mean_or_nan(self.gradient_current_norms),
            "gradient_replay_norm_mean": mean_or_nan(self.gradient_replay_norms),
        }


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_worker_init(seed: int):
    def initialize(worker_id: int):
        worker_seed = seed + worker_id
        random.seed(worker_seed)
        np.random.seed(worker_seed)
        torch.manual_seed(worker_seed)

    return initialize


def build_tasks(
    dataset_name: str,
    root: str,
    classes_per_task: int,
    download: bool,
):
    dataset_class = datasets.CIFAR100 if dataset_name == "cifar100" else datasets.CIFAR10
    num_classes = 100 if dataset_name == "cifar100" else 10
    if num_classes % classes_per_task:
        raise ValueError("classes_per_task must divide the dataset's class count.")

    train_base = dataset_class(root=root, train=True, transform=None, download=download)
    test_base = dataset_class(root=root, train=False, transform=None, download=download)
    mean = (0.5071, 0.4867, 0.4408) if dataset_name == "cifar100" else (0.4914, 0.4822, 0.4465)
    std = (0.2675, 0.2565, 0.2761) if dataset_name == "cifar100" else (0.2470, 0.2435, 0.2616)
    train_transform = transforms.Compose(
        [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    )
    evaluation_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean, std)]
    )

    train_targets = np.asarray(train_base.targets)
    test_targets = np.asarray(test_base.targets)
    train_tasks = []
    memory_tasks = []
    test_tasks = []
    for start in range(0, num_classes, classes_per_task):
        classes = np.arange(start, start + classes_per_task)
        train_indices = np.flatnonzero(np.isin(train_targets, classes)).tolist()
        test_indices = np.flatnonzero(np.isin(test_targets, classes)).tolist()
        train_tasks.append(TaskDataset(train_base, train_indices, train_transform))
        memory_tasks.append(TaskDataset(train_base, train_indices, evaluation_transform))
        test_tasks.append(TaskDataset(test_base, test_indices, evaluation_transform))
    return train_tasks, memory_tasks, test_tasks, num_classes


@torch.no_grad()
def evaluate(model, datasets_by_task, device, batch_size, workers):
    model.eval()
    accuracies = []
    for dataset in datasets_by_task:
        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=workers,
            pin_memory=device.type == "cuda",
        )
        correct = 0
        total = 0
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            predictions = model(images).argmax(dim=1)
            correct += int((predictions == labels).sum())
            total += labels.numel()
        accuracies.append(correct / max(total, 1))
    return accuracies


@torch.no_grad()
def feature_quality(memory: ReplayMemory) -> Dict[str, Dict[str, float]]:
    labels = torch.tensor(memory.labels)
    result = {}
    for name, features in (
        ("shallow", memory.shallow_features),
        ("semantic", memory.semantic_features),
    ):
        if features is None or len(features) < 2:
            continue
        normalized = F.normalize(features, dim=1)
        classes = labels.unique(sorted=True)
        centroids = torch.stack(
            [F.normalize(normalized[labels == class_id].mean(0), dim=0) for class_id in classes]
        )
        centroid_predictions = classes[(normalized @ centroids.T).argmax(dim=1)]
        centroid_accuracy = float((centroid_predictions == labels).float().mean())

        within_values = []
        between_values = []
        for class_id in classes:
            mask = labels == class_id
            class_features = normalized[mask]
            centroid = F.normalize(class_features.mean(0), dim=0)
            within_values.append(float((class_features @ centroid).mean()))
        if len(centroids) > 1:
            matrix = centroids @ centroids.T
            upper = torch.triu_indices(len(centroids), len(centroids), offset=1)
            between_values = matrix[upper[0], upper[1]].tolist()
        mean_within = float(np.mean(within_values))
        mean_between = float(np.mean(between_values)) if between_values else 0.0
        result[name] = {
            "nearest_centroid_accuracy": centroid_accuracy,
            "mean_within_class_cosine": mean_within,
            "mean_between_class_centroid_cosine": mean_between,
            "separation_margin": mean_within - mean_between,
        }
    return result


def backward_transfer(accuracy_matrix: Sequence[Sequence[float]]) -> float:
    if len(accuracy_matrix) <= 1:
        return 0.0
    final_row = accuracy_matrix[-1]
    differences = [
        final_row[task] - accuracy_matrix[task][task]
        for task in range(len(accuracy_matrix) - 1)
    ]
    return float(np.mean(differences))


def run_condition(args, condition: str, seed: int, device: torch.device):
    seed_everything(seed)
    train_tasks, memory_tasks, test_tasks, num_classes = build_tasks(
        args.dataset, args.data_root, args.classes_per_task, not args.no_download
    )
    model = SmallCifarCNN(num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    memory = ReplayMemory(
        per_class_capacity=args.memory_per_class,
        temperature=args.temperature,
        floor_frac=args.floor_frac,
        seed=seed + 10_000,
        feature_transform=args.feature_transform,
        score_normalization=args.score_normalization,
    )
    accuracy_matrix = []
    quality_by_task = {}
    global_step = 0

    for task_id, train_dataset in enumerate(train_tasks):
        if not args.no_reset_optimizer and task_id > 0:
            optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
        if len(memory) and (
            memory.shallow_features is None
            or task_id % max(args.refresh_every, 1) == 0
        ):
            memory.refresh_features(model, device)

        loader_generator = torch.Generator().manual_seed(seed * 1000 + task_id)
        train_loader = DataLoader(
            train_dataset,
            batch_size=args.batch_size,
            shuffle=True,
            num_workers=args.workers,
            pin_memory=device.type == "cuda",
            generator=loader_generator,
            worker_init_fn=make_worker_init(seed * 1000 + task_id),
        )
        model.train()
        for epoch in range(args.epochs):
            running_loss = 0.0
            running_current_loss = 0.0
            running_replay_loss = 0.0
            for images, labels in train_loader:
                global_step += 1
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                current_logits = model(images)
                current_loss = F.cross_entropy(current_logits, labels)
                loss = current_loss
                replay_loss_value = 0.0

                if len(memory):
                    replay_images, replay_labels = memory.select(
                        condition=condition,
                        query_images=images,
                        model=model,
                        replay_size=args.replay_batch_size,
                        device=device,
                        task_id=task_id,
                        global_step=global_step,
                    )
                    replay_logits = model(replay_images)
                    replay_loss = F.cross_entropy(replay_logits, replay_labels)
                    if (
                        args.gradient_diagnostics_every > 0
                        and global_step % args.gradient_diagnostics_every == 0
                    ):
                        memory.record_gradient_diagnostics(
                            current_loss, replay_loss, model
                        )
                    loss = current_loss + args.replay_weight * replay_loss
                    replay_loss_value = float(replay_loss.detach())

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
                optimizer.step()
                running_loss += float(loss.detach())
                running_current_loss += float(current_loss.detach())
                running_replay_loss += replay_loss_value

            batches = max(len(train_loader), 1)
            print(
                f"  [{condition}] task {task_id} epoch {epoch}: "
                f"total={running_loss / batches:.4f} "
                f"current={running_current_loss / batches:.4f} "
                f"replay={running_replay_loss / batches:.4f}"
            )

        # The current task enters memory only after its own training. Consequently,
        # the final task is correctly excluded from retrieval-opportunity analysis.
        memory.add_task_data(memory_tasks[task_id], task_id)
        memory.refresh_features(model, device)
        quality_by_task[str(task_id)] = feature_quality(memory)
        accuracies = evaluate(
            model,
            test_tasks[: task_id + 1],
            device,
            args.eval_batch_size,
            args.workers,
        )
        accuracy_matrix.append(accuracies)
        formatted = ", ".join(f"T{i}={value:.3f}" for i, value in enumerate(accuracies))
        print(
            f"[{condition}] after task {task_id}: {formatted} "
            f"| learned_current={accuracies[task_id]:.3f}"
        )

    eligible_classes = list(range(num_classes - args.classes_per_task))
    diagnostics = memory.diagnostics(eligible_classes)
    diagonal = [accuracy_matrix[task][task] for task in range(len(accuracy_matrix))]
    dead_tasks = [task for task, value in enumerate(diagonal) if value < args.plasticity_floor]
    # BWT compares final accuracy against the diagonal. If the diagonal is ~0 the
    # model never learned the task, so BWT trends toward 0 and looks like "less
    # forgetting" when it actually reflects a failure to learn.
    bwt_interpretable = len(dead_tasks) == 0
    health = {
        "diagonal_accuracy": diagonal,
        "mean_diagonal_accuracy": float(np.mean(diagonal)),
        "tasks_below_plasticity_floor": dead_tasks,
        "plasticity_floor": args.plasticity_floor,
        "bwt_interpretable": bwt_interpretable,
        "optimizer_steps_per_task": int(
            math.ceil(len(train_tasks[0]) / args.batch_size) * args.epochs
        ),
    }
    result = {
        "condition": condition,
        "seed": seed,
        "dataset": args.dataset,
        "avg_accuracy": float(np.mean(accuracy_matrix[-1])),
        "bwt": backward_transfer(accuracy_matrix),
        "accuracy_matrix": accuracy_matrix,
        "run_health": health,
        "feature_quality_by_task": quality_by_task,
        "retrieval_diagnostics": diagnostics,
        "config": vars(args),
    }
    print(
        f"[{condition} | seed {seed}] avg_acc={result['avg_accuracy']:.3f} "
        f"BWT={result['bwt']:.3f} "
        f"mean_diag={health['mean_diagonal_accuracy']:.3f}"
    )
    if not bwt_interpretable:
        print(
            f"[{condition}] WARNING: tasks {dead_tasks} never learned "
            f"(accuracy < {args.plasticity_floor}). BWT is NOT interpretable for this run."
        )
    print(f"[{condition}] RETRIEVAL DIAGNOSTICS")
    print(json.dumps(diagnostics, indent=2, allow_nan=True))
    return result, memory.events


def write_outputs(output_dir: Path, results: List[Dict], all_events: List[Dict]):
    output_dir.mkdir(parents=True, exist_ok=True)
    with (output_dir / "results.json").open("w", encoding="utf-8") as handle:
        json.dump(results, handle, indent=2, allow_nan=True)

    if all_events:
        with (output_dir / "replay_events.csv").open(
            "w", newline="", encoding="utf-8"
        ) as handle:
            writer = csv.DictWriter(handle, fieldnames=list(all_events[0]))
            writer.writeheader()
            writer.writerows(all_events)

    summary = []
    for condition in sorted({result["condition"] for result in results}):
        condition_results = [
            result for result in results if result["condition"] == condition
        ]
        accuracy = np.asarray([result["avg_accuracy"] for result in condition_results])
        bwt = np.asarray([result["bwt"] for result in condition_results])
        diagnostic_keys = (
            "retrieval_count_cv",
            "retrieval_count_gini",
            "exposure_normalized_ratio_cv",
            "observed_vs_exposure_expected_jsd",
            "sample_coverage_fraction",
            "sample_retrieval_gini",
            "mean_selected_pair_similarity",
            "mean_batch_class_entropy",
            "mean_unique_classes_per_replay",
            "mean_score_std",
            "mean_softmax_effective_fraction",
            "gradient_cosine_mean",
            "gradient_conflict_fraction",
        )
        row = {
            "condition": condition,
            "seeds": len(condition_results),
            "avg_accuracy_mean": float(accuracy.mean()),
            "avg_accuracy_std": float(accuracy.std()),
            "bwt_mean": float(bwt.mean()),
            "bwt_std": float(bwt.std()),
            "mean_diagonal_accuracy": float(
                np.mean(
                    [
                        result["run_health"]["mean_diagonal_accuracy"]
                        for result in condition_results
                    ]
                )
            ),
            "runs_with_interpretable_bwt": int(
                sum(
                    1
                    for result in condition_results
                    if result["run_health"]["bwt_interpretable"]
                )
            ),
        }
        for key in diagnostic_keys:
            values = np.asarray(
                [
                    result["retrieval_diagnostics"][key]
                    for result in condition_results
                ],
                dtype=np.float64,
            )
            row[f"{key}_mean"] = float(np.nanmean(values))
            row[f"{key}_std"] = float(np.nanstd(values))
        summary.append(row)

    with (output_dir / "summary.csv").open(
        "w", newline="", encoding="utf-8"
    ) as handle:
        writer = csv.DictWriter(handle, fieldnames=list(summary[0]))
        writer.writeheader()
        writer.writerows(summary)

    summary_by_condition = {row["condition"]: row for row in summary}

    print("\n===== Summary (mean +/- std over seeds) =====")
    for row in summary:
        flag = "" if row["runs_with_interpretable_bwt"] == row["seeds"] else "  [BWT UNRELIABLE]"
        print(
            f"{row['condition']:10s} "
            f"avg_acc={row['avg_accuracy_mean']:.3f} +/- {row['avg_accuracy_std']:.3f}  "
            f"BWT={row['bwt_mean']:.3f} +/- {row['bwt_std']:.3f}  "
            f"mean_diag={row['mean_diagonal_accuracy']:.3f}  "
            f"exposure-JSD={row['observed_vs_exposure_expected_jsd_mean']:.4f}  "
            f"sample-coverage={row['sample_coverage_fraction_mean']:.3f}  "
            f"grad-conflict={row['gradient_conflict_fraction_mean']:.3f}{flag}"
        )

    diagnosis = {
        "status": "insufficient_conditions",
        "interpretation": [
            "Run both random and shallow conditions to generate a direct diagnosis."
        ],
    }
    if "random" in summary_by_condition and "shallow" in summary_by_condition:
        random_row = summary_by_condition["random"]
        shallow_row = summary_by_condition["shallow"]
        findings = []

        degenerate = [
            row["condition"]
            for row in summary
            if row["runs_with_interpretable_bwt"] < row["seeds"]
        ]
        if degenerate:
            diagnosis = {
                "status": "invalid_run",
                "degenerate_conditions": degenerate,
                "interpretation": [
                    "One or more conditions failed to learn some tasks, so their "
                    "diagonal accuracy is ~0 and BWT is driven by lack of plasticity "
                    "rather than by forgetting.",
                    "Fix plasticity first (smaller --batch-size, more --epochs, or a "
                    "lower --replay-weight), then compare retrieval conditions.",
                ],
            }
            with (output_dir / "diagnosis.json").open("w", encoding="utf-8") as handle:
                json.dump(diagnosis, handle, indent=2, allow_nan=True)
            print("\n===== Diagnosis =====")
            print(json.dumps(diagnosis, indent=2))
            return

        random_jsd = random_row["observed_vs_exposure_expected_jsd_mean"]
        shallow_jsd = shallow_row["observed_vs_exposure_expected_jsd_mean"]
        random_coverage = random_row["sample_coverage_fraction_mean"]
        shallow_coverage = shallow_row["sample_coverage_fraction_mean"]
        if shallow_jsd > random_jsd * 1.25 and shallow_coverage < random_coverage - 0.05:
            findings.append(
                {
                    "mechanism": "coverage_bias",
                    "supported": True,
                    "reason": (
                        "Shallow retrieval departs more from its exposure-adjusted "
                        "uniform baseline and reaches fewer stored samples than random."
                    ),
                }
            )
        else:
            findings.append(
                {
                    "mechanism": "coverage_bias",
                    "supported": False,
                    "reason": (
                        "Shallow retrieval does not show both a materially larger "
                        "exposure-adjusted divergence and lower sample coverage."
                    ),
                }
            )

        random_pair = random_row["mean_selected_pair_similarity_mean"]
        shallow_pair = shallow_row["mean_selected_pair_similarity_mean"]
        findings.append(
            {
                "mechanism": "redundant_similarity_replay",
                "supported": bool(
                    np.isfinite(random_pair)
                    and np.isfinite(shallow_pair)
                    and shallow_pair > random_pair + 0.05
                ),
                "reason": (
                    "Selected shallow batches are more internally similar than random."
                    if np.isfinite(random_pair)
                    and np.isfinite(shallow_pair)
                    and shallow_pair > random_pair + 0.05
                    else "Shallow batches are not materially more redundant than random."
                ),
            }
        )

        random_conflict = random_row["gradient_conflict_fraction_mean"]
        shallow_conflict = shallow_row["gradient_conflict_fraction_mean"]
        findings.append(
            {
                "mechanism": "gradient_interference",
                "supported": bool(
                    np.isfinite(random_conflict)
                    and np.isfinite(shallow_conflict)
                    and shallow_conflict > random_conflict + 0.05
                ),
                "reason": (
                    "Shallow replay gradients conflict with current-task gradients more "
                    "often than random replay gradients."
                    if np.isfinite(random_conflict)
                    and np.isfinite(shallow_conflict)
                    and shallow_conflict > random_conflict + 0.05
                    else "Shallow replay does not show materially more gradient conflict."
                ),
            }
        )
        shallow_effective = shallow_row["mean_softmax_effective_fraction_mean"]
        semantic_effective = summary_by_condition.get("semantic", {}).get(
            "mean_softmax_effective_fraction_mean", float("nan")
        )
        findings.append(
            {
                "mechanism": "non_selective_retrieval",
                "supported": bool(
                    np.isfinite(shallow_effective) and shallow_effective > 0.5
                ),
                "shallow_softmax_effective_fraction": shallow_effective,
                "semantic_softmax_effective_fraction": semantic_effective,
                "reason": (
                    "The shallow retrieval distribution stays close to uniform, so "
                    "'shallow' is effectively random replay with extra compute."
                    if np.isfinite(shallow_effective) and shallow_effective > 0.5
                    else "Shallow retrieval is meaningfully peaked, so it is genuinely selective."
                ),
            }
        )
        diagnosis = {
            "status": "diagnosed",
            "thresholds": {
                "coverage_jsd_multiplier": 1.25,
                "sample_coverage_absolute_gap": 0.05,
                "pair_similarity_absolute_gap": 0.05,
                "gradient_conflict_absolute_gap": 0.05,
            },
            "findings": findings,
            "important_note": (
                "Raw class retrieval totals are age-confounded. Prefer the "
                "exposure-normalized JSD and observed/expected ratios when deciding "
                "whether similarity replay is biased."
            ),
        }
    with (output_dir / "diagnosis.json").open("w", encoding="utf-8") as handle:
        json.dump(diagnosis, handle, indent=2, allow_nan=True)

    print("\n===== Diagnosis =====")
    print(json.dumps(diagnosis, indent=2, allow_nan=True))


def parse_arguments():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--condition",
        choices=("random", "semantic", "shallow", "all"),
        default="all",
    )
    parser.add_argument("--dataset", choices=("cifar10", "cifar100"), default="cifar100")
    parser.add_argument("--data-root", default="/kaggle/working/data")
    parser.add_argument("--output-dir", default="/kaggle/working/actr_diagnostics")
    parser.add_argument("--seeds", type=int, default=1)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--classes-per-task", type=int, default=10)
    parser.add_argument("--memory-per-class", type=int, default=40)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--eval-batch-size", type=int, default=256)
    parser.add_argument("--replay-batch-size", type=int, default=32)
    parser.add_argument("--replay-weight", type=float, default=1.0)
    parser.add_argument("--temperature", type=float, default=0.1)
    parser.add_argument("--floor-frac", type=float, default=0.0)
    parser.add_argument("--refresh-every", type=int, default=1)
    parser.add_argument("--gradient-diagnostics-every", type=int, default=25)
    parser.add_argument(
        "--feature-transform",
        choices=("none", "center", "standardize"),
        default="standardize",
    )
    parser.add_argument(
        "--score-normalization", choices=("none", "zscore"), default="zscore"
    )
    parser.add_argument("--plasticity-floor", type=float, default=0.05)
    parser.add_argument("--grad-clip", type=float, default=5.0)
    parser.add_argument("--lr", type=float, default=5e-4)
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument("--no-reset-optimizer", action="store_true")
    parser.add_argument("--no-download", action="store_true")
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"Ignoring notebook arguments: {unknown}")
    if not 0.0 <= args.floor_frac <= 1.0:
        parser.error("--floor-frac must be between 0 and 1.")
    if args.seeds < 1 or args.epochs < 1:
        parser.error("--seeds and --epochs must be positive.")
    return args


def main():
    args = parse_arguments()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    conditions = (
        ["random", "semantic", "shallow"]
        if args.condition == "all"
        else [args.condition]
    )
    results = []
    all_events = []
    for condition in conditions:
        for seed in range(args.seeds):
            print(f"\n===== Running condition: {condition} | seed {seed} =====")
            result, events = run_condition(args, condition, seed, device)
            results.append(result)
            all_events.extend(
                {
                    "condition": condition,
                    "seed": seed,
                    **asdict(event),
                }
                for event in events
            )
    write_outputs(Path(args.output_dir), results, all_events)


if __name__ == "__main__":
    main()


Writing actr_diagnostic_experiment.py


In [6]:
!python actr_diagnostic_experiment.py \
  --data-root /kaggle/working/data \
  --output-dir /kaggle/working/actr_diagnostics \
  --dataset cifar100 --condition all --seeds 1 --epochs 5

Using device: cuda

===== Running condition: random | seed 0 =====
100%|████████████████████████████████████████| 169M/169M [34:32<00:00, 81.6kB/s]
  [random] task 0 epoch 0: total=2.1336 current=2.1336 replay=0.0000
  [random] task 0 epoch 1: total=1.5204 current=1.5204 replay=0.0000
  [random] task 0 epoch 2: total=1.2464 current=1.2464 replay=0.0000
  [random] task 0 epoch 3: total=1.1048 current=1.1048 replay=0.0000
  [random] task 0 epoch 4: total=1.0038 current=1.0038 replay=0.0000
[random] after task 0: T0=0.656 | learned_current=0.656
  [random] task 1 epoch 0: total=4.1904 current=3.1857 replay=1.0048
  [random] task 1 epoch 1: total=2.0064 current=1.6137 replay=0.3926
  [random] task 1 epoch 2: total=1.4776 current=1.2985 replay=0.1791
  [random] task 1 epoch 3: total=1.3199 current=1.2033 replay=0.1165
  [random] task 1 epoch 4: total=1.1627 current=1.0841 replay=0.0786
[random] after task 1: T0=0.425, T1=0.361 | learned_current=0.361
  [random] task 2 epoch 0: total=4.6609 

In [1]:
%%writefile actr_diagnostic_experiment.py
#!/usr/bin/env python3
"""Kaggle-ready continual-learning experiment with retrieval diagnostics."""

import importlib.util
import subprocess
import sys


def install_missing_packages():
    packages = {
        "numpy": "numpy",
        "torch": "torch",
        "torchvision": "torchvision",
    }
    missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *missing]
        )


install_missing_packages()

import argparse
import csv
import json
import math
import os
import random
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms


@dataclass
class ReplayEvent:
    task: int
    step: int
    condition: str
    memory_size: int
    selected: int
    query_selected_similarity: float
    selected_pair_similarity: float
    selected_class_entropy: float
    selected_unique_classes: int
    score_std: float
    softmax_effective_fraction: float


class TaskDataset(Dataset):
    def __init__(self, base_dataset, indices: Sequence[int], transform):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform
        self.targets = [int(base_dataset.targets[i]) for i in self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        image, label = self.base_dataset[self.indices[index]]
        return self.transform(image), int(label)


class SmallCifarCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.projector = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
        )
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x, return_features: bool = False):
        shallow_map = self.block1(x)
        shallow = F.adaptive_avg_pool2d(shallow_map, 1).flatten(1)
        semantic = self.projector(self.block3(self.block2(shallow_map)))
        logits = self.classifier(semantic)
        if return_features:
            return logits, shallow, semantic
        return logits


def normalized_entropy(counts: Sequence[int]) -> float:
    values = np.asarray(counts, dtype=np.float64)
    values = values[values > 0]
    if len(values) <= 1:
        return 0.0
    probabilities = values / values.sum()
    return float(-(probabilities * np.log(probabilities)).sum() / np.log(len(values)))


def gini(values: Sequence[float]) -> float:
    array = np.asarray(values, dtype=np.float64)
    if array.size == 0 or np.allclose(array.sum(), 0.0):
        return 0.0
    array = np.sort(np.maximum(array, 0.0))
    index = np.arange(1, len(array) + 1)
    return float(
        (2.0 * np.sum(index * array) / (len(array) * array.sum()))
        - (len(array) + 1.0) / len(array)
    )


def distribution_jsd(left: Sequence[float], right: Sequence[float]) -> float:
    p = np.asarray(left, dtype=np.float64)
    q = np.asarray(right, dtype=np.float64)
    if p.sum() == 0 or q.sum() == 0:
        return 0.0
    p /= p.sum()
    q /= q.sum()
    middle = 0.5 * (p + q)

    def kl_divergence(a, b):
        mask = a > 0
        return float(np.sum(a[mask] * np.log2(a[mask] / b[mask])))

    return 0.5 * kl_divergence(p, middle) + 0.5 * kl_divergence(q, middle)


class ReplayMemory:
    def __init__(
        self,
        per_class_capacity: int,
        temperature: float,
        floor_frac: float,
        seed: int,
        feature_transform: str = "standardize",
        score_normalization: str = "zscore",
        actr_feature: str = "shallow",
        actr_alpha: float = 1.0,
        actr_beta: float = -1.0,
        actr_gamma: float = 1.0,
        actr_decay: float = 0.5,
        actr_noise: float = 0.0,
    ):
        self.per_class_capacity = per_class_capacity
        self.temperature = temperature
        self.floor_frac = floor_frac
        self.feature_transform = feature_transform
        self.score_normalization = score_normalization
        self.actr_feature = actr_feature
        self.actr_alpha = actr_alpha
        self.actr_beta = actr_beta
        self.actr_gamma = actr_gamma
        self.actr_decay = actr_decay
        self.actr_noise = actr_noise
        self.generator = torch.Generator().manual_seed(seed)
        self.images: List[torch.Tensor] = []
        self.labels: List[int] = []
        self.task_ids: List[int] = []
        self.sample_ids: List[str] = []
        self.retrieval_counts: List[int] = []
        self.expected_retrievals: List[float] = []
        self.last_retrieved_step: List[int] = []
        self.creation_step: List[int] = []
        self.class_retrieval_counts: Counter = Counter()
        self.shallow_features: Optional[torch.Tensor] = None
        self.semantic_features: Optional[torch.Tensor] = None
        self.shallow_stats: Optional[Dict[str, torch.Tensor]] = None
        self.semantic_stats: Optional[Dict[str, torch.Tensor]] = None
        self.raw_shallow_cosine_mean = float("nan")
        self.raw_semantic_cosine_mean = float("nan")
        self.transformed_shallow_cosine_mean = float("nan")
        self.transformed_semantic_cosine_mean = float("nan")
        self.events: List[ReplayEvent] = []
        self.gradient_cosines: List[float] = []
        self.gradient_current_norms: List[float] = []
        self.gradient_replay_norms: List[float] = []

    def __len__(self):
        return len(self.labels)

    def add_task_data(self, dataset: TaskDataset, task_id: int, current_step: int = 0):
        class_to_local_indices = defaultdict(list)
        for local_index, label in enumerate(dataset.targets):
            class_to_local_indices[label].append(local_index)

        for label in sorted(class_to_local_indices):
            candidates = class_to_local_indices[label]
            permutation = torch.randperm(len(candidates), generator=self.generator).tolist()
            chosen = [candidates[i] for i in permutation[: self.per_class_capacity]]
            for local_index in chosen:
                image, item_label = dataset[local_index]
                source_index = dataset.indices[local_index]
                self.images.append(image.cpu())
                self.labels.append(item_label)
                self.task_ids.append(task_id)
                self.sample_ids.append(f"task{task_id}:source{source_index}")
                self.retrieval_counts.append(0)
                self.expected_retrievals.append(0.0)
                self.last_retrieved_step.append(-1)
                self.creation_step.append(current_step)

        self.shallow_features = None
        self.semantic_features = None
        self.shallow_stats = None
        self.semantic_stats = None

    def _fit_transform_stats(self, raw: torch.Tensor) -> Dict[str, torch.Tensor]:
        mean = raw.mean(dim=0, keepdim=True)
        std = raw.std(dim=0, keepdim=True).clamp_min(1e-6)
        return {"mean": mean, "std": std}

    def _apply_transform(
        self, raw: torch.Tensor, stats: Optional[Dict[str, torch.Tensor]]
    ) -> torch.Tensor:
        if self.feature_transform != "none" and stats is not None:
            raw = raw - stats["mean"]
            if self.feature_transform == "standardize":
                raw = raw / stats["std"]
        return F.normalize(raw, dim=1)

    @torch.no_grad()
    def refresh_features(self, model: nn.Module, device: torch.device, batch_size: int = 256):
        if not self.images:
            return
        was_training = model.training
        model.eval()
        shallow_parts = []
        semantic_parts = []
        for start in range(0, len(self.images), batch_size):
            batch = torch.stack(self.images[start : start + batch_size]).to(device)
            _, shallow, semantic = model(batch, return_features=True)
            shallow_parts.append(shallow.cpu())
            semantic_parts.append(semantic.cpu())
        raw_shallow = torch.cat(shallow_parts)
        raw_semantic = torch.cat(semantic_parts)

        # Post-ReLU pooled activations are non-negative, so raw cosine similarity
        # saturates near 1 and carries almost no ranking signal. Centering (and
        # optionally scaling) restores usable geometry before normalization.
        self.shallow_stats = self._fit_transform_stats(raw_shallow)
        self.semantic_stats = self._fit_transform_stats(raw_semantic)
        self.raw_shallow_cosine_mean = float(
            self._mean_pairwise_cosine(F.normalize(raw_shallow, dim=1))
        )
        self.raw_semantic_cosine_mean = float(
            self._mean_pairwise_cosine(F.normalize(raw_semantic, dim=1))
        )
        self.shallow_features = self._apply_transform(raw_shallow, self.shallow_stats)
        self.semantic_features = self._apply_transform(raw_semantic, self.semantic_stats)
        self.transformed_shallow_cosine_mean = float(
            self._mean_pairwise_cosine(self.shallow_features)
        )
        self.transformed_semantic_cosine_mean = float(
            self._mean_pairwise_cosine(self.semantic_features)
        )
        model.train(was_training)

    @staticmethod
    def _mean_pairwise_cosine(features: torch.Tensor, sample_limit: int = 512) -> float:
        if len(features) < 2:
            return float("nan")
        # Memory is stored in task order, so an evenly spaced stride keeps the
        # estimate representative instead of biasing it toward the earliest task.
        if len(features) > sample_limit:
            positions = torch.linspace(0, len(features) - 1, sample_limit).long()
            subset = features[positions]
        else:
            subset = features
        matrix = subset @ subset.T
        upper = torch.triu_indices(len(subset), len(subset), offset=1)
        return float(matrix[upper[0], upper[1]].mean())

    def _update_expected_counts(self, replay_size: int):
        if not self.labels:
            return
        probability = min(replay_size, len(self.labels)) / len(self.labels)
        for index in range(len(self.labels)):
            self.expected_retrievals[index] += probability

    def _coverage_floor_indices(self, count: int) -> List[int]:
        if count <= 0:
            return []
        by_class = defaultdict(list)
        for index, label in enumerate(self.labels):
            by_class[label].append(index)
        classes = sorted(by_class)
        class_order = torch.randperm(len(classes), generator=self.generator).tolist()
        selected = []
        cursor = 0
        while len(selected) < count:
            label = classes[class_order[cursor % len(classes)]]
            candidates = by_class[label]
            chosen = candidates[
                torch.randint(len(candidates), (1,), generator=self.generator).item()
            ]
            if chosen not in selected:
                selected.append(chosen)
            cursor += 1
            if cursor > count * len(classes) * 4:
                break
        return selected

    def _random_indices(self, replay_size: int) -> List[int]:
        count = min(replay_size, len(self.labels))
        return torch.randperm(len(self.labels), generator=self.generator)[:count].tolist()

    def _base_level_activation(self, global_step: int) -> torch.Tensor:
        # ACT-R optimized-learning approximation:
        #   B_i = ln(n_i / (1 - d)) - d * ln(T_i)
        # n_i counts the encoding event plus every retrieval; T_i is the age of
        # the trace. High B_i means the item is already highly accessible.
        counts = torch.tensor(self.retrieval_counts, dtype=torch.float32) + 1.0
        age = (
            torch.full((len(self.labels),), float(global_step))
            - torch.tensor(self.creation_step, dtype=torch.float32)
        ).clamp_min(1.0)
        return torch.log(counts / (1.0 - self.actr_decay)) - self.actr_decay * torch.log(age)

    def _class_overrepresentation(self) -> torch.Tensor:
        # How much each item's class has been replayed relative to a uniform
        # share, expressed per item so it can be combined with the other terms.
        labels = torch.tensor(self.labels, dtype=torch.long)
        total = float(sum(self.class_retrieval_counts.values()))
        distinct = len(set(self.labels))
        if total <= 0 or distinct == 0:
            return torch.zeros(len(self.labels), dtype=torch.float32)
        uniform_share = total / distinct
        per_class = torch.tensor(
            [self.class_retrieval_counts.get(int(label), 0) for label in labels],
            dtype=torch.float32,
        )
        return per_class / uniform_share

    @staticmethod
    def _zscore(values: torch.Tensor) -> torch.Tensor:
        return (values - values.mean()) / values.std().clamp_min(1e-6)

    @torch.no_grad()
    def select(
        self,
        condition: str,
        query_images: torch.Tensor,
        model: nn.Module,
        replay_size: int,
        device: torch.device,
        task_id: int,
        global_step: int,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        count = min(replay_size, len(self.labels))
        if count == 0:
            raise RuntimeError("Replay selection requested from an empty memory.")

        self._update_expected_counts(count)
        query_selected_similarity = float("nan")
        score_std = float("nan")
        softmax_effective_fraction = float("nan")

        if condition == "random":
            selected = self._random_indices(count)
            diagnostic_features = self.semantic_features
        else:
            if condition == "shallow":
                feature_name = "shallow"
            elif condition == "semantic":
                feature_name = "semantic"
            else:
                feature_name = self.actr_feature
            features = (
                self.shallow_features if feature_name == "shallow" else self.semantic_features
            )
            stats = (
                self.shallow_stats if feature_name == "shallow" else self.semantic_stats
            )
            if features is None:
                raise RuntimeError("Memory features must be refreshed before similarity replay.")

            was_training = model.training
            model.eval()
            _, query_shallow, query_semantic = model(
                query_images.to(device), return_features=True
            )
            model.train(was_training)
            query_raw = (query_shallow if feature_name == "shallow" else query_semantic).cpu()
            query_features = self._apply_transform(query_raw, stats)

            # Max-over-query implements cue-driven retrieval without collapsing
            # a heterogeneous current batch into a potentially meaningless centroid.
            similarities = query_features @ features.T
            scores = similarities.max(dim=0).values

            if condition == "actr":
                # Cue relevance alone concentrates replay on a small subset. The
                # base-level and overrepresentation terms restore coverage by
                # discounting traces that are already accessible or already
                # over-replayed at the class level.
                scores = (
                    self.actr_alpha * self._zscore(scores)
                    + self.actr_beta * self._zscore(self._base_level_activation(global_step))
                    - self.actr_gamma * self._zscore(self._class_overrepresentation())
                )
                if self.actr_noise > 0:
                    uniform = torch.rand(
                        len(scores), generator=self.generator, dtype=torch.float32
                    ).clamp(1e-6, 1 - 1e-6)
                    scores = scores + self.actr_noise * torch.log(uniform / (1 - uniform))

            score_std = float(scores.std())
            floor_count = min(count, int(round(count * self.floor_frac)))
            floor_indices = self._coverage_floor_indices(floor_count)
            available_mask = torch.ones(len(self.labels), dtype=torch.bool)
            if floor_indices:
                available_mask[floor_indices] = False
            remaining = count - len(floor_indices)
            available_indices = torch.where(available_mask)[0]
            if remaining:
                available_scores = scores[available_indices]
                # Absolute cosine ranges differ wildly between representations, so a
                # fixed temperature is not comparable across conditions. Z-scoring
                # makes the temperature control selectivity, not feature scale.
                if self.score_normalization == "zscore":
                    available_scores = (
                        available_scores - available_scores.mean()
                    ) / available_scores.std().clamp_min(1e-6)
                probabilities = torch.softmax(
                    available_scores / max(self.temperature, 1e-8), dim=0
                )
                softmax_effective_fraction = float(
                    torch.exp(
                        -(probabilities * torch.log(probabilities.clamp_min(1e-12))).sum()
                    )
                    / len(probabilities)
                )
                sampled_positions = torch.multinomial(
                    probabilities,
                    remaining,
                    replacement=False,
                    generator=self.generator,
                )
                similarity_indices = available_indices[sampled_positions].tolist()
            else:
                similarity_indices = []
            selected = floor_indices + similarity_indices
            query_selected_similarity = float(similarities[:, selected].max(dim=0).values.mean())
            diagnostic_features = features

        for index in selected:
            self.retrieval_counts[index] += 1
            self.last_retrieved_step[index] = global_step
            self.class_retrieval_counts[self.labels[index]] += 1

        selected_labels = [self.labels[index] for index in selected]
        label_counts = Counter(selected_labels)
        pair_similarity = float("nan")
        if diagnostic_features is not None and len(selected) > 1:
            chosen_features = F.normalize(diagnostic_features[selected], dim=1)
            similarity_matrix = chosen_features @ chosen_features.T
            upper = torch.triu_indices(len(selected), len(selected), offset=1)
            pair_similarity = float(similarity_matrix[upper[0], upper[1]].mean())

        self.events.append(
            ReplayEvent(
                task=task_id,
                step=global_step,
                condition=condition,
                memory_size=len(self.labels),
                selected=len(selected),
                query_selected_similarity=query_selected_similarity,
                selected_pair_similarity=pair_similarity,
                selected_class_entropy=normalized_entropy(list(label_counts.values())),
                selected_unique_classes=len(label_counts),
                score_std=score_std,
                softmax_effective_fraction=softmax_effective_fraction,
            )
        )
        images = torch.stack([self.images[index] for index in selected]).to(device)
        labels = torch.tensor(
            [self.labels[index] for index in selected], dtype=torch.long, device=device
        )
        return images, labels

    def record_gradient_diagnostics(
        self,
        current_loss: torch.Tensor,
        replay_loss: torch.Tensor,
        model: nn.Module,
    ):
        parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
        current_gradients = torch.autograd.grad(
            current_loss, parameters, retain_graph=True, allow_unused=True
        )
        replay_gradients = torch.autograd.grad(
            replay_loss, parameters, retain_graph=True, allow_unused=True
        )
        current_parts = []
        replay_parts = []
        for current, replay, parameter in zip(
            current_gradients, replay_gradients, parameters
        ):
            current_parts.append(
                current.detach().flatten()
                if current is not None
                else torch.zeros_like(parameter).flatten()
            )
            replay_parts.append(
                replay.detach().flatten()
                if replay is not None
                else torch.zeros_like(parameter).flatten()
            )
        current_vector = torch.cat(current_parts)
        replay_vector = torch.cat(replay_parts)
        current_norm = current_vector.norm()
        replay_norm = replay_vector.norm()
        cosine = F.cosine_similarity(current_vector, replay_vector, dim=0)
        self.gradient_cosines.append(float(cosine))
        self.gradient_current_norms.append(float(current_norm))
        self.gradient_replay_norms.append(float(replay_norm))

    def diagnostics(self, eligible_classes: Sequence[int]) -> Dict:
        eligible = set(eligible_classes)
        observed_by_class = Counter()
        expected_by_class = defaultdict(float)
        item_counts = []
        retrieved_items = 0
        eligible_items = 0
        task_counts = Counter()

        for index, label in enumerate(self.labels):
            if label not in eligible:
                continue
            observed = self.retrieval_counts[index]
            observed_by_class[label] += observed
            expected_by_class[label] += self.expected_retrievals[index]
            item_counts.append(observed)
            eligible_items += 1
            retrieved_items += int(observed > 0)
            task_counts[self.task_ids[index]] += observed

        classes = sorted(eligible)
        observed = np.asarray([observed_by_class[c] for c in classes], dtype=np.float64)
        expected = np.asarray([expected_by_class[c] for c in classes], dtype=np.float64)
        ratio = np.divide(
            observed,
            expected,
            out=np.zeros_like(observed),
            where=expected > 0,
        )
        nonzero_expected = expected > 0
        zero_classes = [
            class_id
            for class_id, obs, exp in zip(classes, observed, expected)
            if exp > 0 and obs == 0
        ]
        ranked_classes = sorted(
            ((class_id, int(observed_by_class[class_id])) for class_id in classes),
            key=lambda item: (item[1], item[0]),
        )
        event_values = {
            field: [
                getattr(event, field)
                for event in self.events
                if not math.isnan(getattr(event, field))
            ]
            for field in (
                "query_selected_similarity",
                "selected_pair_similarity",
                "selected_class_entropy",
                "selected_unique_classes",
                "score_std",
                "softmax_effective_fraction",
            )
        }

        def mean_or_nan(values):
            return float(np.mean(values)) if values else float("nan")

        return {
            "eligible_classes": classes,
            "zero_retrieval_classes": zero_classes,
            "zero_retrieval_class_count": len(zero_classes),
            "least_retrieved_classes": [
                {"class": class_id, "count": count}
                for class_id, count in ranked_classes[:10]
            ],
            "most_retrieved_classes": [
                {"class": class_id, "count": count}
                for class_id, count in reversed(ranked_classes[-10:])
            ],
            "class_retrieval_counts": {
                str(class_id): int(observed_by_class[class_id]) for class_id in classes
            },
            "class_expected_uniform_counts": {
                str(class_id): round(float(expected_by_class[class_id]), 4)
                for class_id in classes
            },
            "class_observed_expected_ratio": {
                str(class_id): round(float(value), 4)
                for class_id, value in zip(classes, ratio)
            },
            "retrieval_count_mean": float(observed.mean()) if len(observed) else 0.0,
            "retrieval_count_std": float(observed.std()) if len(observed) else 0.0,
            "retrieval_count_cv": float(observed.std() / observed.mean())
            if len(observed) and observed.mean() > 0
            else 0.0,
            "retrieval_count_gini": gini(observed),
            "exposure_normalized_ratio_cv": float(ratio[nonzero_expected].std())
            if nonzero_expected.any()
            else 0.0,
            "observed_vs_exposure_expected_jsd": distribution_jsd(observed, expected),
            "sample_coverage_fraction": retrieved_items / max(eligible_items, 1),
            "sample_retrieval_gini": gini(item_counts),
            "retrievals_by_source_task": {
                str(task): int(count) for task, count in sorted(task_counts.items())
            },
            "mean_query_selected_similarity": mean_or_nan(
                event_values["query_selected_similarity"]
            ),
            "mean_selected_pair_similarity": mean_or_nan(
                event_values["selected_pair_similarity"]
            ),
            "mean_batch_class_entropy": mean_or_nan(
                event_values["selected_class_entropy"]
            ),
            "mean_unique_classes_per_replay": mean_or_nan(
                event_values["selected_unique_classes"]
            ),
            "mean_score_std": mean_or_nan(event_values["score_std"]),
            "mean_softmax_effective_fraction": mean_or_nan(
                event_values["softmax_effective_fraction"]
            ),
            "raw_shallow_cosine_mean": self.raw_shallow_cosine_mean,
            "raw_semantic_cosine_mean": self.raw_semantic_cosine_mean,
            "transformed_shallow_cosine_mean": self.transformed_shallow_cosine_mean,
            "transformed_semantic_cosine_mean": self.transformed_semantic_cosine_mean,
            "gradient_cosine_mean": mean_or_nan(self.gradient_cosines),
            "gradient_cosine_std": float(np.std(self.gradient_cosines))
            if self.gradient_cosines
            else float("nan"),
            "gradient_conflict_fraction": float(
                np.mean(np.asarray(self.gradient_cosines) < 0)
            )
            if self.gradient_cosines
            else float("nan"),
            "gradient_current_norm_mean": mean_or_nan(self.gradient_current_norms),
            "gradient_replay_norm_mean": mean_or_nan(self.gradient_replay_norms),
        }


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_worker_init(seed: int):
    def initialize(worker_id: int):
        worker_seed = seed + worker_id
        random.seed(worker_seed)
        np.random.seed(worker_seed)
        torch.manual_seed(worker_seed)

    return initialize


def build_tasks(
    dataset_name: str,
    root: str,
    classes_per_task: int,
    download: bool,
):
    dataset_class = datasets.CIFAR100 if dataset_name == "cifar100" else datasets.CIFAR10
    num_classes = 100 if dataset_name == "cifar100" else 10
    if num_classes % classes_per_task:
        raise ValueError("classes_per_task must divide the dataset's class count.")

    train_base = dataset_class(root=root, train=True, transform=None, download=download)
    test_base = dataset_class(root=root, train=False, transform=None, download=download)
    mean = (0.5071, 0.4867, 0.4408) if dataset_name == "cifar100" else (0.4914, 0.4822, 0.4465)
    std = (0.2675, 0.2565, 0.2761) if dataset_name == "cifar100" else (0.2470, 0.2435, 0.2616)
    train_transform = transforms.Compose(
        [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    )
    evaluation_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean, std)]
    )

    train_targets = np.asarray(train_base.targets)
    test_targets = np.asarray(test_base.targets)
    train_tasks = []
    memory_tasks = []
    test_tasks = []
    for start in range(0, num_classes, classes_per_task):
        classes = np.arange(start, start + classes_per_task)
        train_indices = np.flatnonzero(np.isin(train_targets, classes)).tolist()
        test_indices = np.flatnonzero(np.isin(test_targets, classes)).tolist()
        train_tasks.append(TaskDataset(train_base, train_indices, train_transform))
        memory_tasks.append(TaskDataset(train_base, train_indices, evaluation_transform))
        test_tasks.append(TaskDataset(test_base, test_indices, evaluation_transform))
    return train_tasks, memory_tasks, test_tasks, num_classes


@torch.no_grad()
def evaluate(model, datasets_by_task, device, batch_size, workers):
    model.eval()
    accuracies = []
    for dataset in datasets_by_task:
        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=workers,
            pin_memory=device.type == "cuda",
        )
        correct = 0
        total = 0
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            predictions = model(images).argmax(dim=1)
            correct += int((predictions == labels).sum())
            total += labels.numel()
        accuracies.append(correct / max(total, 1))
    return accuracies


@torch.no_grad()
def feature_quality(memory: ReplayMemory) -> Dict[str, Dict[str, float]]:
    labels = torch.tensor(memory.labels)
    result = {}
    for name, features in (
        ("shallow", memory.shallow_features),
        ("semantic", memory.semantic_features),
    ):
        if features is None or len(features) < 2:
            continue
        normalized = F.normalize(features, dim=1)
        classes = labels.unique(sorted=True)
        centroids = torch.stack(
            [F.normalize(normalized[labels == class_id].mean(0), dim=0) for class_id in classes]
        )
        centroid_predictions = classes[(normalized @ centroids.T).argmax(dim=1)]
        centroid_accuracy = float((centroid_predictions == labels).float().mean())

        within_values = []
        between_values = []
        for class_id in classes:
            mask = labels == class_id
            class_features = normalized[mask]
            centroid = F.normalize(class_features.mean(0), dim=0)
            within_values.append(float((class_features @ centroid).mean()))
        if len(centroids) > 1:
            matrix = centroids @ centroids.T
            upper = torch.triu_indices(len(centroids), len(centroids), offset=1)
            between_values = matrix[upper[0], upper[1]].tolist()
        mean_within = float(np.mean(within_values))
        mean_between = float(np.mean(between_values)) if between_values else 0.0
        result[name] = {
            "nearest_centroid_accuracy": centroid_accuracy,
            "mean_within_class_cosine": mean_within,
            "mean_between_class_centroid_cosine": mean_between,
            "separation_margin": mean_within - mean_between,
        }
    return result


def backward_transfer(accuracy_matrix: Sequence[Sequence[float]]) -> float:
    if len(accuracy_matrix) <= 1:
        return 0.0
    final_row = accuracy_matrix[-1]
    differences = [
        final_row[task] - accuracy_matrix[task][task]
        for task in range(len(accuracy_matrix) - 1)
    ]
    return float(np.mean(differences))


def run_condition(args, condition: str, seed: int, device: torch.device):
    seed_everything(seed)
    train_tasks, memory_tasks, test_tasks, num_classes = build_tasks(
        args.dataset, args.data_root, args.classes_per_task, not args.no_download
    )
    model = SmallCifarCNN(num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    memory = ReplayMemory(
        per_class_capacity=args.memory_per_class,
        temperature=args.temperature,
        floor_frac=args.floor_frac,
        seed=seed + 10_000,
        feature_transform=args.feature_transform,
        score_normalization=args.score_normalization,
        actr_feature=args.actr_feature,
        actr_alpha=args.actr_alpha,
        actr_beta=args.actr_beta,
        actr_gamma=args.actr_gamma,
        actr_decay=args.actr_decay,
        actr_noise=args.actr_noise,
    )
    accuracy_matrix = []
    quality_by_task = {}
    global_step = 0

    for task_id, train_dataset in enumerate(train_tasks):
        if not args.no_reset_optimizer and task_id > 0:
            optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
        if len(memory) and (
            memory.shallow_features is None
            or task_id % max(args.refresh_every, 1) == 0
        ):
            memory.refresh_features(model, device)

        loader_generator = torch.Generator().manual_seed(seed * 1000 + task_id)
        train_loader = DataLoader(
            train_dataset,
            batch_size=args.batch_size,
            shuffle=True,
            num_workers=args.workers,
            pin_memory=device.type == "cuda",
            generator=loader_generator,
            worker_init_fn=make_worker_init(seed * 1000 + task_id),
        )
        model.train()
        for epoch in range(args.epochs):
            running_loss = 0.0
            running_current_loss = 0.0
            running_replay_loss = 0.0
            for images, labels in train_loader:
                global_step += 1
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                current_logits = model(images)
                current_loss = F.cross_entropy(current_logits, labels)
                loss = current_loss
                replay_loss_value = 0.0

                if len(memory):
                    replay_images, replay_labels = memory.select(
                        condition=condition,
                        query_images=images,
                        model=model,
                        replay_size=args.replay_batch_size,
                        device=device,
                        task_id=task_id,
                        global_step=global_step,
                    )
                    replay_logits = model(replay_images)
                    replay_loss = F.cross_entropy(replay_logits, replay_labels)
                    if (
                        args.gradient_diagnostics_every > 0
                        and global_step % args.gradient_diagnostics_every == 0
                    ):
                        memory.record_gradient_diagnostics(
                            current_loss, replay_loss, model
                        )
                    loss = current_loss + args.replay_weight * replay_loss
                    replay_loss_value = float(replay_loss.detach())

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
                optimizer.step()
                running_loss += float(loss.detach())
                running_current_loss += float(current_loss.detach())
                running_replay_loss += replay_loss_value

            batches = max(len(train_loader), 1)
            print(
                f"  [{condition}] task {task_id} epoch {epoch}: "
                f"total={running_loss / batches:.4f} "
                f"current={running_current_loss / batches:.4f} "
                f"replay={running_replay_loss / batches:.4f}"
            )

        # The current task enters memory only after its own training. Consequently,
        # the final task is correctly excluded from retrieval-opportunity analysis.
        memory.add_task_data(memory_tasks[task_id], task_id, current_step=global_step)
        memory.refresh_features(model, device)
        quality_by_task[str(task_id)] = feature_quality(memory)
        accuracies = evaluate(
            model,
            test_tasks[: task_id + 1],
            device,
            args.eval_batch_size,
            args.workers,
        )
        accuracy_matrix.append(accuracies)
        formatted = ", ".join(f"T{i}={value:.3f}" for i, value in enumerate(accuracies))
        print(
            f"[{condition}] after task {task_id}: {formatted} "
            f"| learned_current={accuracies[task_id]:.3f}"
        )

    eligible_classes = list(range(num_classes - args.classes_per_task))
    diagnostics = memory.diagnostics(eligible_classes)
    diagonal = [accuracy_matrix[task][task] for task in range(len(accuracy_matrix))]
    dead_tasks = [task for task, value in enumerate(diagonal) if value < args.plasticity_floor]
    # BWT compares final accuracy against the diagonal. If the diagonal is ~0 the
    # model never learned the task, so BWT trends toward 0 and looks like "less
    # forgetting" when it actually reflects a failure to learn.
    bwt_interpretable = len(dead_tasks) == 0
    health = {
        "diagonal_accuracy": diagonal,
        "mean_diagonal_accuracy": float(np.mean(diagonal)),
        "tasks_below_plasticity_floor": dead_tasks,
        "plasticity_floor": args.plasticity_floor,
        "bwt_interpretable": bwt_interpretable,
        "optimizer_steps_per_task": int(
            math.ceil(len(train_tasks[0]) / args.batch_size) * args.epochs
        ),
    }
    result = {
        "condition": condition,
        "seed": seed,
        "dataset": args.dataset,
        "avg_accuracy": float(np.mean(accuracy_matrix[-1])),
        "bwt": backward_transfer(accuracy_matrix),
        "accuracy_matrix": accuracy_matrix,
        "run_health": health,
        "feature_quality_by_task": quality_by_task,
        "retrieval_diagnostics": diagnostics,
        "config": vars(args),
    }
    print(
        f"[{condition} | seed {seed}] avg_acc={result['avg_accuracy']:.3f} "
        f"BWT={result['bwt']:.3f} "
        f"mean_diag={health['mean_diagonal_accuracy']:.3f}"
    )
    if not bwt_interpretable:
        print(
            f"[{condition}] WARNING: tasks {dead_tasks} never learned "
            f"(accuracy < {args.plasticity_floor}). BWT is NOT interpretable for this run."
        )
    print(f"[{condition}] RETRIEVAL DIAGNOSTICS")
    print(json.dumps(diagnostics, indent=2, allow_nan=True))
    return result, memory.events


def write_outputs(output_dir: Path, results: List[Dict], all_events: List[Dict]):
    output_dir.mkdir(parents=True, exist_ok=True)
    with (output_dir / "results.json").open("w", encoding="utf-8") as handle:
        json.dump(results, handle, indent=2, allow_nan=True)

    if all_events:
        with (output_dir / "replay_events.csv").open(
            "w", newline="", encoding="utf-8"
        ) as handle:
            writer = csv.DictWriter(handle, fieldnames=list(all_events[0]))
            writer.writeheader()
            writer.writerows(all_events)

    summary = []
    for condition in sorted({result["condition"] for result in results}):
        condition_results = [
            result for result in results if result["condition"] == condition
        ]
        accuracy = np.asarray([result["avg_accuracy"] for result in condition_results])
        bwt = np.asarray([result["bwt"] for result in condition_results])
        diagnostic_keys = (
            "retrieval_count_cv",
            "retrieval_count_gini",
            "exposure_normalized_ratio_cv",
            "observed_vs_exposure_expected_jsd",
            "sample_coverage_fraction",
            "sample_retrieval_gini",
            "mean_selected_pair_similarity",
            "mean_batch_class_entropy",
            "mean_unique_classes_per_replay",
            "mean_score_std",
            "mean_softmax_effective_fraction",
            "gradient_cosine_mean",
            "gradient_conflict_fraction",
        )
        row = {
            "condition": condition,
            "seeds": len(condition_results),
            "avg_accuracy_mean": float(accuracy.mean()),
            "avg_accuracy_std": float(accuracy.std()),
            "bwt_mean": float(bwt.mean()),
            "bwt_std": float(bwt.std()),
            "mean_diagonal_accuracy": float(
                np.mean(
                    [
                        result["run_health"]["mean_diagonal_accuracy"]
                        for result in condition_results
                    ]
                )
            ),
            "runs_with_interpretable_bwt": int(
                sum(
                    1
                    for result in condition_results
                    if result["run_health"]["bwt_interpretable"]
                )
            ),
        }
        for key in diagnostic_keys:
            values = np.asarray(
                [
                    result["retrieval_diagnostics"][key]
                    for result in condition_results
                ],
                dtype=np.float64,
            )
            # Random replay computes no similarity scores, so score-based metrics
            # are legitimately all-NaN for that condition.
            if values.size == 0 or np.all(np.isnan(values)):
                row[f"{key}_mean"] = float("nan")
                row[f"{key}_std"] = float("nan")
            else:
                row[f"{key}_mean"] = float(np.nanmean(values))
                row[f"{key}_std"] = float(np.nanstd(values))
        summary.append(row)

    with (output_dir / "summary.csv").open(
        "w", newline="", encoding="utf-8"
    ) as handle:
        writer = csv.DictWriter(handle, fieldnames=list(summary[0]))
        writer.writeheader()
        writer.writerows(summary)

    summary_by_condition = {row["condition"]: row for row in summary}

    print("\n===== Summary (mean +/- std over seeds) =====")
    for row in summary:
        flag = "" if row["runs_with_interpretable_bwt"] == row["seeds"] else "  [BWT UNRELIABLE]"
        print(
            f"{row['condition']:10s} "
            f"avg_acc={row['avg_accuracy_mean']:.3f} +/- {row['avg_accuracy_std']:.3f}  "
            f"BWT={row['bwt_mean']:.3f} +/- {row['bwt_std']:.3f}  "
            f"mean_diag={row['mean_diagonal_accuracy']:.3f}  "
            f"exposure-JSD={row['observed_vs_exposure_expected_jsd_mean']:.4f}  "
            f"sample-coverage={row['sample_coverage_fraction_mean']:.3f}  "
            f"grad-conflict={row['gradient_conflict_fraction_mean']:.3f}{flag}"
        )

    diagnosis = {
        "status": "insufficient_conditions",
        "interpretation": [
            "Run both random and shallow conditions to generate a direct diagnosis."
        ],
    }
    if "random" in summary_by_condition and "shallow" in summary_by_condition:
        random_row = summary_by_condition["random"]
        shallow_row = summary_by_condition["shallow"]
        findings = []

        degenerate = [
            row["condition"]
            for row in summary
            if row["runs_with_interpretable_bwt"] < row["seeds"]
        ]
        if degenerate:
            diagnosis = {
                "status": "invalid_run",
                "degenerate_conditions": degenerate,
                "interpretation": [
                    "One or more conditions failed to learn some tasks, so their "
                    "diagonal accuracy is ~0 and BWT is driven by lack of plasticity "
                    "rather than by forgetting.",
                    "Fix plasticity first (smaller --batch-size, more --epochs, or a "
                    "lower --replay-weight), then compare retrieval conditions.",
                ],
            }
            with (output_dir / "diagnosis.json").open("w", encoding="utf-8") as handle:
                json.dump(diagnosis, handle, indent=2, allow_nan=True)
            print("\n===== Diagnosis =====")
            print(json.dumps(diagnosis, indent=2))
            return

        random_jsd = random_row["observed_vs_exposure_expected_jsd_mean"]
        shallow_jsd = shallow_row["observed_vs_exposure_expected_jsd_mean"]
        random_coverage = random_row["sample_coverage_fraction_mean"]
        shallow_coverage = shallow_row["sample_coverage_fraction_mean"]
        if shallow_jsd > random_jsd * 1.25 and shallow_coverage < random_coverage - 0.05:
            findings.append(
                {
                    "mechanism": "coverage_bias",
                    "supported": True,
                    "reason": (
                        "Shallow retrieval departs more from its exposure-adjusted "
                        "uniform baseline and reaches fewer stored samples than random."
                    ),
                }
            )
        else:
            findings.append(
                {
                    "mechanism": "coverage_bias",
                    "supported": False,
                    "reason": (
                        "Shallow retrieval does not show both a materially larger "
                        "exposure-adjusted divergence and lower sample coverage."
                    ),
                }
            )

        random_pair = random_row["mean_selected_pair_similarity_mean"]
        shallow_pair = shallow_row["mean_selected_pair_similarity_mean"]
        findings.append(
            {
                "mechanism": "redundant_similarity_replay",
                "supported": bool(
                    np.isfinite(random_pair)
                    and np.isfinite(shallow_pair)
                    and shallow_pair > random_pair + 0.05
                ),
                "reason": (
                    "Selected shallow batches are more internally similar than random."
                    if np.isfinite(random_pair)
                    and np.isfinite(shallow_pair)
                    and shallow_pair > random_pair + 0.05
                    else "Shallow batches are not materially more redundant than random."
                ),
            }
        )

        random_conflict = random_row["gradient_conflict_fraction_mean"]
        shallow_conflict = shallow_row["gradient_conflict_fraction_mean"]
        findings.append(
            {
                "mechanism": "gradient_interference",
                "supported": bool(
                    np.isfinite(random_conflict)
                    and np.isfinite(shallow_conflict)
                    and shallow_conflict > random_conflict + 0.05
                ),
                "reason": (
                    "Shallow replay gradients conflict with current-task gradients more "
                    "often than random replay gradients."
                    if np.isfinite(random_conflict)
                    and np.isfinite(shallow_conflict)
                    and shallow_conflict > random_conflict + 0.05
                    else "Shallow replay does not show materially more gradient conflict."
                ),
            }
        )
        shallow_effective = shallow_row["mean_softmax_effective_fraction_mean"]
        semantic_effective = summary_by_condition.get("semantic", {}).get(
            "mean_softmax_effective_fraction_mean", float("nan")
        )
        findings.append(
            {
                "mechanism": "non_selective_retrieval",
                "supported": bool(
                    np.isfinite(shallow_effective) and shallow_effective > 0.5
                ),
                "shallow_softmax_effective_fraction": shallow_effective,
                "semantic_softmax_effective_fraction": semantic_effective,
                "reason": (
                    "The shallow retrieval distribution stays close to uniform, so "
                    "'shallow' is effectively random replay with extra compute."
                    if np.isfinite(shallow_effective) and shallow_effective > 0.5
                    else "Shallow retrieval is meaningfully peaked, so it is genuinely selective."
                ),
            }
        )
        diagnosis = {
            "status": "diagnosed",
            "thresholds": {
                "coverage_jsd_multiplier": 1.25,
                "sample_coverage_absolute_gap": 0.05,
                "pair_similarity_absolute_gap": 0.05,
                "gradient_conflict_absolute_gap": 0.05,
            },
            "findings": findings,
            "important_note": (
                "Raw class retrieval totals are age-confounded. Prefer the "
                "exposure-normalized JSD and observed/expected ratios when deciding "
                "whether similarity replay is biased."
            ),
        }

        if "actr" in summary_by_condition:
            actr_row = summary_by_condition["actr"]
            diagnosis["actr_intervention"] = {
                "coverage_recovered": bool(
                    actr_row["sample_coverage_fraction_mean"]
                    > shallow_row["sample_coverage_fraction_mean"] + 0.05
                ),
                "beats_shallow_on_bwt": bool(
                    actr_row["bwt_mean"] > shallow_row["bwt_mean"]
                ),
                "beats_random_on_bwt": bool(
                    actr_row["bwt_mean"] > random_row["bwt_mean"]
                ),
                "beats_random_on_accuracy": bool(
                    actr_row["avg_accuracy_mean"] > random_row["avg_accuracy_mean"]
                ),
                "sample_coverage": {
                    "random": random_row["sample_coverage_fraction_mean"],
                    "shallow": shallow_row["sample_coverage_fraction_mean"],
                    "actr": actr_row["sample_coverage_fraction_mean"],
                },
                "bwt": {
                    "random": random_row["bwt_mean"],
                    "shallow": shallow_row["bwt_mean"],
                    "actr": actr_row["bwt_mean"],
                },
                "note": (
                    "Recovering coverage without beating random means coverage was "
                    "necessary but not sufficient, which is itself a reportable result."
                ),
            }
    with (output_dir / "diagnosis.json").open("w", encoding="utf-8") as handle:
        json.dump(diagnosis, handle, indent=2, allow_nan=True)

    print("\n===== Diagnosis =====")
    print(json.dumps(diagnosis, indent=2, allow_nan=True))


def parse_arguments():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--condition",
        choices=("random", "semantic", "shallow", "actr", "all", "compare"),
        default="all",
    )
    parser.add_argument("--dataset", choices=("cifar10", "cifar100"), default="cifar100")
    parser.add_argument("--data-root", default="/kaggle/working/data")
    parser.add_argument("--output-dir", default="/kaggle/working/actr_diagnostics")
    parser.add_argument("--seeds", type=int, default=1)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--classes-per-task", type=int, default=10)
    parser.add_argument("--memory-per-class", type=int, default=40)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--eval-batch-size", type=int, default=256)
    parser.add_argument("--replay-batch-size", type=int, default=32)
    parser.add_argument("--replay-weight", type=float, default=1.0)
    parser.add_argument("--temperature", type=float, default=0.1)
    parser.add_argument("--floor-frac", type=float, default=0.0)
    parser.add_argument("--refresh-every", type=int, default=1)
    parser.add_argument("--gradient-diagnostics-every", type=int, default=25)
    parser.add_argument(
        "--feature-transform",
        choices=("none", "center", "standardize"),
        default="standardize",
    )
    parser.add_argument(
        "--score-normalization", choices=("none", "zscore"), default="zscore"
    )
    parser.add_argument("--plasticity-floor", type=float, default=0.05)
    parser.add_argument(
        "--actr-feature", choices=("shallow", "semantic"), default="shallow"
    )
    parser.add_argument("--actr-alpha", type=float, default=1.0)
    parser.add_argument("--actr-beta", type=float, default=-1.0)
    parser.add_argument("--actr-gamma", type=float, default=1.0)
    parser.add_argument("--actr-decay", type=float, default=0.5)
    parser.add_argument("--actr-noise", type=float, default=0.0)
    parser.add_argument("--grad-clip", type=float, default=5.0)
    parser.add_argument("--lr", type=float, default=5e-4)
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument("--no-reset-optimizer", action="store_true")
    parser.add_argument("--no-download", action="store_true")
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"Ignoring notebook arguments: {unknown}")
    if not 0.0 <= args.floor_frac <= 1.0:
        parser.error("--floor-frac must be between 0 and 1.")
    if not 0.0 < args.actr_decay < 1.0:
        parser.error("--actr-decay must be strictly between 0 and 1.")
    if args.seeds < 1 or args.epochs < 1:
        parser.error("--seeds and --epochs must be positive.")
    return args


def main():
    args = parse_arguments()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    if args.condition == "all":
        conditions = ["random", "semantic", "shallow"]
    elif args.condition == "compare":
        conditions = ["random", "shallow", "actr"]
    else:
        conditions = [args.condition]
    results = []
    all_events = []
    for condition in conditions:
        for seed in range(args.seeds):
            print(f"\n===== Running condition: {condition} | seed {seed} =====")
            result, events = run_condition(args, condition, seed, device)
            results.append(result)
            all_events.extend(
                {
                    "condition": condition,
                    "seed": seed,
                    **asdict(event),
                }
                for event in events
            )
    write_outputs(Path(args.output_dir), results, all_events)


if __name__ == "__main__":
    main()

Writing actr_diagnostic_experiment.py


In [2]:
!python actr_diagnostic_experiment.py \
  --data-root /kaggle/working/data \
  --output-dir /kaggle/working/actr_compare \
  --dataset cifar100 --condition compare --seeds 1 --epochs 5

Using device: cuda

===== Running condition: random | seed 0 =====
100%|█████████████████████████████████████████| 169M/169M [26:16<00:00, 107kB/s]
  [random] task 0 epoch 0: total=2.1336 current=2.1336 replay=0.0000
  [random] task 0 epoch 1: total=1.5204 current=1.5204 replay=0.0000
  [random] task 0 epoch 2: total=1.2464 current=1.2464 replay=0.0000
  [random] task 0 epoch 3: total=1.1048 current=1.1048 replay=0.0000
  [random] task 0 epoch 4: total=1.0038 current=1.0038 replay=0.0000
[random] after task 0: T0=0.656 | learned_current=0.656
  [random] task 1 epoch 0: total=4.1904 current=3.1857 replay=1.0048
  [random] task 1 epoch 1: total=2.0064 current=1.6137 replay=0.3926
  [random] task 1 epoch 2: total=1.4776 current=1.2985 replay=0.1791
  [random] task 1 epoch 3: total=1.3199 current=1.2033 replay=0.1165
  [random] task 1 epoch 4: total=1.1627 current=1.0841 replay=0.0786
[random] after task 1: T0=0.425, T1=0.361 | learned_current=0.361
  [random] task 2 epoch 0: total=4.6609 

In [1]:
%%writefile actr_diagnostic_experiment.py
#!/usr/bin/env python3
"""Kaggle-ready continual-learning experiment with retrieval diagnostics."""

import importlib.util
import subprocess
import sys


def install_missing_packages():
    packages = {
        "numpy": "numpy",
        "torch": "torch",
        "torchvision": "torchvision",
    }
    missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *missing]
        )


install_missing_packages()

import argparse
import csv
import json
import math
import os
import random
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms


@dataclass
class ReplayEvent:
    task: int
    step: int
    condition: str
    memory_size: int
    selected: int
    query_selected_similarity: float
    selected_pair_similarity: float
    selected_class_entropy: float
    selected_unique_classes: int
    score_std: float
    softmax_effective_fraction: float


class TaskDataset(Dataset):
    def __init__(self, base_dataset, indices: Sequence[int], transform):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform
        self.targets = [int(base_dataset.targets[i]) for i in self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        image, label = self.base_dataset[self.indices[index]]
        return self.transform(image), int(label)


class SmallCifarCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.projector = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
        )
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x, return_features: bool = False):
        shallow_map = self.block1(x)
        shallow = F.adaptive_avg_pool2d(shallow_map, 1).flatten(1)
        semantic = self.projector(self.block3(self.block2(shallow_map)))
        logits = self.classifier(semantic)
        if return_features:
            return logits, shallow, semantic
        return logits


def normalized_entropy(counts: Sequence[int]) -> float:
    values = np.asarray(counts, dtype=np.float64)
    values = values[values > 0]
    if len(values) <= 1:
        return 0.0
    probabilities = values / values.sum()
    return float(-(probabilities * np.log(probabilities)).sum() / np.log(len(values)))


def gini(values: Sequence[float]) -> float:
    array = np.asarray(values, dtype=np.float64)
    if array.size == 0 or np.allclose(array.sum(), 0.0):
        return 0.0
    array = np.sort(np.maximum(array, 0.0))
    index = np.arange(1, len(array) + 1)
    return float(
        (2.0 * np.sum(index * array) / (len(array) * array.sum()))
        - (len(array) + 1.0) / len(array)
    )


def distribution_jsd(left: Sequence[float], right: Sequence[float]) -> float:
    p = np.asarray(left, dtype=np.float64)
    q = np.asarray(right, dtype=np.float64)
    if p.sum() == 0 or q.sum() == 0:
        return 0.0
    p /= p.sum()
    q /= q.sum()
    middle = 0.5 * (p + q)

    def kl_divergence(a, b):
        mask = a > 0
        return float(np.sum(a[mask] * np.log2(a[mask] / b[mask])))

    return 0.5 * kl_divergence(p, middle) + 0.5 * kl_divergence(q, middle)


class ReplayMemory:
    def __init__(
        self,
        per_class_capacity: int,
        temperature: float,
        floor_frac: float,
        seed: int,
        feature_transform: str = "standardize",
        score_normalization: str = "zscore",
        actr_feature: str = "shallow",
        actr_alpha: float = 1.0,
        actr_beta: float = -1.0,
        actr_gamma: float = 1.0,
        actr_decay: float = 0.5,
        actr_noise: float = 0.0,
    ):
        self.per_class_capacity = per_class_capacity
        self.temperature = temperature
        self.floor_frac = floor_frac
        self.feature_transform = feature_transform
        self.score_normalization = score_normalization
        self.actr_feature = actr_feature
        self.actr_alpha = actr_alpha
        self.actr_beta = actr_beta
        self.actr_gamma = actr_gamma
        self.actr_decay = actr_decay
        self.actr_noise = actr_noise
        self.generator = torch.Generator().manual_seed(seed)
        self.images: List[torch.Tensor] = []
        self.labels: List[int] = []
        self.task_ids: List[int] = []
        self.sample_ids: List[str] = []
        self.retrieval_counts: List[int] = []
        self.expected_retrievals: List[float] = []
        self.last_retrieved_step: List[int] = []
        self.creation_step: List[int] = []
        self.class_retrieval_counts: Counter = Counter()
        self.shallow_features: Optional[torch.Tensor] = None
        self.semantic_features: Optional[torch.Tensor] = None
        self.shallow_stats: Optional[Dict[str, torch.Tensor]] = None
        self.semantic_stats: Optional[Dict[str, torch.Tensor]] = None
        self.raw_shallow_cosine_mean = float("nan")
        self.raw_semantic_cosine_mean = float("nan")
        self.transformed_shallow_cosine_mean = float("nan")
        self.transformed_semantic_cosine_mean = float("nan")
        self.events: List[ReplayEvent] = []
        self.gradient_cosines: List[float] = []
        self.gradient_current_norms: List[float] = []
        self.gradient_replay_norms: List[float] = []

    def __len__(self):
        return len(self.labels)

    def add_task_data(self, dataset: TaskDataset, task_id: int, current_step: int = 0):
        class_to_local_indices = defaultdict(list)
        for local_index, label in enumerate(dataset.targets):
            class_to_local_indices[label].append(local_index)

        for label in sorted(class_to_local_indices):
            candidates = class_to_local_indices[label]
            permutation = torch.randperm(len(candidates), generator=self.generator).tolist()
            chosen = [candidates[i] for i in permutation[: self.per_class_capacity]]
            for local_index in chosen:
                image, item_label = dataset[local_index]
                source_index = dataset.indices[local_index]
                self.images.append(image.cpu())
                self.labels.append(item_label)
                self.task_ids.append(task_id)
                self.sample_ids.append(f"task{task_id}:source{source_index}")
                self.retrieval_counts.append(0)
                self.expected_retrievals.append(0.0)
                self.last_retrieved_step.append(-1)
                self.creation_step.append(current_step)

        self.shallow_features = None
        self.semantic_features = None
        self.shallow_stats = None
        self.semantic_stats = None

    def _fit_transform_stats(self, raw: torch.Tensor) -> Dict[str, torch.Tensor]:
        mean = raw.mean(dim=0, keepdim=True)
        std = raw.std(dim=0, keepdim=True).clamp_min(1e-6)
        return {"mean": mean, "std": std}

    def _apply_transform(
        self, raw: torch.Tensor, stats: Optional[Dict[str, torch.Tensor]]
    ) -> torch.Tensor:
        if self.feature_transform != "none" and stats is not None:
            raw = raw - stats["mean"]
            if self.feature_transform == "standardize":
                raw = raw / stats["std"]
        return F.normalize(raw, dim=1)

    @torch.no_grad()
    def refresh_features(self, model: nn.Module, device: torch.device, batch_size: int = 256):
        if not self.images:
            return
        was_training = model.training
        model.eval()
        shallow_parts = []
        semantic_parts = []
        for start in range(0, len(self.images), batch_size):
            batch = torch.stack(self.images[start : start + batch_size]).to(device)
            _, shallow, semantic = model(batch, return_features=True)
            shallow_parts.append(shallow.cpu())
            semantic_parts.append(semantic.cpu())
        raw_shallow = torch.cat(shallow_parts)
        raw_semantic = torch.cat(semantic_parts)

        # Post-ReLU pooled activations are non-negative, so raw cosine similarity
        # saturates near 1 and carries almost no ranking signal. Centering (and
        # optionally scaling) restores usable geometry before normalization.
        self.shallow_stats = self._fit_transform_stats(raw_shallow)
        self.semantic_stats = self._fit_transform_stats(raw_semantic)
        self.raw_shallow_cosine_mean = float(
            self._mean_pairwise_cosine(F.normalize(raw_shallow, dim=1))
        )
        self.raw_semantic_cosine_mean = float(
            self._mean_pairwise_cosine(F.normalize(raw_semantic, dim=1))
        )
        self.shallow_features = self._apply_transform(raw_shallow, self.shallow_stats)
        self.semantic_features = self._apply_transform(raw_semantic, self.semantic_stats)
        self.transformed_shallow_cosine_mean = float(
            self._mean_pairwise_cosine(self.shallow_features)
        )
        self.transformed_semantic_cosine_mean = float(
            self._mean_pairwise_cosine(self.semantic_features)
        )
        model.train(was_training)

    @staticmethod
    def _mean_pairwise_cosine(features: torch.Tensor, sample_limit: int = 512) -> float:
        if len(features) < 2:
            return float("nan")
        # Memory is stored in task order, so an evenly spaced stride keeps the
        # estimate representative instead of biasing it toward the earliest task.
        if len(features) > sample_limit:
            positions = torch.linspace(0, len(features) - 1, sample_limit).long()
            subset = features[positions]
        else:
            subset = features
        matrix = subset @ subset.T
        upper = torch.triu_indices(len(subset), len(subset), offset=1)
        return float(matrix[upper[0], upper[1]].mean())

    def _update_expected_counts(self, replay_size: int):
        if not self.labels:
            return
        probability = min(replay_size, len(self.labels)) / len(self.labels)
        for index in range(len(self.labels)):
            self.expected_retrievals[index] += probability

    def _coverage_floor_indices(self, count: int) -> List[int]:
        if count <= 0:
            return []
        by_class = defaultdict(list)
        for index, label in enumerate(self.labels):
            by_class[label].append(index)
        classes = sorted(by_class)
        class_order = torch.randperm(len(classes), generator=self.generator).tolist()
        selected = []
        cursor = 0
        while len(selected) < count:
            label = classes[class_order[cursor % len(classes)]]
            candidates = by_class[label]
            chosen = candidates[
                torch.randint(len(candidates), (1,), generator=self.generator).item()
            ]
            if chosen not in selected:
                selected.append(chosen)
            cursor += 1
            if cursor > count * len(classes) * 4:
                break
        return selected

    def _random_indices(self, replay_size: int) -> List[int]:
        count = min(replay_size, len(self.labels))
        return torch.randperm(len(self.labels), generator=self.generator)[:count].tolist()

    def _base_level_activation(self, global_step: int) -> torch.Tensor:
        # ACT-R optimized-learning approximation:
        #   B_i = ln(n_i / (1 - d)) - d * ln(T_i)
        # n_i counts the encoding event plus every retrieval; T_i is the age of
        # the trace. High B_i means the item is already highly accessible.
        counts = torch.tensor(self.retrieval_counts, dtype=torch.float32) + 1.0
        age = (
            torch.full((len(self.labels),), float(global_step))
            - torch.tensor(self.creation_step, dtype=torch.float32)
        ).clamp_min(1.0)
        return torch.log(counts / (1.0 - self.actr_decay)) - self.actr_decay * torch.log(age)

    def _class_overrepresentation(self) -> torch.Tensor:
        # How much each item's class has been replayed relative to a uniform
        # share, expressed per item so it can be combined with the other terms.
        labels = torch.tensor(self.labels, dtype=torch.long)
        total = float(sum(self.class_retrieval_counts.values()))
        distinct = len(set(self.labels))
        if total <= 0 or distinct == 0:
            return torch.zeros(len(self.labels), dtype=torch.float32)
        uniform_share = total / distinct
        per_class = torch.tensor(
            [self.class_retrieval_counts.get(int(label), 0) for label in labels],
            dtype=torch.float32,
        )
        return per_class / uniform_share

    @staticmethod
    def _zscore(values: torch.Tensor) -> torch.Tensor:
        return (values - values.mean()) / values.std().clamp_min(1e-6)

    @torch.no_grad()
    def select(
        self,
        condition: str,
        query_images: torch.Tensor,
        model: nn.Module,
        replay_size: int,
        device: torch.device,
        task_id: int,
        global_step: int,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        count = min(replay_size, len(self.labels))
        if count == 0:
            raise RuntimeError("Replay selection requested from an empty memory.")

        self._update_expected_counts(count)
        query_selected_similarity = float("nan")
        score_std = float("nan")
        softmax_effective_fraction = float("nan")

        if condition == "random":
            selected = self._random_indices(count)
            diagnostic_features = self.semantic_features
        else:
            if condition == "shallow":
                feature_name = "shallow"
            elif condition == "semantic":
                feature_name = "semantic"
            else:
                feature_name = self.actr_feature
            features = (
                self.shallow_features if feature_name == "shallow" else self.semantic_features
            )
            stats = (
                self.shallow_stats if feature_name == "shallow" else self.semantic_stats
            )
            if features is None:
                raise RuntimeError("Memory features must be refreshed before similarity replay.")

            was_training = model.training
            model.eval()
            _, query_shallow, query_semantic = model(
                query_images.to(device), return_features=True
            )
            model.train(was_training)
            query_raw = (query_shallow if feature_name == "shallow" else query_semantic).cpu()
            query_features = self._apply_transform(query_raw, stats)

            # Max-over-query implements cue-driven retrieval without collapsing
            # a heterogeneous current batch into a potentially meaningless centroid.
            similarities = query_features @ features.T
            scores = similarities.max(dim=0).values

            if condition == "actr":
                # Cue relevance alone concentrates replay on a small subset. The
                # base-level and overrepresentation terms restore coverage by
                # discounting traces that are already accessible or already
                # over-replayed at the class level.
                scores = (
                    self.actr_alpha * self._zscore(scores)
                    + self.actr_beta * self._zscore(self._base_level_activation(global_step))
                    - self.actr_gamma * self._zscore(self._class_overrepresentation())
                )
                if self.actr_noise > 0:
                    uniform = torch.rand(
                        len(scores), generator=self.generator, dtype=torch.float32
                    ).clamp(1e-6, 1 - 1e-6)
                    scores = scores + self.actr_noise * torch.log(uniform / (1 - uniform))

            score_std = float(scores.std())
            floor_count = min(count, int(round(count * self.floor_frac)))
            floor_indices = self._coverage_floor_indices(floor_count)
            available_mask = torch.ones(len(self.labels), dtype=torch.bool)
            if floor_indices:
                available_mask[floor_indices] = False
            remaining = count - len(floor_indices)
            available_indices = torch.where(available_mask)[0]
            if remaining:
                available_scores = scores[available_indices]
                # Absolute cosine ranges differ wildly between representations, so a
                # fixed temperature is not comparable across conditions. Z-scoring
                # makes the temperature control selectivity, not feature scale.
                if self.score_normalization == "zscore":
                    available_scores = (
                        available_scores - available_scores.mean()
                    ) / available_scores.std().clamp_min(1e-6)
                probabilities = torch.softmax(
                    available_scores / max(self.temperature, 1e-8), dim=0
                )
                softmax_effective_fraction = float(
                    torch.exp(
                        -(probabilities * torch.log(probabilities.clamp_min(1e-12))).sum()
                    )
                    / len(probabilities)
                )
                sampled_positions = torch.multinomial(
                    probabilities,
                    remaining,
                    replacement=False,
                    generator=self.generator,
                )
                similarity_indices = available_indices[sampled_positions].tolist()
            else:
                similarity_indices = []
            selected = floor_indices + similarity_indices
            query_selected_similarity = float(similarities[:, selected].max(dim=0).values.mean())
            diagnostic_features = features

        for index in selected:
            self.retrieval_counts[index] += 1
            self.last_retrieved_step[index] = global_step
            self.class_retrieval_counts[self.labels[index]] += 1

        selected_labels = [self.labels[index] for index in selected]
        label_counts = Counter(selected_labels)
        pair_similarity = float("nan")
        if diagnostic_features is not None and len(selected) > 1:
            chosen_features = F.normalize(diagnostic_features[selected], dim=1)
            similarity_matrix = chosen_features @ chosen_features.T
            upper = torch.triu_indices(len(selected), len(selected), offset=1)
            pair_similarity = float(similarity_matrix[upper[0], upper[1]].mean())

        self.events.append(
            ReplayEvent(
                task=task_id,
                step=global_step,
                condition=condition,
                memory_size=len(self.labels),
                selected=len(selected),
                query_selected_similarity=query_selected_similarity,
                selected_pair_similarity=pair_similarity,
                selected_class_entropy=normalized_entropy(list(label_counts.values())),
                selected_unique_classes=len(label_counts),
                score_std=score_std,
                softmax_effective_fraction=softmax_effective_fraction,
            )
        )
        images = torch.stack([self.images[index] for index in selected]).to(device)
        labels = torch.tensor(
            [self.labels[index] for index in selected], dtype=torch.long, device=device
        )
        return images, labels

    def record_gradient_diagnostics(
        self,
        current_loss: torch.Tensor,
        replay_loss: torch.Tensor,
        model: nn.Module,
    ):
        parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
        current_gradients = torch.autograd.grad(
            current_loss, parameters, retain_graph=True, allow_unused=True
        )
        replay_gradients = torch.autograd.grad(
            replay_loss, parameters, retain_graph=True, allow_unused=True
        )
        current_parts = []
        replay_parts = []
        for current, replay, parameter in zip(
            current_gradients, replay_gradients, parameters
        ):
            current_parts.append(
                current.detach().flatten()
                if current is not None
                else torch.zeros_like(parameter).flatten()
            )
            replay_parts.append(
                replay.detach().flatten()
                if replay is not None
                else torch.zeros_like(parameter).flatten()
            )
        current_vector = torch.cat(current_parts)
        replay_vector = torch.cat(replay_parts)
        current_norm = current_vector.norm()
        replay_norm = replay_vector.norm()
        cosine = F.cosine_similarity(current_vector, replay_vector, dim=0)
        self.gradient_cosines.append(float(cosine))
        self.gradient_current_norms.append(float(current_norm))
        self.gradient_replay_norms.append(float(replay_norm))

    def diagnostics(self, eligible_classes: Sequence[int]) -> Dict:
        eligible = set(eligible_classes)
        observed_by_class = Counter()
        expected_by_class = defaultdict(float)
        item_counts = []
        retrieved_items = 0
        eligible_items = 0
        task_counts = Counter()

        for index, label in enumerate(self.labels):
            if label not in eligible:
                continue
            observed = self.retrieval_counts[index]
            observed_by_class[label] += observed
            expected_by_class[label] += self.expected_retrievals[index]
            item_counts.append(observed)
            eligible_items += 1
            retrieved_items += int(observed > 0)
            task_counts[self.task_ids[index]] += observed

        classes = sorted(eligible)
        observed = np.asarray([observed_by_class[c] for c in classes], dtype=np.float64)
        expected = np.asarray([expected_by_class[c] for c in classes], dtype=np.float64)
        ratio = np.divide(
            observed,
            expected,
            out=np.zeros_like(observed),
            where=expected > 0,
        )
        nonzero_expected = expected > 0
        zero_classes = [
            class_id
            for class_id, obs, exp in zip(classes, observed, expected)
            if exp > 0 and obs == 0
        ]
        ranked_classes = sorted(
            ((class_id, int(observed_by_class[class_id])) for class_id in classes),
            key=lambda item: (item[1], item[0]),
        )
        event_values = {
            field: [
                getattr(event, field)
                for event in self.events
                if not math.isnan(getattr(event, field))
            ]
            for field in (
                "query_selected_similarity",
                "selected_pair_similarity",
                "selected_class_entropy",
                "selected_unique_classes",
                "score_std",
                "softmax_effective_fraction",
            )
        }

        def mean_or_nan(values):
            return float(np.mean(values)) if values else float("nan")

        return {
            "eligible_classes": classes,
            "zero_retrieval_classes": zero_classes,
            "zero_retrieval_class_count": len(zero_classes),
            "least_retrieved_classes": [
                {"class": class_id, "count": count}
                for class_id, count in ranked_classes[:10]
            ],
            "most_retrieved_classes": [
                {"class": class_id, "count": count}
                for class_id, count in reversed(ranked_classes[-10:])
            ],
            "class_retrieval_counts": {
                str(class_id): int(observed_by_class[class_id]) for class_id in classes
            },
            "class_expected_uniform_counts": {
                str(class_id): round(float(expected_by_class[class_id]), 4)
                for class_id in classes
            },
            "class_observed_expected_ratio": {
                str(class_id): round(float(value), 4)
                for class_id, value in zip(classes, ratio)
            },
            "retrieval_count_mean": float(observed.mean()) if len(observed) else 0.0,
            "retrieval_count_std": float(observed.std()) if len(observed) else 0.0,
            "retrieval_count_cv": float(observed.std() / observed.mean())
            if len(observed) and observed.mean() > 0
            else 0.0,
            "retrieval_count_gini": gini(observed),
            "exposure_normalized_ratio_cv": float(ratio[nonzero_expected].std())
            if nonzero_expected.any()
            else 0.0,
            "observed_vs_exposure_expected_jsd": distribution_jsd(observed, expected),
            "sample_coverage_fraction": retrieved_items / max(eligible_items, 1),
            "sample_retrieval_gini": gini(item_counts),
            "retrievals_by_source_task": {
                str(task): int(count) for task, count in sorted(task_counts.items())
            },
            "mean_query_selected_similarity": mean_or_nan(
                event_values["query_selected_similarity"]
            ),
            "mean_selected_pair_similarity": mean_or_nan(
                event_values["selected_pair_similarity"]
            ),
            "mean_batch_class_entropy": mean_or_nan(
                event_values["selected_class_entropy"]
            ),
            "mean_unique_classes_per_replay": mean_or_nan(
                event_values["selected_unique_classes"]
            ),
            "mean_score_std": mean_or_nan(event_values["score_std"]),
            "mean_softmax_effective_fraction": mean_or_nan(
                event_values["softmax_effective_fraction"]
            ),
            "raw_shallow_cosine_mean": self.raw_shallow_cosine_mean,
            "raw_semantic_cosine_mean": self.raw_semantic_cosine_mean,
            "transformed_shallow_cosine_mean": self.transformed_shallow_cosine_mean,
            "transformed_semantic_cosine_mean": self.transformed_semantic_cosine_mean,
            "gradient_cosine_mean": mean_or_nan(self.gradient_cosines),
            "gradient_cosine_std": float(np.std(self.gradient_cosines))
            if self.gradient_cosines
            else float("nan"),
            "gradient_conflict_fraction": float(
                np.mean(np.asarray(self.gradient_cosines) < 0)
            )
            if self.gradient_cosines
            else float("nan"),
            "gradient_current_norm_mean": mean_or_nan(self.gradient_current_norms),
            "gradient_replay_norm_mean": mean_or_nan(self.gradient_replay_norms),
        }


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_worker_init(seed: int):
    def initialize(worker_id: int):
        worker_seed = seed + worker_id
        random.seed(worker_seed)
        np.random.seed(worker_seed)
        torch.manual_seed(worker_seed)

    return initialize


def build_tasks(
    dataset_name: str,
    root: str,
    classes_per_task: int,
    download: bool,
):
    dataset_class = datasets.CIFAR100 if dataset_name == "cifar100" else datasets.CIFAR10
    num_classes = 100 if dataset_name == "cifar100" else 10
    if num_classes % classes_per_task:
        raise ValueError("classes_per_task must divide the dataset's class count.")

    train_base = dataset_class(root=root, train=True, transform=None, download=download)
    test_base = dataset_class(root=root, train=False, transform=None, download=download)
    mean = (0.5071, 0.4867, 0.4408) if dataset_name == "cifar100" else (0.4914, 0.4822, 0.4465)
    std = (0.2675, 0.2565, 0.2761) if dataset_name == "cifar100" else (0.2470, 0.2435, 0.2616)
    train_transform = transforms.Compose(
        [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    )
    evaluation_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean, std)]
    )

    train_targets = np.asarray(train_base.targets)
    test_targets = np.asarray(test_base.targets)
    train_tasks = []
    memory_tasks = []
    test_tasks = []
    for start in range(0, num_classes, classes_per_task):
        classes = np.arange(start, start + classes_per_task)
        train_indices = np.flatnonzero(np.isin(train_targets, classes)).tolist()
        test_indices = np.flatnonzero(np.isin(test_targets, classes)).tolist()
        train_tasks.append(TaskDataset(train_base, train_indices, train_transform))
        memory_tasks.append(TaskDataset(train_base, train_indices, evaluation_transform))
        test_tasks.append(TaskDataset(test_base, test_indices, evaluation_transform))
    return train_tasks, memory_tasks, test_tasks, num_classes


@torch.no_grad()
def evaluate(model, datasets_by_task, device, batch_size, workers):
    model.eval()
    accuracies = []
    for dataset in datasets_by_task:
        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=workers,
            pin_memory=device.type == "cuda",
        )
        correct = 0
        total = 0
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            predictions = model(images).argmax(dim=1)
            correct += int((predictions == labels).sum())
            total += labels.numel()
        accuracies.append(correct / max(total, 1))
    return accuracies


@torch.no_grad()
def feature_quality(memory: ReplayMemory) -> Dict[str, Dict[str, float]]:
    labels = torch.tensor(memory.labels)
    result = {}
    for name, features in (
        ("shallow", memory.shallow_features),
        ("semantic", memory.semantic_features),
    ):
        if features is None or len(features) < 2:
            continue
        normalized = F.normalize(features, dim=1)
        classes = labels.unique(sorted=True)
        centroids = torch.stack(
            [F.normalize(normalized[labels == class_id].mean(0), dim=0) for class_id in classes]
        )
        centroid_predictions = classes[(normalized @ centroids.T).argmax(dim=1)]
        centroid_accuracy = float((centroid_predictions == labels).float().mean())

        within_values = []
        between_values = []
        for class_id in classes:
            mask = labels == class_id
            class_features = normalized[mask]
            centroid = F.normalize(class_features.mean(0), dim=0)
            within_values.append(float((class_features @ centroid).mean()))
        if len(centroids) > 1:
            matrix = centroids @ centroids.T
            upper = torch.triu_indices(len(centroids), len(centroids), offset=1)
            between_values = matrix[upper[0], upper[1]].tolist()
        mean_within = float(np.mean(within_values))
        mean_between = float(np.mean(between_values)) if between_values else 0.0
        result[name] = {
            "nearest_centroid_accuracy": centroid_accuracy,
            "mean_within_class_cosine": mean_within,
            "mean_between_class_centroid_cosine": mean_between,
            "separation_margin": mean_within - mean_between,
        }
    return result


def backward_transfer(accuracy_matrix: Sequence[Sequence[float]]) -> float:
    if len(accuracy_matrix) <= 1:
        return 0.0
    final_row = accuracy_matrix[-1]
    differences = [
        final_row[task] - accuracy_matrix[task][task]
        for task in range(len(accuracy_matrix) - 1)
    ]
    return float(np.mean(differences))


def run_condition(args, condition: str, seed: int, device: torch.device):
    seed_everything(seed)
    train_tasks, memory_tasks, test_tasks, num_classes = build_tasks(
        args.dataset, args.data_root, args.classes_per_task, not args.no_download
    )
    model = SmallCifarCNN(num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    memory = ReplayMemory(
        per_class_capacity=args.memory_per_class,
        temperature=args.temperature,
        floor_frac=args.floor_frac,
        seed=seed + 10_000,
        feature_transform=args.feature_transform,
        score_normalization=args.score_normalization,
        actr_feature=args.actr_feature,
        actr_alpha=args.actr_alpha,
        actr_beta=args.actr_beta,
        actr_gamma=args.actr_gamma,
        actr_decay=args.actr_decay,
        actr_noise=args.actr_noise,
    )
    accuracy_matrix = []
    quality_by_task = {}
    global_step = 0

    for task_id, train_dataset in enumerate(train_tasks):
        if not args.no_reset_optimizer and task_id > 0:
            optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
        if len(memory) and (
            memory.shallow_features is None
            or task_id % max(args.refresh_every, 1) == 0
        ):
            memory.refresh_features(model, device)

        loader_generator = torch.Generator().manual_seed(seed * 1000 + task_id)
        train_loader = DataLoader(
            train_dataset,
            batch_size=args.batch_size,
            shuffle=True,
            num_workers=args.workers,
            pin_memory=device.type == "cuda",
            generator=loader_generator,
            worker_init_fn=make_worker_init(seed * 1000 + task_id),
        )
        model.train()
        for epoch in range(args.epochs):
            running_loss = 0.0
            running_current_loss = 0.0
            running_replay_loss = 0.0
            for images, labels in train_loader:
                global_step += 1
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                current_logits = model(images)
                current_loss = F.cross_entropy(current_logits, labels)
                loss = current_loss
                replay_loss_value = 0.0

                if len(memory):
                    replay_images, replay_labels = memory.select(
                        condition=condition,
                        query_images=images,
                        model=model,
                        replay_size=args.replay_batch_size,
                        device=device,
                        task_id=task_id,
                        global_step=global_step,
                    )
                    replay_logits = model(replay_images)
                    replay_loss = F.cross_entropy(replay_logits, replay_labels)
                    if (
                        args.gradient_diagnostics_every > 0
                        and global_step % args.gradient_diagnostics_every == 0
                    ):
                        memory.record_gradient_diagnostics(
                            current_loss, replay_loss, model
                        )
                    loss = current_loss + args.replay_weight * replay_loss
                    replay_loss_value = float(replay_loss.detach())

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
                optimizer.step()
                running_loss += float(loss.detach())
                running_current_loss += float(current_loss.detach())
                running_replay_loss += replay_loss_value

            batches = max(len(train_loader), 1)
            print(
                f"  [{condition}] task {task_id} epoch {epoch}: "
                f"total={running_loss / batches:.4f} "
                f"current={running_current_loss / batches:.4f} "
                f"replay={running_replay_loss / batches:.4f}"
            )

        # The current task enters memory only after its own training. Consequently,
        # the final task is correctly excluded from retrieval-opportunity analysis.
        memory.add_task_data(memory_tasks[task_id], task_id, current_step=global_step)
        memory.refresh_features(model, device)
        quality_by_task[str(task_id)] = feature_quality(memory)
        accuracies = evaluate(
            model,
            test_tasks[: task_id + 1],
            device,
            args.eval_batch_size,
            args.workers,
        )
        accuracy_matrix.append(accuracies)
        formatted = ", ".join(f"T{i}={value:.3f}" for i, value in enumerate(accuracies))
        print(
            f"[{condition}] after task {task_id}: {formatted} "
            f"| learned_current={accuracies[task_id]:.3f}"
        )

    eligible_classes = list(range(num_classes - args.classes_per_task))
    diagnostics = memory.diagnostics(eligible_classes)
    diagonal = [accuracy_matrix[task][task] for task in range(len(accuracy_matrix))]
    dead_tasks = [task for task, value in enumerate(diagonal) if value < args.plasticity_floor]
    # BWT compares final accuracy against the diagonal. If the diagonal is ~0 the
    # model never learned the task, so BWT trends toward 0 and looks like "less
    # forgetting" when it actually reflects a failure to learn.
    bwt_interpretable = len(dead_tasks) == 0
    health = {
        "diagonal_accuracy": diagonal,
        "mean_diagonal_accuracy": float(np.mean(diagonal)),
        "tasks_below_plasticity_floor": dead_tasks,
        "plasticity_floor": args.plasticity_floor,
        "bwt_interpretable": bwt_interpretable,
        "optimizer_steps_per_task": int(
            math.ceil(len(train_tasks[0]) / args.batch_size) * args.epochs
        ),
    }
    result = {
        "condition": condition,
        "seed": seed,
        "dataset": args.dataset,
        "avg_accuracy": float(np.mean(accuracy_matrix[-1])),
        "bwt": backward_transfer(accuracy_matrix),
        "accuracy_matrix": accuracy_matrix,
        "run_health": health,
        "feature_quality_by_task": quality_by_task,
        "retrieval_diagnostics": diagnostics,
        "config": vars(args),
    }
    print(
        f"[{condition} | seed {seed}] avg_acc={result['avg_accuracy']:.3f} "
        f"BWT={result['bwt']:.3f} "
        f"mean_diag={health['mean_diagonal_accuracy']:.3f}"
    )
    if not bwt_interpretable:
        print(
            f"[{condition}] WARNING: tasks {dead_tasks} never learned "
            f"(accuracy < {args.plasticity_floor}). BWT is NOT interpretable for this run."
        )
    print(f"[{condition}] RETRIEVAL DIAGNOSTICS")
    print(json.dumps(diagnostics, indent=2, allow_nan=True))
    return result, memory.events


def write_outputs(output_dir: Path, results: List[Dict], all_events: List[Dict]):
    output_dir.mkdir(parents=True, exist_ok=True)
    with (output_dir / "results.json").open("w", encoding="utf-8") as handle:
        json.dump(results, handle, indent=2, allow_nan=True)

    if all_events:
        with (output_dir / "replay_events.csv").open(
            "w", newline="", encoding="utf-8"
        ) as handle:
            writer = csv.DictWriter(handle, fieldnames=list(all_events[0]))
            writer.writeheader()
            writer.writerows(all_events)

    summary = []
    for condition in sorted({result["condition"] for result in results}):
        condition_results = [
            result for result in results if result["condition"] == condition
        ]
        accuracy = np.asarray([result["avg_accuracy"] for result in condition_results])
        bwt = np.asarray([result["bwt"] for result in condition_results])
        diagnostic_keys = (
            "retrieval_count_cv",
            "retrieval_count_gini",
            "exposure_normalized_ratio_cv",
            "observed_vs_exposure_expected_jsd",
            "sample_coverage_fraction",
            "sample_retrieval_gini",
            "mean_selected_pair_similarity",
            "mean_batch_class_entropy",
            "mean_unique_classes_per_replay",
            "mean_score_std",
            "mean_softmax_effective_fraction",
            "gradient_cosine_mean",
            "gradient_conflict_fraction",
        )
        row = {
            "condition": condition,
            "seeds": len(condition_results),
            "avg_accuracy_mean": float(accuracy.mean()),
            "avg_accuracy_std": float(accuracy.std()),
            "bwt_mean": float(bwt.mean()),
            "bwt_std": float(bwt.std()),
            "mean_diagonal_accuracy": float(
                np.mean(
                    [
                        result["run_health"]["mean_diagonal_accuracy"]
                        for result in condition_results
                    ]
                )
            ),
            "runs_with_interpretable_bwt": int(
                sum(
                    1
                    for result in condition_results
                    if result["run_health"]["bwt_interpretable"]
                )
            ),
        }
        for key in diagnostic_keys:
            values = np.asarray(
                [
                    result["retrieval_diagnostics"][key]
                    for result in condition_results
                ],
                dtype=np.float64,
            )
            # Random replay computes no similarity scores, so score-based metrics
            # are legitimately all-NaN for that condition.
            if values.size == 0 or np.all(np.isnan(values)):
                row[f"{key}_mean"] = float("nan")
                row[f"{key}_std"] = float("nan")
            else:
                row[f"{key}_mean"] = float(np.nanmean(values))
                row[f"{key}_std"] = float(np.nanstd(values))
        summary.append(row)

    with (output_dir / "summary.csv").open(
        "w", newline="", encoding="utf-8"
    ) as handle:
        writer = csv.DictWriter(handle, fieldnames=list(summary[0]))
        writer.writeheader()
        writer.writerows(summary)

    summary_by_condition = {row["condition"]: row for row in summary}

    print("\n===== Summary (mean +/- std over seeds) =====")
    for row in summary:
        flag = "" if row["runs_with_interpretable_bwt"] == row["seeds"] else "  [BWT UNRELIABLE]"
        print(
            f"{row['condition']:10s} "
            f"avg_acc={row['avg_accuracy_mean']:.3f} +/- {row['avg_accuracy_std']:.3f}  "
            f"BWT={row['bwt_mean']:.3f} +/- {row['bwt_std']:.3f}  "
            f"mean_diag={row['mean_diagonal_accuracy']:.3f}  "
            f"exposure-JSD={row['observed_vs_exposure_expected_jsd_mean']:.4f}  "
            f"sample-coverage={row['sample_coverage_fraction_mean']:.3f}  "
            f"grad-conflict={row['gradient_conflict_fraction_mean']:.3f}{flag}"
        )

    diagnosis = {
        "status": "insufficient_conditions",
        "interpretation": [
            "Run both random and shallow conditions to generate a direct diagnosis."
        ],
    }
    if "random" in summary_by_condition and "shallow" in summary_by_condition:
        random_row = summary_by_condition["random"]
        shallow_row = summary_by_condition["shallow"]
        findings = []

        degenerate = [
            row["condition"]
            for row in summary
            if row["runs_with_interpretable_bwt"] < row["seeds"]
        ]
        if degenerate:
            diagnosis = {
                "status": "invalid_run",
                "degenerate_conditions": degenerate,
                "interpretation": [
                    "One or more conditions failed to learn some tasks, so their "
                    "diagonal accuracy is ~0 and BWT is driven by lack of plasticity "
                    "rather than by forgetting.",
                    "Fix plasticity first (smaller --batch-size, more --epochs, or a "
                    "lower --replay-weight), then compare retrieval conditions.",
                ],
            }
            with (output_dir / "diagnosis.json").open("w", encoding="utf-8") as handle:
                json.dump(diagnosis, handle, indent=2, allow_nan=True)
            print("\n===== Diagnosis =====")
            print(json.dumps(diagnosis, indent=2))
            return

        random_jsd = random_row["observed_vs_exposure_expected_jsd_mean"]
        shallow_jsd = shallow_row["observed_vs_exposure_expected_jsd_mean"]
        random_coverage = random_row["sample_coverage_fraction_mean"]
        shallow_coverage = shallow_row["sample_coverage_fraction_mean"]
        if shallow_jsd > random_jsd * 1.25 and shallow_coverage < random_coverage - 0.05:
            findings.append(
                {
                    "mechanism": "coverage_bias",
                    "supported": True,
                    "reason": (
                        "Shallow retrieval departs more from its exposure-adjusted "
                        "uniform baseline and reaches fewer stored samples than random."
                    ),
                }
            )
        else:
            findings.append(
                {
                    "mechanism": "coverage_bias",
                    "supported": False,
                    "reason": (
                        "Shallow retrieval does not show both a materially larger "
                        "exposure-adjusted divergence and lower sample coverage."
                    ),
                }
            )

        random_pair = random_row["mean_selected_pair_similarity_mean"]
        shallow_pair = shallow_row["mean_selected_pair_similarity_mean"]
        findings.append(
            {
                "mechanism": "redundant_similarity_replay",
                "supported": bool(
                    np.isfinite(random_pair)
                    and np.isfinite(shallow_pair)
                    and shallow_pair > random_pair + 0.05
                ),
                "reason": (
                    "Selected shallow batches are more internally similar than random."
                    if np.isfinite(random_pair)
                    and np.isfinite(shallow_pair)
                    and shallow_pair > random_pair + 0.05
                    else "Shallow batches are not materially more redundant than random."
                ),
            }
        )

        random_conflict = random_row["gradient_conflict_fraction_mean"]
        shallow_conflict = shallow_row["gradient_conflict_fraction_mean"]
        findings.append(
            {
                "mechanism": "gradient_interference",
                "supported": bool(
                    np.isfinite(random_conflict)
                    and np.isfinite(shallow_conflict)
                    and shallow_conflict > random_conflict + 0.05
                ),
                "reason": (
                    "Shallow replay gradients conflict with current-task gradients more "
                    "often than random replay gradients."
                    if np.isfinite(random_conflict)
                    and np.isfinite(shallow_conflict)
                    and shallow_conflict > random_conflict + 0.05
                    else "Shallow replay does not show materially more gradient conflict."
                ),
            }
        )
        shallow_effective = shallow_row["mean_softmax_effective_fraction_mean"]
        semantic_effective = summary_by_condition.get("semantic", {}).get(
            "mean_softmax_effective_fraction_mean", float("nan")
        )
        findings.append(
            {
                "mechanism": "non_selective_retrieval",
                "supported": bool(
                    np.isfinite(shallow_effective) and shallow_effective > 0.5
                ),
                "shallow_softmax_effective_fraction": shallow_effective,
                "semantic_softmax_effective_fraction": semantic_effective,
                "reason": (
                    "The shallow retrieval distribution stays close to uniform, so "
                    "'shallow' is effectively random replay with extra compute."
                    if np.isfinite(shallow_effective) and shallow_effective > 0.5
                    else "Shallow retrieval is meaningfully peaked, so it is genuinely selective."
                ),
            }
        )
        diagnosis = {
            "status": "diagnosed",
            "thresholds": {
                "coverage_jsd_multiplier": 1.25,
                "sample_coverage_absolute_gap": 0.05,
                "pair_similarity_absolute_gap": 0.05,
                "gradient_conflict_absolute_gap": 0.05,
            },
            "findings": findings,
            "important_note": (
                "Raw class retrieval totals are age-confounded. Prefer the "
                "exposure-normalized JSD and observed/expected ratios when deciding "
                "whether similarity replay is biased."
            ),
        }

        if "actr" in summary_by_condition:
            actr_row = summary_by_condition["actr"]
            diagnosis["actr_intervention"] = {
                "coverage_recovered": bool(
                    actr_row["sample_coverage_fraction_mean"]
                    > shallow_row["sample_coverage_fraction_mean"] + 0.05
                ),
                "beats_shallow_on_bwt": bool(
                    actr_row["bwt_mean"] > shallow_row["bwt_mean"]
                ),
                "beats_random_on_bwt": bool(
                    actr_row["bwt_mean"] > random_row["bwt_mean"]
                ),
                "beats_random_on_accuracy": bool(
                    actr_row["avg_accuracy_mean"] > random_row["avg_accuracy_mean"]
                ),
                "sample_coverage": {
                    "random": random_row["sample_coverage_fraction_mean"],
                    "shallow": shallow_row["sample_coverage_fraction_mean"],
                    "actr": actr_row["sample_coverage_fraction_mean"],
                },
                "bwt": {
                    "random": random_row["bwt_mean"],
                    "shallow": shallow_row["bwt_mean"],
                    "actr": actr_row["bwt_mean"],
                },
                "note": (
                    "Recovering coverage without beating random means coverage was "
                    "necessary but not sufficient, which is itself a reportable result."
                ),
            }
    with (output_dir / "diagnosis.json").open("w", encoding="utf-8") as handle:
        json.dump(diagnosis, handle, indent=2, allow_nan=True)

    print("\n===== Diagnosis =====")
    print(json.dumps(diagnosis, indent=2, allow_nan=True))


def parse_arguments():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--condition",
        choices=("random", "semantic", "shallow", "actr", "all", "compare"),
        default="all",
    )
    parser.add_argument("--dataset", choices=("cifar10", "cifar100"), default="cifar100")
    parser.add_argument("--data-root", default="/kaggle/working/data")
    parser.add_argument("--output-dir", default="/kaggle/working/actr_diagnostics")
    parser.add_argument("--seeds", type=int, default=1)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--classes-per-task", type=int, default=10)
    parser.add_argument("--memory-per-class", type=int, default=40)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--eval-batch-size", type=int, default=256)
    parser.add_argument("--replay-batch-size", type=int, default=32)
    parser.add_argument("--replay-weight", type=float, default=1.0)
    parser.add_argument("--temperature", type=float, default=0.1)
    parser.add_argument("--floor-frac", type=float, default=0.0)
    parser.add_argument("--refresh-every", type=int, default=1)
    parser.add_argument("--gradient-diagnostics-every", type=int, default=25)
    parser.add_argument(
        "--feature-transform",
        choices=("none", "center", "standardize"),
        default="standardize",
    )
    parser.add_argument(
        "--score-normalization", choices=("none", "zscore"), default="zscore"
    )
    parser.add_argument("--plasticity-floor", type=float, default=0.05)
    parser.add_argument(
        "--actr-feature", choices=("shallow", "semantic"), default="shallow"
    )
    parser.add_argument("--actr-alpha", type=float, default=1.0)
    parser.add_argument("--actr-beta", type=float, default=-1.0)
    parser.add_argument("--actr-gamma", type=float, default=1.0)
    parser.add_argument("--actr-decay", type=float, default=0.5)
    parser.add_argument("--actr-noise", type=float, default=0.0)
    parser.add_argument("--grad-clip", type=float, default=5.0)
    parser.add_argument("--lr", type=float, default=5e-4)
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument("--no-reset-optimizer", action="store_true")
    parser.add_argument("--no-download", action="store_true")
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"Ignoring notebook arguments: {unknown}")
    if not 0.0 <= args.floor_frac <= 1.0:
        parser.error("--floor-frac must be between 0 and 1.")
    if not 0.0 < args.actr_decay < 1.0:
        parser.error("--actr-decay must be strictly between 0 and 1.")
    if args.seeds < 1 or args.epochs < 1:
        parser.error("--seeds and --epochs must be positive.")
    return args


def main():
    args = parse_arguments()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    if args.condition == "all":
        conditions = ["random", "semantic", "shallow"]
    elif args.condition == "compare":
        conditions = ["random", "shallow", "actr"]
    else:
        conditions = [args.condition]
    results = []
    all_events = []
    for condition in conditions:
        for seed in range(args.seeds):
            print(f"\n===== Running condition: {condition} | seed {seed} =====")
            result, events = run_condition(args, condition, seed, device)
            results.append(result)
            all_events.extend(
                {
                    "condition": condition,
                    "seed": seed,
                    **asdict(event),
                }
                for event in events
            )
    write_outputs(Path(args.output_dir), results, all_events)


if __name__ == "__main__":
    main()


Writing actr_diagnostic_experiment.py


In [2]:
!python actr_diagnostic_experiment.py \
  --data-root /kaggle/working/data \
  --output-dir /kaggle/working/actr_compare_5seeds \
  --dataset cifar100 \
  --condition compare \
  --seeds 5 \
  --epochs 5

Using device: cuda

===== Running condition: random | seed 0 =====
100%|█████████████████████████████████████████| 169M/169M [23:35<00:00, 119kB/s]
  [random] task 0 epoch 0: total=2.1336 current=2.1336 replay=0.0000
  [random] task 0 epoch 1: total=1.5204 current=1.5204 replay=0.0000
  [random] task 0 epoch 2: total=1.2464 current=1.2464 replay=0.0000
  [random] task 0 epoch 3: total=1.1048 current=1.1048 replay=0.0000
  [random] task 0 epoch 4: total=1.0038 current=1.0038 replay=0.0000
[random] after task 0: T0=0.656 | learned_current=0.656
  [random] task 1 epoch 0: total=4.1904 current=3.1857 replay=1.0048
  [random] task 1 epoch 1: total=2.0064 current=1.6137 replay=0.3926
  [random] task 1 epoch 2: total=1.4776 current=1.2985 replay=0.1791
  [random] task 1 epoch 3: total=1.3199 current=1.2033 replay=0.1165
  [random] task 1 epoch 4: total=1.1627 current=1.0841 replay=0.0786
[random] after task 1: T0=0.425, T1=0.361 | learned_current=0.361
  [random] task 2 epoch 0: total=4.6609 

In [1]:
%%writefile actr_diagnostic_experiment.py
#!/usr/bin/env python3
"""Kaggle-ready continual-learning experiment with retrieval diagnostics."""

import importlib.util
import subprocess
import sys


def install_missing_packages():
    packages = {
        "numpy": "numpy",
        "torch": "torch",
        "torchvision": "torchvision",
    }
    missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *missing]
        )


install_missing_packages()

import argparse
import csv
import json
import math
import os
import random
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms


@dataclass
class ReplayEvent:
    task: int
    step: int
    condition: str
    memory_size: int
    selected: int
    query_selected_similarity: float
    selected_pair_similarity: float
    selected_class_entropy: float
    selected_unique_classes: int
    score_std: float
    softmax_effective_fraction: float


class TaskDataset(Dataset):
    def __init__(self, base_dataset, indices: Sequence[int], transform):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform
        self.targets = [int(base_dataset.targets[i]) for i in self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        image, label = self.base_dataset[self.indices[index]]
        return self.transform(image), int(label)


class SmallCifarCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.projector = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
        )
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x, return_features: bool = False):
        shallow_map = self.block1(x)
        shallow = F.adaptive_avg_pool2d(shallow_map, 1).flatten(1)
        semantic = self.projector(self.block3(self.block2(shallow_map)))
        logits = self.classifier(semantic)
        if return_features:
            return logits, shallow, semantic
        return logits


def normalized_entropy(counts: Sequence[int]) -> float:
    values = np.asarray(counts, dtype=np.float64)
    values = values[values > 0]
    if len(values) <= 1:
        return 0.0
    probabilities = values / values.sum()
    return float(-(probabilities * np.log(probabilities)).sum() / np.log(len(values)))


def gini(values: Sequence[float]) -> float:
    array = np.asarray(values, dtype=np.float64)
    if array.size == 0 or np.allclose(array.sum(), 0.0):
        return 0.0
    array = np.sort(np.maximum(array, 0.0))
    index = np.arange(1, len(array) + 1)
    return float(
        (2.0 * np.sum(index * array) / (len(array) * array.sum()))
        - (len(array) + 1.0) / len(array)
    )


def distribution_jsd(left: Sequence[float], right: Sequence[float]) -> float:
    p = np.asarray(left, dtype=np.float64)
    q = np.asarray(right, dtype=np.float64)
    if p.sum() == 0 or q.sum() == 0:
        return 0.0
    p /= p.sum()
    q /= q.sum()
    middle = 0.5 * (p + q)

    def kl_divergence(a, b):
        mask = a > 0
        return float(np.sum(a[mask] * np.log2(a[mask] / b[mask])))

    return 0.5 * kl_divergence(p, middle) + 0.5 * kl_divergence(q, middle)


class ReplayMemory:
    def __init__(
        self,
        per_class_capacity: int,
        temperature: float,
        floor_frac: float,
        seed: int,
        feature_transform: str = "standardize",
        score_normalization: str = "zscore",
        actr_feature: str = "shallow",
        actr_alpha: float = 1.0,
        actr_beta: float = -1.0,
        actr_gamma: float = 1.0,
        actr_decay: float = 0.5,
        actr_noise: float = 0.0,
        max_per_class: int = 0,
    ):
        self.per_class_capacity = per_class_capacity
        self.temperature = temperature
        self.floor_frac = floor_frac
        self.feature_transform = feature_transform
        self.score_normalization = score_normalization
        self.actr_feature = actr_feature
        self.actr_alpha = actr_alpha
        self.actr_beta = actr_beta
        self.actr_gamma = actr_gamma
        self.actr_decay = actr_decay
        self.actr_noise = actr_noise
        self.max_per_class = max_per_class
        self.generator = torch.Generator().manual_seed(seed)
        self.images: List[torch.Tensor] = []
        self.labels: List[int] = []
        self.task_ids: List[int] = []
        self.sample_ids: List[str] = []
        self.retrieval_counts: List[int] = []
        self.expected_retrievals: List[float] = []
        self.last_retrieved_step: List[int] = []
        self.creation_step: List[int] = []
        self.class_retrieval_counts: Counter = Counter()
        self.shallow_features: Optional[torch.Tensor] = None
        self.semantic_features: Optional[torch.Tensor] = None
        self.shallow_stats: Optional[Dict[str, torch.Tensor]] = None
        self.semantic_stats: Optional[Dict[str, torch.Tensor]] = None
        self.raw_shallow_cosine_mean = float("nan")
        self.raw_semantic_cosine_mean = float("nan")
        self.transformed_shallow_cosine_mean = float("nan")
        self.transformed_semantic_cosine_mean = float("nan")
        self.events: List[ReplayEvent] = []
        self.gradient_cosines: List[float] = []
        self.gradient_current_norms: List[float] = []
        self.gradient_replay_norms: List[float] = []

    def __len__(self):
        return len(self.labels)

    def add_task_data(self, dataset: TaskDataset, task_id: int, current_step: int = 0):
        class_to_local_indices = defaultdict(list)
        for local_index, label in enumerate(dataset.targets):
            class_to_local_indices[label].append(local_index)

        for label in sorted(class_to_local_indices):
            candidates = class_to_local_indices[label]
            permutation = torch.randperm(len(candidates), generator=self.generator).tolist()
            chosen = [candidates[i] for i in permutation[: self.per_class_capacity]]
            for local_index in chosen:
                image, item_label = dataset[local_index]
                source_index = dataset.indices[local_index]
                self.images.append(image.cpu())
                self.labels.append(item_label)
                self.task_ids.append(task_id)
                self.sample_ids.append(f"task{task_id}:source{source_index}")
                self.retrieval_counts.append(0)
                self.expected_retrievals.append(0.0)
                self.last_retrieved_step.append(-1)
                self.creation_step.append(current_step)

        self.shallow_features = None
        self.semantic_features = None
        self.shallow_stats = None
        self.semantic_stats = None

    def _fit_transform_stats(self, raw: torch.Tensor) -> Dict[str, torch.Tensor]:
        mean = raw.mean(dim=0, keepdim=True)
        std = raw.std(dim=0, keepdim=True).clamp_min(1e-6)
        return {"mean": mean, "std": std}

    def _apply_transform(
        self, raw: torch.Tensor, stats: Optional[Dict[str, torch.Tensor]]
    ) -> torch.Tensor:
        if self.feature_transform != "none" and stats is not None:
            raw = raw - stats["mean"]
            if self.feature_transform == "standardize":
                raw = raw / stats["std"]
        return F.normalize(raw, dim=1)

    @torch.no_grad()
    def refresh_features(self, model: nn.Module, device: torch.device, batch_size: int = 256):
        if not self.images:
            return
        was_training = model.training
        model.eval()
        shallow_parts = []
        semantic_parts = []
        for start in range(0, len(self.images), batch_size):
            batch = torch.stack(self.images[start : start + batch_size]).to(device)
            _, shallow, semantic = model(batch, return_features=True)
            shallow_parts.append(shallow.cpu())
            semantic_parts.append(semantic.cpu())
        raw_shallow = torch.cat(shallow_parts)
        raw_semantic = torch.cat(semantic_parts)

        # Post-ReLU pooled activations are non-negative, so raw cosine similarity
        # saturates near 1 and carries almost no ranking signal. Centering (and
        # optionally scaling) restores usable geometry before normalization.
        self.shallow_stats = self._fit_transform_stats(raw_shallow)
        self.semantic_stats = self._fit_transform_stats(raw_semantic)
        self.raw_shallow_cosine_mean = float(
            self._mean_pairwise_cosine(F.normalize(raw_shallow, dim=1))
        )
        self.raw_semantic_cosine_mean = float(
            self._mean_pairwise_cosine(F.normalize(raw_semantic, dim=1))
        )
        self.shallow_features = self._apply_transform(raw_shallow, self.shallow_stats)
        self.semantic_features = self._apply_transform(raw_semantic, self.semantic_stats)
        self.transformed_shallow_cosine_mean = float(
            self._mean_pairwise_cosine(self.shallow_features)
        )
        self.transformed_semantic_cosine_mean = float(
            self._mean_pairwise_cosine(self.semantic_features)
        )
        model.train(was_training)

    @staticmethod
    def _mean_pairwise_cosine(features: torch.Tensor, sample_limit: int = 512) -> float:
        if len(features) < 2:
            return float("nan")
        # Memory is stored in task order, so an evenly spaced stride keeps the
        # estimate representative instead of biasing it toward the earliest task.
        if len(features) > sample_limit:
            positions = torch.linspace(0, len(features) - 1, sample_limit).long()
            subset = features[positions]
        else:
            subset = features
        matrix = subset @ subset.T
        upper = torch.triu_indices(len(subset), len(subset), offset=1)
        return float(matrix[upper[0], upper[1]].mean())

    def _update_expected_counts(self, replay_size: int):
        if not self.labels:
            return
        probability = min(replay_size, len(self.labels)) / len(self.labels)
        for index in range(len(self.labels)):
            self.expected_retrievals[index] += probability

    def _coverage_floor_indices(self, count: int) -> List[int]:
        if count <= 0:
            return []
        by_class = defaultdict(list)
        for index, label in enumerate(self.labels):
            by_class[label].append(index)
        classes = sorted(by_class)
        class_order = torch.randperm(len(classes), generator=self.generator).tolist()
        selected = []
        cursor = 0
        while len(selected) < count:
            label = classes[class_order[cursor % len(classes)]]
            candidates = by_class[label]
            chosen = candidates[
                torch.randint(len(candidates), (1,), generator=self.generator).item()
            ]
            if chosen not in selected:
                selected.append(chosen)
            cursor += 1
            if cursor > count * len(classes) * 4:
                break
        return selected

    def _random_indices(self, replay_size: int) -> List[int]:
        count = min(replay_size, len(self.labels))
        return torch.randperm(len(self.labels), generator=self.generator)[:count].tolist()

    def _base_level_activation(self, global_step: int) -> torch.Tensor:
        # ACT-R optimized-learning approximation:
        #   B_i = ln(n_i / (1 - d)) - d * ln(T_i)
        # n_i counts the encoding event plus every retrieval; T_i is the age of
        # the trace. High B_i means the item is already highly accessible.
        counts = torch.tensor(self.retrieval_counts, dtype=torch.float32) + 1.0
        age = (
            torch.full((len(self.labels),), float(global_step))
            - torch.tensor(self.creation_step, dtype=torch.float32)
        ).clamp_min(1.0)
        return torch.log(counts / (1.0 - self.actr_decay)) - self.actr_decay * torch.log(age)

    def _class_overrepresentation(self) -> torch.Tensor:
        # How much each item's class has been replayed relative to a uniform
        # share, expressed per item so it can be combined with the other terms.
        labels = torch.tensor(self.labels, dtype=torch.long)
        total = float(sum(self.class_retrieval_counts.values()))
        distinct = len(set(self.labels))
        if total <= 0 or distinct == 0:
            return torch.zeros(len(self.labels), dtype=torch.float32)
        uniform_share = total / distinct
        per_class = torch.tensor(
            [self.class_retrieval_counts.get(int(label), 0) for label in labels],
            dtype=torch.float32,
        )
        return per_class / uniform_share

    @staticmethod
    def _zscore(values: torch.Tensor) -> torch.Tensor:
        return (values - values.mean()) / values.std().clamp_min(1e-6)

    def _sample_with_class_cap(
        self,
        available_scores: torch.Tensor,
        available_indices: torch.Tensor,
        remaining: int,
    ) -> List[int]:
        # Draw items one at a time and retire a class once it reaches the cap.
        # This raises per-batch class diversity without discarding the ranking,
        # so cue relevance still decides which item represents each class.
        available_labels = torch.tensor(
            [self.labels[int(index)] for index in available_indices], dtype=torch.long
        )
        active = torch.ones(len(available_indices), dtype=torch.bool)
        class_counts = Counter()
        chosen: List[int] = []
        for _ in range(remaining):
            if not bool(active.any()):
                break
            masked = available_scores.masked_fill(~active, float("-inf"))
            probabilities = torch.softmax(masked / max(self.temperature, 1e-8), dim=0)
            position = int(
                torch.multinomial(probabilities, 1, generator=self.generator).item()
            )
            chosen.append(int(available_indices[position]))
            active[position] = False
            label = int(available_labels[position])
            class_counts[label] += 1
            if class_counts[label] >= self.max_per_class:
                active &= available_labels != label
        return chosen

    @torch.no_grad()
    def select(
        self,
        condition: str,
        query_images: torch.Tensor,
        model: nn.Module,
        replay_size: int,
        device: torch.device,
        task_id: int,
        global_step: int,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        count = min(replay_size, len(self.labels))
        if count == 0:
            raise RuntimeError("Replay selection requested from an empty memory.")

        self._update_expected_counts(count)
        query_selected_similarity = float("nan")
        score_std = float("nan")
        softmax_effective_fraction = float("nan")

        if condition == "random":
            selected = self._random_indices(count)
            diagnostic_features = self.semantic_features
        else:
            if condition == "shallow":
                feature_name = "shallow"
            elif condition == "semantic":
                feature_name = "semantic"
            else:
                feature_name = self.actr_feature
            features = (
                self.shallow_features if feature_name == "shallow" else self.semantic_features
            )
            stats = (
                self.shallow_stats if feature_name == "shallow" else self.semantic_stats
            )
            if features is None:
                raise RuntimeError("Memory features must be refreshed before similarity replay.")

            was_training = model.training
            model.eval()
            _, query_shallow, query_semantic = model(
                query_images.to(device), return_features=True
            )
            model.train(was_training)
            query_raw = (query_shallow if feature_name == "shallow" else query_semantic).cpu()
            query_features = self._apply_transform(query_raw, stats)

            # Max-over-query implements cue-driven retrieval without collapsing
            # a heterogeneous current batch into a potentially meaningless centroid.
            similarities = query_features @ features.T
            scores = similarities.max(dim=0).values

            if condition == "actr":
                # Cue relevance alone concentrates replay on a small subset. The
                # base-level and overrepresentation terms restore coverage by
                # discounting traces that are already accessible or already
                # over-replayed at the class level.
                scores = (
                    self.actr_alpha * self._zscore(scores)
                    + self.actr_beta * self._zscore(self._base_level_activation(global_step))
                    - self.actr_gamma * self._zscore(self._class_overrepresentation())
                )
                if self.actr_noise > 0:
                    uniform = torch.rand(
                        len(scores), generator=self.generator, dtype=torch.float32
                    ).clamp(1e-6, 1 - 1e-6)
                    scores = scores + self.actr_noise * torch.log(uniform / (1 - uniform))

            score_std = float(scores.std())
            floor_count = min(count, int(round(count * self.floor_frac)))
            floor_indices = self._coverage_floor_indices(floor_count)
            available_mask = torch.ones(len(self.labels), dtype=torch.bool)
            if floor_indices:
                available_mask[floor_indices] = False
            remaining = count - len(floor_indices)
            available_indices = torch.where(available_mask)[0]
            if remaining:
                available_scores = scores[available_indices]
                # Absolute cosine ranges differ wildly between representations, so a
                # fixed temperature is not comparable across conditions. Z-scoring
                # makes the temperature control selectivity, not feature scale.
                if self.score_normalization == "zscore":
                    available_scores = (
                        available_scores - available_scores.mean()
                    ) / available_scores.std().clamp_min(1e-6)
                probabilities = torch.softmax(
                    available_scores / max(self.temperature, 1e-8), dim=0
                )
                softmax_effective_fraction = float(
                    torch.exp(
                        -(probabilities * torch.log(probabilities.clamp_min(1e-12))).sum()
                    )
                    / len(probabilities)
                )
                if self.max_per_class > 0:
                    similarity_indices = self._sample_with_class_cap(
                        available_scores, available_indices, remaining
                    )
                else:
                    sampled_positions = torch.multinomial(
                        probabilities,
                        remaining,
                        replacement=False,
                        generator=self.generator,
                    )
                    similarity_indices = available_indices[sampled_positions].tolist()
            else:
                similarity_indices = []
            selected = floor_indices + similarity_indices
            query_selected_similarity = float(similarities[:, selected].max(dim=0).values.mean())
            diagnostic_features = features

        for index in selected:
            self.retrieval_counts[index] += 1
            self.last_retrieved_step[index] = global_step
            self.class_retrieval_counts[self.labels[index]] += 1

        selected_labels = [self.labels[index] for index in selected]
        label_counts = Counter(selected_labels)
        pair_similarity = float("nan")
        if diagnostic_features is not None and len(selected) > 1:
            chosen_features = F.normalize(diagnostic_features[selected], dim=1)
            similarity_matrix = chosen_features @ chosen_features.T
            upper = torch.triu_indices(len(selected), len(selected), offset=1)
            pair_similarity = float(similarity_matrix[upper[0], upper[1]].mean())

        self.events.append(
            ReplayEvent(
                task=task_id,
                step=global_step,
                condition=condition,
                memory_size=len(self.labels),
                selected=len(selected),
                query_selected_similarity=query_selected_similarity,
                selected_pair_similarity=pair_similarity,
                selected_class_entropy=normalized_entropy(list(label_counts.values())),
                selected_unique_classes=len(label_counts),
                score_std=score_std,
                softmax_effective_fraction=softmax_effective_fraction,
            )
        )
        images = torch.stack([self.images[index] for index in selected]).to(device)
        labels = torch.tensor(
            [self.labels[index] for index in selected], dtype=torch.long, device=device
        )
        return images, labels

    def record_gradient_diagnostics(
        self,
        current_loss: torch.Tensor,
        replay_loss: torch.Tensor,
        model: nn.Module,
    ):
        parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
        current_gradients = torch.autograd.grad(
            current_loss, parameters, retain_graph=True, allow_unused=True
        )
        replay_gradients = torch.autograd.grad(
            replay_loss, parameters, retain_graph=True, allow_unused=True
        )
        current_parts = []
        replay_parts = []
        for current, replay, parameter in zip(
            current_gradients, replay_gradients, parameters
        ):
            current_parts.append(
                current.detach().flatten()
                if current is not None
                else torch.zeros_like(parameter).flatten()
            )
            replay_parts.append(
                replay.detach().flatten()
                if replay is not None
                else torch.zeros_like(parameter).flatten()
            )
        current_vector = torch.cat(current_parts)
        replay_vector = torch.cat(replay_parts)
        current_norm = current_vector.norm()
        replay_norm = replay_vector.norm()
        cosine = F.cosine_similarity(current_vector, replay_vector, dim=0)
        self.gradient_cosines.append(float(cosine))
        self.gradient_current_norms.append(float(current_norm))
        self.gradient_replay_norms.append(float(replay_norm))

    def diagnostics(self, eligible_classes: Sequence[int]) -> Dict:
        eligible = set(eligible_classes)
        observed_by_class = Counter()
        expected_by_class = defaultdict(float)
        item_counts = []
        retrieved_items = 0
        eligible_items = 0
        task_counts = Counter()

        for index, label in enumerate(self.labels):
            if label not in eligible:
                continue
            observed = self.retrieval_counts[index]
            observed_by_class[label] += observed
            expected_by_class[label] += self.expected_retrievals[index]
            item_counts.append(observed)
            eligible_items += 1
            retrieved_items += int(observed > 0)
            task_counts[self.task_ids[index]] += observed

        classes = sorted(eligible)
        observed = np.asarray([observed_by_class[c] for c in classes], dtype=np.float64)
        expected = np.asarray([expected_by_class[c] for c in classes], dtype=np.float64)
        ratio = np.divide(
            observed,
            expected,
            out=np.zeros_like(observed),
            where=expected > 0,
        )
        nonzero_expected = expected > 0
        zero_classes = [
            class_id
            for class_id, obs, exp in zip(classes, observed, expected)
            if exp > 0 and obs == 0
        ]
        ranked_classes = sorted(
            ((class_id, int(observed_by_class[class_id])) for class_id in classes),
            key=lambda item: (item[1], item[0]),
        )
        event_values = {
            field: [
                getattr(event, field)
                for event in self.events
                if not math.isnan(getattr(event, field))
            ]
            for field in (
                "query_selected_similarity",
                "selected_pair_similarity",
                "selected_class_entropy",
                "selected_unique_classes",
                "score_std",
                "softmax_effective_fraction",
            )
        }

        def mean_or_nan(values):
            return float(np.mean(values)) if values else float("nan")

        return {
            "eligible_classes": classes,
            "zero_retrieval_classes": zero_classes,
            "zero_retrieval_class_count": len(zero_classes),
            "least_retrieved_classes": [
                {"class": class_id, "count": count}
                for class_id, count in ranked_classes[:10]
            ],
            "most_retrieved_classes": [
                {"class": class_id, "count": count}
                for class_id, count in reversed(ranked_classes[-10:])
            ],
            "class_retrieval_counts": {
                str(class_id): int(observed_by_class[class_id]) for class_id in classes
            },
            "class_expected_uniform_counts": {
                str(class_id): round(float(expected_by_class[class_id]), 4)
                for class_id in classes
            },
            "class_observed_expected_ratio": {
                str(class_id): round(float(value), 4)
                for class_id, value in zip(classes, ratio)
            },
            "retrieval_count_mean": float(observed.mean()) if len(observed) else 0.0,
            "retrieval_count_std": float(observed.std()) if len(observed) else 0.0,
            "retrieval_count_cv": float(observed.std() / observed.mean())
            if len(observed) and observed.mean() > 0
            else 0.0,
            "retrieval_count_gini": gini(observed),
            "exposure_normalized_ratio_cv": float(ratio[nonzero_expected].std())
            if nonzero_expected.any()
            else 0.0,
            "observed_vs_exposure_expected_jsd": distribution_jsd(observed, expected),
            "sample_coverage_fraction": retrieved_items / max(eligible_items, 1),
            "sample_retrieval_gini": gini(item_counts),
            "retrievals_by_source_task": {
                str(task): int(count) for task, count in sorted(task_counts.items())
            },
            "mean_query_selected_similarity": mean_or_nan(
                event_values["query_selected_similarity"]
            ),
            "mean_selected_pair_similarity": mean_or_nan(
                event_values["selected_pair_similarity"]
            ),
            "mean_batch_class_entropy": mean_or_nan(
                event_values["selected_class_entropy"]
            ),
            "mean_unique_classes_per_replay": mean_or_nan(
                event_values["selected_unique_classes"]
            ),
            "mean_score_std": mean_or_nan(event_values["score_std"]),
            "mean_softmax_effective_fraction": mean_or_nan(
                event_values["softmax_effective_fraction"]
            ),
            "raw_shallow_cosine_mean": self.raw_shallow_cosine_mean,
            "raw_semantic_cosine_mean": self.raw_semantic_cosine_mean,
            "transformed_shallow_cosine_mean": self.transformed_shallow_cosine_mean,
            "transformed_semantic_cosine_mean": self.transformed_semantic_cosine_mean,
            "gradient_cosine_mean": mean_or_nan(self.gradient_cosines),
            "gradient_cosine_std": float(np.std(self.gradient_cosines))
            if self.gradient_cosines
            else float("nan"),
            "gradient_conflict_fraction": float(
                np.mean(np.asarray(self.gradient_cosines) < 0)
            )
            if self.gradient_cosines
            else float("nan"),
            "gradient_current_norm_mean": mean_or_nan(self.gradient_current_norms),
            "gradient_replay_norm_mean": mean_or_nan(self.gradient_replay_norms),
        }


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_worker_init(seed: int):
    def initialize(worker_id: int):
        worker_seed = seed + worker_id
        random.seed(worker_seed)
        np.random.seed(worker_seed)
        torch.manual_seed(worker_seed)

    return initialize


def build_tasks(
    dataset_name: str,
    root: str,
    classes_per_task: int,
    download: bool,
):
    dataset_class = datasets.CIFAR100 if dataset_name == "cifar100" else datasets.CIFAR10
    num_classes = 100 if dataset_name == "cifar100" else 10
    if num_classes % classes_per_task:
        raise ValueError("classes_per_task must divide the dataset's class count.")

    train_base = dataset_class(root=root, train=True, transform=None, download=download)
    test_base = dataset_class(root=root, train=False, transform=None, download=download)
    mean = (0.5071, 0.4867, 0.4408) if dataset_name == "cifar100" else (0.4914, 0.4822, 0.4465)
    std = (0.2675, 0.2565, 0.2761) if dataset_name == "cifar100" else (0.2470, 0.2435, 0.2616)
    train_transform = transforms.Compose(
        [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    )
    evaluation_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean, std)]
    )

    train_targets = np.asarray(train_base.targets)
    test_targets = np.asarray(test_base.targets)
    train_tasks = []
    memory_tasks = []
    test_tasks = []
    for start in range(0, num_classes, classes_per_task):
        classes = np.arange(start, start + classes_per_task)
        train_indices = np.flatnonzero(np.isin(train_targets, classes)).tolist()
        test_indices = np.flatnonzero(np.isin(test_targets, classes)).tolist()
        train_tasks.append(TaskDataset(train_base, train_indices, train_transform))
        memory_tasks.append(TaskDataset(train_base, train_indices, evaluation_transform))
        test_tasks.append(TaskDataset(test_base, test_indices, evaluation_transform))
    return train_tasks, memory_tasks, test_tasks, num_classes


@torch.no_grad()
def evaluate(model, datasets_by_task, device, batch_size, workers):
    model.eval()
    accuracies = []
    for dataset in datasets_by_task:
        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=workers,
            pin_memory=device.type == "cuda",
        )
        correct = 0
        total = 0
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            predictions = model(images).argmax(dim=1)
            correct += int((predictions == labels).sum())
            total += labels.numel()
        accuracies.append(correct / max(total, 1))
    return accuracies


@torch.no_grad()
def feature_quality(memory: ReplayMemory) -> Dict[str, Dict[str, float]]:
    labels = torch.tensor(memory.labels)
    result = {}
    for name, features in (
        ("shallow", memory.shallow_features),
        ("semantic", memory.semantic_features),
    ):
        if features is None or len(features) < 2:
            continue
        normalized = F.normalize(features, dim=1)
        classes = labels.unique(sorted=True)
        centroids = torch.stack(
            [F.normalize(normalized[labels == class_id].mean(0), dim=0) for class_id in classes]
        )
        centroid_predictions = classes[(normalized @ centroids.T).argmax(dim=1)]
        centroid_accuracy = float((centroid_predictions == labels).float().mean())

        within_values = []
        between_values = []
        for class_id in classes:
            mask = labels == class_id
            class_features = normalized[mask]
            centroid = F.normalize(class_features.mean(0), dim=0)
            within_values.append(float((class_features @ centroid).mean()))
        if len(centroids) > 1:
            matrix = centroids @ centroids.T
            upper = torch.triu_indices(len(centroids), len(centroids), offset=1)
            between_values = matrix[upper[0], upper[1]].tolist()
        mean_within = float(np.mean(within_values))
        mean_between = float(np.mean(between_values)) if between_values else 0.0
        result[name] = {
            "nearest_centroid_accuracy": centroid_accuracy,
            "mean_within_class_cosine": mean_within,
            "mean_between_class_centroid_cosine": mean_between,
            "separation_margin": mean_within - mean_between,
        }
    return result


def backward_transfer(accuracy_matrix: Sequence[Sequence[float]]) -> float:
    if len(accuracy_matrix) <= 1:
        return 0.0
    final_row = accuracy_matrix[-1]
    differences = [
        final_row[task] - accuracy_matrix[task][task]
        for task in range(len(accuracy_matrix) - 1)
    ]
    return float(np.mean(differences))


def run_condition(args, condition: str, seed: int, device: torch.device):
    seed_everything(seed)
    train_tasks, memory_tasks, test_tasks, num_classes = build_tasks(
        args.dataset, args.data_root, args.classes_per_task, not args.no_download
    )
    model = SmallCifarCNN(num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    memory = ReplayMemory(
        per_class_capacity=args.memory_per_class,
        temperature=args.temperature,
        floor_frac=args.floor_frac,
        seed=seed + 10_000,
        feature_transform=args.feature_transform,
        score_normalization=args.score_normalization,
        actr_feature=args.actr_feature,
        actr_alpha=args.actr_alpha,
        actr_beta=args.actr_beta,
        actr_gamma=args.actr_gamma,
        actr_decay=args.actr_decay,
        actr_noise=args.actr_noise,
        max_per_class=args.max_per_class,
    )
    accuracy_matrix = []
    quality_by_task = {}
    global_step = 0

    for task_id, train_dataset in enumerate(train_tasks):
        if not args.no_reset_optimizer and task_id > 0:
            optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
        if len(memory) and (
            memory.shallow_features is None
            or task_id % max(args.refresh_every, 1) == 0
        ):
            memory.refresh_features(model, device)

        loader_generator = torch.Generator().manual_seed(seed * 1000 + task_id)
        train_loader = DataLoader(
            train_dataset,
            batch_size=args.batch_size,
            shuffle=True,
            num_workers=args.workers,
            pin_memory=device.type == "cuda",
            generator=loader_generator,
            worker_init_fn=make_worker_init(seed * 1000 + task_id),
        )
        model.train()
        for epoch in range(args.epochs):
            running_loss = 0.0
            running_current_loss = 0.0
            running_replay_loss = 0.0
            for images, labels in train_loader:
                global_step += 1
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                current_logits = model(images)
                current_loss = F.cross_entropy(current_logits, labels)
                loss = current_loss
                replay_loss_value = 0.0

                if len(memory):
                    replay_images, replay_labels = memory.select(
                        condition=condition,
                        query_images=images,
                        model=model,
                        replay_size=args.replay_batch_size,
                        device=device,
                        task_id=task_id,
                        global_step=global_step,
                    )
                    replay_logits = model(replay_images)
                    replay_loss = F.cross_entropy(replay_logits, replay_labels)
                    if (
                        args.gradient_diagnostics_every > 0
                        and global_step % args.gradient_diagnostics_every == 0
                    ):
                        memory.record_gradient_diagnostics(
                            current_loss, replay_loss, model
                        )
                    loss = current_loss + args.replay_weight * replay_loss
                    replay_loss_value = float(replay_loss.detach())

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
                optimizer.step()
                running_loss += float(loss.detach())
                running_current_loss += float(current_loss.detach())
                running_replay_loss += replay_loss_value

            batches = max(len(train_loader), 1)
            print(
                f"  [{condition}] task {task_id} epoch {epoch}: "
                f"total={running_loss / batches:.4f} "
                f"current={running_current_loss / batches:.4f} "
                f"replay={running_replay_loss / batches:.4f}"
            )

        # The current task enters memory only after its own training. Consequently,
        # the final task is correctly excluded from retrieval-opportunity analysis.
        memory.add_task_data(memory_tasks[task_id], task_id, current_step=global_step)
        memory.refresh_features(model, device)
        quality_by_task[str(task_id)] = feature_quality(memory)
        accuracies = evaluate(
            model,
            test_tasks[: task_id + 1],
            device,
            args.eval_batch_size,
            args.workers,
        )
        accuracy_matrix.append(accuracies)
        formatted = ", ".join(f"T{i}={value:.3f}" for i, value in enumerate(accuracies))
        print(
            f"[{condition}] after task {task_id}: {formatted} "
            f"| learned_current={accuracies[task_id]:.3f}"
        )

    eligible_classes = list(range(num_classes - args.classes_per_task))
    diagnostics = memory.diagnostics(eligible_classes)
    diagonal = [accuracy_matrix[task][task] for task in range(len(accuracy_matrix))]
    dead_tasks = [task for task, value in enumerate(diagonal) if value < args.plasticity_floor]
    # BWT compares final accuracy against the diagonal. If the diagonal is ~0 the
    # model never learned the task, so BWT trends toward 0 and looks like "less
    # forgetting" when it actually reflects a failure to learn.
    bwt_interpretable = len(dead_tasks) == 0
    health = {
        "diagonal_accuracy": diagonal,
        "mean_diagonal_accuracy": float(np.mean(diagonal)),
        "tasks_below_plasticity_floor": dead_tasks,
        "plasticity_floor": args.plasticity_floor,
        "bwt_interpretable": bwt_interpretable,
        "optimizer_steps_per_task": int(
            math.ceil(len(train_tasks[0]) / args.batch_size) * args.epochs
        ),
    }
    result = {
        "condition": condition,
        "seed": seed,
        "dataset": args.dataset,
        "avg_accuracy": float(np.mean(accuracy_matrix[-1])),
        "bwt": backward_transfer(accuracy_matrix),
        "accuracy_matrix": accuracy_matrix,
        "run_health": health,
        "feature_quality_by_task": quality_by_task,
        "retrieval_diagnostics": diagnostics,
        "config": vars(args),
    }
    print(
        f"[{condition} | seed {seed}] avg_acc={result['avg_accuracy']:.3f} "
        f"BWT={result['bwt']:.3f} "
        f"mean_diag={health['mean_diagonal_accuracy']:.3f}"
    )
    if not bwt_interpretable:
        print(
            f"[{condition}] WARNING: tasks {dead_tasks} never learned "
            f"(accuracy < {args.plasticity_floor}). BWT is NOT interpretable for this run."
        )
    print(f"[{condition}] RETRIEVAL DIAGNOSTICS")
    print(json.dumps(diagnostics, indent=2, allow_nan=True))
    return result, memory.events


def write_outputs(output_dir: Path, results: List[Dict], all_events: List[Dict]):
    output_dir.mkdir(parents=True, exist_ok=True)
    with (output_dir / "results.json").open("w", encoding="utf-8") as handle:
        json.dump(results, handle, indent=2, allow_nan=True)

    if all_events:
        with (output_dir / "replay_events.csv").open(
            "w", newline="", encoding="utf-8"
        ) as handle:
            writer = csv.DictWriter(handle, fieldnames=list(all_events[0]))
            writer.writeheader()
            writer.writerows(all_events)

    summary = []
    for condition in sorted({result["condition"] for result in results}):
        condition_results = [
            result for result in results if result["condition"] == condition
        ]
        accuracy = np.asarray([result["avg_accuracy"] for result in condition_results])
        bwt = np.asarray([result["bwt"] for result in condition_results])
        diagnostic_keys = (
            "retrieval_count_cv",
            "retrieval_count_gini",
            "exposure_normalized_ratio_cv",
            "observed_vs_exposure_expected_jsd",
            "sample_coverage_fraction",
            "sample_retrieval_gini",
            "mean_selected_pair_similarity",
            "mean_batch_class_entropy",
            "mean_unique_classes_per_replay",
            "mean_score_std",
            "mean_softmax_effective_fraction",
            "gradient_cosine_mean",
            "gradient_conflict_fraction",
        )
        row = {
            "condition": condition,
            "seeds": len(condition_results),
            "avg_accuracy_mean": float(accuracy.mean()),
            "avg_accuracy_std": float(accuracy.std()),
            "bwt_mean": float(bwt.mean()),
            "bwt_std": float(bwt.std()),
            "mean_diagonal_accuracy": float(
                np.mean(
                    [
                        result["run_health"]["mean_diagonal_accuracy"]
                        for result in condition_results
                    ]
                )
            ),
            "runs_with_interpretable_bwt": int(
                sum(
                    1
                    for result in condition_results
                    if result["run_health"]["bwt_interpretable"]
                )
            ),
        }
        for key in diagnostic_keys:
            values = np.asarray(
                [
                    result["retrieval_diagnostics"][key]
                    for result in condition_results
                ],
                dtype=np.float64,
            )
            # Random replay computes no similarity scores, so score-based metrics
            # are legitimately all-NaN for that condition.
            if values.size == 0 or np.all(np.isnan(values)):
                row[f"{key}_mean"] = float("nan")
                row[f"{key}_std"] = float("nan")
            else:
                row[f"{key}_mean"] = float(np.nanmean(values))
                row[f"{key}_std"] = float(np.nanstd(values))
        summary.append(row)

    with (output_dir / "summary.csv").open(
        "w", newline="", encoding="utf-8"
    ) as handle:
        writer = csv.DictWriter(handle, fieldnames=list(summary[0]))
        writer.writeheader()
        writer.writerows(summary)

    summary_by_condition = {row["condition"]: row for row in summary}

    print("\n===== Summary (mean +/- std over seeds) =====")
    for row in summary:
        flag = "" if row["runs_with_interpretable_bwt"] == row["seeds"] else "  [BWT UNRELIABLE]"
        print(
            f"{row['condition']:10s} "
            f"avg_acc={row['avg_accuracy_mean']:.3f} +/- {row['avg_accuracy_std']:.3f}  "
            f"BWT={row['bwt_mean']:.3f} +/- {row['bwt_std']:.3f}  "
            f"mean_diag={row['mean_diagonal_accuracy']:.3f}  "
            f"exposure-JSD={row['observed_vs_exposure_expected_jsd_mean']:.4f}  "
            f"sample-coverage={row['sample_coverage_fraction_mean']:.3f}  "
            f"grad-conflict={row['gradient_conflict_fraction_mean']:.3f}{flag}"
        )

    diagnosis = {
        "status": "insufficient_conditions",
        "interpretation": [
            "Run both random and shallow conditions to generate a direct diagnosis."
        ],
    }
    if "random" in summary_by_condition and "shallow" in summary_by_condition:
        random_row = summary_by_condition["random"]
        shallow_row = summary_by_condition["shallow"]
        findings = []

        degenerate = [
            row["condition"]
            for row in summary
            if row["runs_with_interpretable_bwt"] < row["seeds"]
        ]
        if degenerate:
            diagnosis = {
                "status": "invalid_run",
                "degenerate_conditions": degenerate,
                "interpretation": [
                    "One or more conditions failed to learn some tasks, so their "
                    "diagonal accuracy is ~0 and BWT is driven by lack of plasticity "
                    "rather than by forgetting.",
                    "Fix plasticity first (smaller --batch-size, more --epochs, or a "
                    "lower --replay-weight), then compare retrieval conditions.",
                ],
            }
            with (output_dir / "diagnosis.json").open("w", encoding="utf-8") as handle:
                json.dump(diagnosis, handle, indent=2, allow_nan=True)
            print("\n===== Diagnosis =====")
            print(json.dumps(diagnosis, indent=2))
            return

        random_jsd = random_row["observed_vs_exposure_expected_jsd_mean"]
        shallow_jsd = shallow_row["observed_vs_exposure_expected_jsd_mean"]
        random_coverage = random_row["sample_coverage_fraction_mean"]
        shallow_coverage = shallow_row["sample_coverage_fraction_mean"]
        if shallow_jsd > random_jsd * 1.25 and shallow_coverage < random_coverage - 0.05:
            findings.append(
                {
                    "mechanism": "coverage_bias",
                    "supported": True,
                    "reason": (
                        "Shallow retrieval departs more from its exposure-adjusted "
                        "uniform baseline and reaches fewer stored samples than random."
                    ),
                }
            )
        else:
            findings.append(
                {
                    "mechanism": "coverage_bias",
                    "supported": False,
                    "reason": (
                        "Shallow retrieval does not show both a materially larger "
                        "exposure-adjusted divergence and lower sample coverage."
                    ),
                }
            )

        random_pair = random_row["mean_selected_pair_similarity_mean"]
        shallow_pair = shallow_row["mean_selected_pair_similarity_mean"]
        findings.append(
            {
                "mechanism": "redundant_similarity_replay",
                "supported": bool(
                    np.isfinite(random_pair)
                    and np.isfinite(shallow_pair)
                    and shallow_pair > random_pair + 0.05
                ),
                "reason": (
                    "Selected shallow batches are more internally similar than random."
                    if np.isfinite(random_pair)
                    and np.isfinite(shallow_pair)
                    and shallow_pair > random_pair + 0.05
                    else "Shallow batches are not materially more redundant than random."
                ),
            }
        )

        random_conflict = random_row["gradient_conflict_fraction_mean"]
        shallow_conflict = shallow_row["gradient_conflict_fraction_mean"]
        findings.append(
            {
                "mechanism": "gradient_interference",
                "supported": bool(
                    np.isfinite(random_conflict)
                    and np.isfinite(shallow_conflict)
                    and shallow_conflict > random_conflict + 0.05
                ),
                "reason": (
                    "Shallow replay gradients conflict with current-task gradients more "
                    "often than random replay gradients."
                    if np.isfinite(random_conflict)
                    and np.isfinite(shallow_conflict)
                    and shallow_conflict > random_conflict + 0.05
                    else "Shallow replay does not show materially more gradient conflict."
                ),
            }
        )
        shallow_effective = shallow_row["mean_softmax_effective_fraction_mean"]
        semantic_effective = summary_by_condition.get("semantic", {}).get(
            "mean_softmax_effective_fraction_mean", float("nan")
        )
        findings.append(
            {
                "mechanism": "non_selective_retrieval",
                "supported": bool(
                    np.isfinite(shallow_effective) and shallow_effective > 0.5
                ),
                "shallow_softmax_effective_fraction": shallow_effective,
                "semantic_softmax_effective_fraction": semantic_effective,
                "reason": (
                    "The shallow retrieval distribution stays close to uniform, so "
                    "'shallow' is effectively random replay with extra compute."
                    if np.isfinite(shallow_effective) and shallow_effective > 0.5
                    else "Shallow retrieval is meaningfully peaked, so it is genuinely selective."
                ),
            }
        )
        diagnosis = {
            "status": "diagnosed",
            "thresholds": {
                "coverage_jsd_multiplier": 1.25,
                "sample_coverage_absolute_gap": 0.05,
                "pair_similarity_absolute_gap": 0.05,
                "gradient_conflict_absolute_gap": 0.05,
            },
            "findings": findings,
            "important_note": (
                "Raw class retrieval totals are age-confounded. Prefer the "
                "exposure-normalized JSD and observed/expected ratios when deciding "
                "whether similarity replay is biased."
            ),
        }

        if "actr" in summary_by_condition:
            actr_row = summary_by_condition["actr"]
            diagnosis["actr_intervention"] = {
                "coverage_recovered": bool(
                    actr_row["sample_coverage_fraction_mean"]
                    > shallow_row["sample_coverage_fraction_mean"] + 0.05
                ),
                "beats_shallow_on_bwt": bool(
                    actr_row["bwt_mean"] > shallow_row["bwt_mean"]
                ),
                "beats_random_on_bwt": bool(
                    actr_row["bwt_mean"] > random_row["bwt_mean"]
                ),
                "beats_random_on_accuracy": bool(
                    actr_row["avg_accuracy_mean"] > random_row["avg_accuracy_mean"]
                ),
                "sample_coverage": {
                    "random": random_row["sample_coverage_fraction_mean"],
                    "shallow": shallow_row["sample_coverage_fraction_mean"],
                    "actr": actr_row["sample_coverage_fraction_mean"],
                },
                "bwt": {
                    "random": random_row["bwt_mean"],
                    "shallow": shallow_row["bwt_mean"],
                    "actr": actr_row["bwt_mean"],
                },
                "note": (
                    "Recovering coverage without beating random means coverage was "
                    "necessary but not sufficient, which is itself a reportable result."
                ),
            }
    with (output_dir / "diagnosis.json").open("w", encoding="utf-8") as handle:
        json.dump(diagnosis, handle, indent=2, allow_nan=True)

    print("\n===== Diagnosis =====")
    print(json.dumps(diagnosis, indent=2, allow_nan=True))


def parse_arguments():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--condition",
        choices=("random", "semantic", "shallow", "actr", "all", "compare"),
        default="all",
    )
    parser.add_argument("--dataset", choices=("cifar10", "cifar100"), default="cifar100")
    parser.add_argument("--data-root", default="/kaggle/working/data")
    parser.add_argument("--output-dir", default="/kaggle/working/actr_diagnostics")
    parser.add_argument("--seeds", type=int, default=1)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--classes-per-task", type=int, default=10)
    parser.add_argument("--memory-per-class", type=int, default=40)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--eval-batch-size", type=int, default=256)
    parser.add_argument("--replay-batch-size", type=int, default=32)
    parser.add_argument("--replay-weight", type=float, default=1.0)
    parser.add_argument("--temperature", type=float, default=0.1)
    parser.add_argument("--floor-frac", type=float, default=0.0)
    parser.add_argument("--refresh-every", type=int, default=1)
    parser.add_argument("--gradient-diagnostics-every", type=int, default=25)
    parser.add_argument(
        "--feature-transform",
        choices=("none", "center", "standardize"),
        default="standardize",
    )
    parser.add_argument(
        "--score-normalization", choices=("none", "zscore"), default="zscore"
    )
    parser.add_argument("--plasticity-floor", type=float, default=0.05)
    parser.add_argument(
        "--actr-feature", choices=("shallow", "semantic"), default="shallow"
    )
    parser.add_argument("--actr-alpha", type=float, default=1.0)
    parser.add_argument("--actr-beta", type=float, default=-1.0)
    parser.add_argument("--actr-gamma", type=float, default=1.0)
    parser.add_argument("--actr-decay", type=float, default=0.5)
    parser.add_argument("--actr-noise", type=float, default=0.0)
    parser.add_argument(
        "--max-per-class",
        type=int,
        default=0,
        help="Cap items per class within a replay batch (0 disables the cap).",
    )
    parser.add_argument("--grad-clip", type=float, default=5.0)
    parser.add_argument("--lr", type=float, default=5e-4)
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument("--no-reset-optimizer", action="store_true")
    parser.add_argument("--no-download", action="store_true")
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"Ignoring notebook arguments: {unknown}")
    if not 0.0 <= args.floor_frac <= 1.0:
        parser.error("--floor-frac must be between 0 and 1.")
    if not 0.0 < args.actr_decay < 1.0:
        parser.error("--actr-decay must be strictly between 0 and 1.")
    if args.max_per_class < 0:
        parser.error("--max-per-class must be 0 (disabled) or positive.")
    if args.seeds < 1 or args.epochs < 1:
        parser.error("--seeds and --epochs must be positive.")
    return args


def main():
    args = parse_arguments()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    if args.condition == "all":
        conditions = ["random", "semantic", "shallow"]
    elif args.condition == "compare":
        conditions = ["random", "shallow", "actr"]
    else:
        conditions = [args.condition]
    results = []
    all_events = []
    for condition in conditions:
        for seed in range(args.seeds):
            print(f"\n===== Running condition: {condition} | seed {seed} =====")
            result, events = run_condition(args, condition, seed, device)
            results.append(result)
            all_events.extend(
                {
                    "condition": condition,
                    "seed": seed,
                    **asdict(event),
                }
                for event in events
            )
    write_outputs(Path(args.output_dir), results, all_events)


if __name__ == "__main__":
    main()


Writing actr_diagnostic_experiment.py


In [2]:
# class balance only (gamma)
!python actr_diagnostic_experiment.py --data-root /kaggle/working/data \
  --output-dir /kaggle/working/abl_gamma --condition actr \
  --seeds 5 --epochs 5 --actr-beta 0 --actr-gamma 1

# base-level only (beta)
!python actr_diagnostic_experiment.py --data-root /kaggle/working/data \
  --output-dir /kaggle/working/abl_beta --condition actr \
  --seeds 5 --epochs 5 --actr-beta -1 --actr-gamma 0

Using device: cuda

===== Running condition: actr | seed 0 =====
100%|████████████████████████████████████████| 169M/169M [00:35<00:00, 4.76MB/s]
  [actr] task 0 epoch 0: total=2.1336 current=2.1336 replay=0.0000
  [actr] task 0 epoch 1: total=1.5204 current=1.5204 replay=0.0000
  [actr] task 0 epoch 2: total=1.2464 current=1.2464 replay=0.0000
  [actr] task 0 epoch 3: total=1.1048 current=1.1048 replay=0.0000
  [actr] task 0 epoch 4: total=1.0038 current=1.0038 replay=0.0000
[actr] after task 0: T0=0.656 | learned_current=0.656
  [actr] task 1 epoch 0: total=4.2520 current=3.2391 replay=1.0129
  [actr] task 1 epoch 1: total=2.0172 current=1.5915 replay=0.4256
  [actr] task 1 epoch 2: total=1.5661 current=1.3060 replay=0.2601
  [actr] task 1 epoch 3: total=1.4203 current=1.2050 replay=0.2153
  [actr] task 1 epoch 4: total=1.2234 current=1.0945 replay=0.1289
[actr] after task 1: T0=0.414, T1=0.388 | learned_current=0.388
  [actr] task 2 epoch 0: total=4.0685 current=3.5087 replay=0.5597

In [3]:
!python actr_diagnostic_experiment.py --data-root /kaggle/working/data \
  --output-dir /kaggle/working/cap2 --condition actr \
  --seeds 5 --epochs 5 --max-per-class 2

Using device: cuda

===== Running condition: actr | seed 0 =====
  [actr] task 0 epoch 0: total=2.1336 current=2.1336 replay=0.0000
  [actr] task 0 epoch 1: total=1.5204 current=1.5204 replay=0.0000
  [actr] task 0 epoch 2: total=1.2464 current=1.2464 replay=0.0000
  [actr] task 0 epoch 3: total=1.1048 current=1.1048 replay=0.0000
  [actr] task 0 epoch 4: total=1.0038 current=1.0038 replay=0.0000
[actr] after task 0: T0=0.656 | learned_current=0.656
  [actr] task 1 epoch 0: total=4.1713 current=3.1868 replay=0.9845
  [actr] task 1 epoch 1: total=1.9856 current=1.6072 replay=0.3784
  [actr] task 1 epoch 2: total=1.4801 current=1.3126 replay=0.1675
  [actr] task 1 epoch 3: total=1.3109 current=1.2056 replay=0.1052
  [actr] task 1 epoch 4: total=1.1612 current=1.0932 replay=0.0680
[actr] after task 1: T0=0.517, T1=0.224 | learned_current=0.224
  [actr] task 2 epoch 0: total=4.5805 current=3.5385 replay=1.0419
  [actr] task 2 epoch 1: total=1.8817 current=1.4165 replay=0.4652
  [actr] task

In [1]:
%%writefile actr_diagnostic_experiment.py
#!/usr/bin/env python3
"""Kaggle-ready continual-learning experiment with retrieval diagnostics."""

import importlib.util
import subprocess
import sys


def install_missing_packages():
    packages = {
        "numpy": "numpy",
        "torch": "torch",
        "torchvision": "torchvision",
    }
    missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *missing]
        )


install_missing_packages()

import argparse
import csv
import json
import math
import os
import random
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms


@dataclass
class ReplayEvent:
    task: int
    step: int
    condition: str
    memory_size: int
    selected: int
    query_selected_similarity: float
    selected_pair_similarity: float
    selected_class_entropy: float
    selected_unique_classes: int
    score_std: float
    softmax_effective_fraction: float


class TaskDataset(Dataset):
    def __init__(self, base_dataset, indices: Sequence[int], transform):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform
        self.targets = [int(base_dataset.targets[i]) for i in self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        image, label = self.base_dataset[self.indices[index]]
        return self.transform(image), int(label)


class SmallCifarCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.projector = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
        )
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x, return_features: bool = False):
        shallow_map = self.block1(x)
        shallow = F.adaptive_avg_pool2d(shallow_map, 1).flatten(1)
        semantic = self.projector(self.block3(self.block2(shallow_map)))
        logits = self.classifier(semantic)
        if return_features:
            return logits, shallow, semantic
        return logits


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_planes, planes, 3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out, inplace=True)


class CifarResNet18(nn.Module):
    """ResNet-18 with a 3x3 stem and no max-pool, the standard CIFAR variant."""

    def __init__(self, num_classes: int):
        super().__init__()
        self.in_planes = 64
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.layer1 = self._make_layer(64, 2, 1)
        self.layer2 = self._make_layer(128, 2, 2)
        self.layer3 = self._make_layer(256, 2, 2)
        self.layer4 = self._make_layer(512, 2, 2)
        self.classifier = nn.Linear(512, num_classes)

    def _make_layer(self, planes: int, blocks: int, stride: int):
        layers = []
        for current_stride in [stride] + [1] * (blocks - 1):
            layers.append(BasicBlock(self.in_planes, planes, current_stride))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x, return_features: bool = False):
        # shallow == after layer1 (early perceptual), semantic == penultimate,
        # matching the two retrieval representations used by SmallCifarCNN.
        shallow_map = self.layer1(self.stem(x))
        shallow = F.adaptive_avg_pool2d(shallow_map, 1).flatten(1)
        deep = self.layer4(self.layer3(self.layer2(shallow_map)))
        semantic = F.adaptive_avg_pool2d(deep, 1).flatten(1)
        logits = self.classifier(semantic)
        if return_features:
            return logits, shallow, semantic
        return logits


def build_model(arch: str, num_classes: int) -> nn.Module:
    if arch == "resnet18":
        return CifarResNet18(num_classes)
    return SmallCifarCNN(num_classes)


def normalized_entropy(counts: Sequence[int]) -> float:
    values = np.asarray(counts, dtype=np.float64)
    values = values[values > 0]
    if len(values) <= 1:
        return 0.0
    probabilities = values / values.sum()
    return float(-(probabilities * np.log(probabilities)).sum() / np.log(len(values)))


def gini(values: Sequence[float]) -> float:
    array = np.asarray(values, dtype=np.float64)
    if array.size == 0 or np.allclose(array.sum(), 0.0):
        return 0.0
    array = np.sort(np.maximum(array, 0.0))
    index = np.arange(1, len(array) + 1)
    return float(
        (2.0 * np.sum(index * array) / (len(array) * array.sum()))
        - (len(array) + 1.0) / len(array)
    )


def distribution_jsd(left: Sequence[float], right: Sequence[float]) -> float:
    p = np.asarray(left, dtype=np.float64)
    q = np.asarray(right, dtype=np.float64)
    if p.sum() == 0 or q.sum() == 0:
        return 0.0
    p /= p.sum()
    q /= q.sum()
    middle = 0.5 * (p + q)

    def kl_divergence(a, b):
        mask = a > 0
        return float(np.sum(a[mask] * np.log2(a[mask] / b[mask])))

    return 0.5 * kl_divergence(p, middle) + 0.5 * kl_divergence(q, middle)


class ReplayMemory:
    def __init__(
        self,
        per_class_capacity: int,
        temperature: float,
        floor_frac: float,
        seed: int,
        feature_transform: str = "standardize",
        score_normalization: str = "zscore",
        actr_feature: str = "shallow",
        actr_alpha: float = 1.0,
        actr_beta: float = -1.0,
        actr_gamma: float = 1.0,
        actr_decay: float = 0.5,
        actr_noise: float = 0.0,
        max_per_class: int = 0,
    ):
        self.per_class_capacity = per_class_capacity
        self.temperature = temperature
        self.floor_frac = floor_frac
        self.feature_transform = feature_transform
        self.score_normalization = score_normalization
        self.actr_feature = actr_feature
        self.actr_alpha = actr_alpha
        self.actr_beta = actr_beta
        self.actr_gamma = actr_gamma
        self.actr_decay = actr_decay
        self.actr_noise = actr_noise
        self.max_per_class = max_per_class
        self.generator = torch.Generator().manual_seed(seed)
        self.images: List[torch.Tensor] = []
        self.labels: List[int] = []
        self.task_ids: List[int] = []
        self.sample_ids: List[str] = []
        self.retrieval_counts: List[int] = []
        self.expected_retrievals: List[float] = []
        self.last_retrieved_step: List[int] = []
        self.creation_step: List[int] = []
        self.class_retrieval_counts: Counter = Counter()
        self.shallow_features: Optional[torch.Tensor] = None
        self.semantic_features: Optional[torch.Tensor] = None
        self.shallow_stats: Optional[Dict[str, torch.Tensor]] = None
        self.semantic_stats: Optional[Dict[str, torch.Tensor]] = None
        self.raw_shallow_cosine_mean = float("nan")
        self.raw_semantic_cosine_mean = float("nan")
        self.transformed_shallow_cosine_mean = float("nan")
        self.transformed_semantic_cosine_mean = float("nan")
        self.events: List[ReplayEvent] = []
        self.gradient_cosines: List[float] = []
        self.gradient_current_norms: List[float] = []
        self.gradient_replay_norms: List[float] = []

    def __len__(self):
        return len(self.labels)

    def add_task_data(self, dataset: TaskDataset, task_id: int, current_step: int = 0):
        class_to_local_indices = defaultdict(list)
        for local_index, label in enumerate(dataset.targets):
            class_to_local_indices[label].append(local_index)

        for label in sorted(class_to_local_indices):
            candidates = class_to_local_indices[label]
            permutation = torch.randperm(len(candidates), generator=self.generator).tolist()
            chosen = [candidates[i] for i in permutation[: self.per_class_capacity]]
            for local_index in chosen:
                image, item_label = dataset[local_index]
                source_index = dataset.indices[local_index]
                self.images.append(image.cpu())
                self.labels.append(item_label)
                self.task_ids.append(task_id)
                self.sample_ids.append(f"task{task_id}:source{source_index}")
                self.retrieval_counts.append(0)
                self.expected_retrievals.append(0.0)
                self.last_retrieved_step.append(-1)
                self.creation_step.append(current_step)

        self.shallow_features = None
        self.semantic_features = None
        self.shallow_stats = None
        self.semantic_stats = None

    def _fit_transform_stats(self, raw: torch.Tensor) -> Dict[str, torch.Tensor]:
        mean = raw.mean(dim=0, keepdim=True)
        std = raw.std(dim=0, keepdim=True).clamp_min(1e-6)
        return {"mean": mean, "std": std}

    def _apply_transform(
        self, raw: torch.Tensor, stats: Optional[Dict[str, torch.Tensor]]
    ) -> torch.Tensor:
        if self.feature_transform != "none" and stats is not None:
            raw = raw - stats["mean"]
            if self.feature_transform == "standardize":
                raw = raw / stats["std"]
        return F.normalize(raw, dim=1)

    @torch.no_grad()
    def refresh_features(self, model: nn.Module, device: torch.device, batch_size: int = 256):
        if not self.images:
            return
        was_training = model.training
        model.eval()
        shallow_parts = []
        semantic_parts = []
        for start in range(0, len(self.images), batch_size):
            batch = torch.stack(self.images[start : start + batch_size]).to(device)
            _, shallow, semantic = model(batch, return_features=True)
            shallow_parts.append(shallow.cpu())
            semantic_parts.append(semantic.cpu())
        raw_shallow = torch.cat(shallow_parts)
        raw_semantic = torch.cat(semantic_parts)

        # Post-ReLU pooled activations are non-negative, so raw cosine similarity
        # saturates near 1 and carries almost no ranking signal. Centering (and
        # optionally scaling) restores usable geometry before normalization.
        self.shallow_stats = self._fit_transform_stats(raw_shallow)
        self.semantic_stats = self._fit_transform_stats(raw_semantic)
        self.raw_shallow_cosine_mean = float(
            self._mean_pairwise_cosine(F.normalize(raw_shallow, dim=1))
        )
        self.raw_semantic_cosine_mean = float(
            self._mean_pairwise_cosine(F.normalize(raw_semantic, dim=1))
        )
        self.shallow_features = self._apply_transform(raw_shallow, self.shallow_stats)
        self.semantic_features = self._apply_transform(raw_semantic, self.semantic_stats)
        self.transformed_shallow_cosine_mean = float(
            self._mean_pairwise_cosine(self.shallow_features)
        )
        self.transformed_semantic_cosine_mean = float(
            self._mean_pairwise_cosine(self.semantic_features)
        )
        model.train(was_training)

    @staticmethod
    def _mean_pairwise_cosine(features: torch.Tensor, sample_limit: int = 512) -> float:
        if len(features) < 2:
            return float("nan")
        # Memory is stored in task order, so an evenly spaced stride keeps the
        # estimate representative instead of biasing it toward the earliest task.
        if len(features) > sample_limit:
            positions = torch.linspace(0, len(features) - 1, sample_limit).long()
            subset = features[positions]
        else:
            subset = features
        matrix = subset @ subset.T
        upper = torch.triu_indices(len(subset), len(subset), offset=1)
        return float(matrix[upper[0], upper[1]].mean())

    def _update_expected_counts(self, replay_size: int):
        if not self.labels:
            return
        probability = min(replay_size, len(self.labels)) / len(self.labels)
        for index in range(len(self.labels)):
            self.expected_retrievals[index] += probability

    def _coverage_floor_indices(self, count: int) -> List[int]:
        if count <= 0:
            return []
        by_class = defaultdict(list)
        for index, label in enumerate(self.labels):
            by_class[label].append(index)
        classes = sorted(by_class)
        class_order = torch.randperm(len(classes), generator=self.generator).tolist()
        selected = []
        cursor = 0
        while len(selected) < count:
            label = classes[class_order[cursor % len(classes)]]
            candidates = by_class[label]
            chosen = candidates[
                torch.randint(len(candidates), (1,), generator=self.generator).item()
            ]
            if chosen not in selected:
                selected.append(chosen)
            cursor += 1
            if cursor > count * len(classes) * 4:
                break
        return selected

    def _random_indices(self, replay_size: int) -> List[int]:
        count = min(replay_size, len(self.labels))
        return torch.randperm(len(self.labels), generator=self.generator)[:count].tolist()

    def _base_level_activation(self, global_step: int) -> torch.Tensor:
        # ACT-R optimized-learning approximation:
        #   B_i = ln(n_i / (1 - d)) - d * ln(T_i)
        # n_i counts the encoding event plus every retrieval; T_i is the age of
        # the trace. High B_i means the item is already highly accessible.
        counts = torch.tensor(self.retrieval_counts, dtype=torch.float32) + 1.0
        age = (
            torch.full((len(self.labels),), float(global_step))
            - torch.tensor(self.creation_step, dtype=torch.float32)
        ).clamp_min(1.0)
        return torch.log(counts / (1.0 - self.actr_decay)) - self.actr_decay * torch.log(age)

    def _class_overrepresentation(self) -> torch.Tensor:
        # How much each item's class has been replayed relative to a uniform
        # share, expressed per item so it can be combined with the other terms.
        labels = torch.tensor(self.labels, dtype=torch.long)
        total = float(sum(self.class_retrieval_counts.values()))
        distinct = len(set(self.labels))
        if total <= 0 or distinct == 0:
            return torch.zeros(len(self.labels), dtype=torch.float32)
        uniform_share = total / distinct
        per_class = torch.tensor(
            [self.class_retrieval_counts.get(int(label), 0) for label in labels],
            dtype=torch.float32,
        )
        return per_class / uniform_share

    @staticmethod
    def _zscore(values: torch.Tensor) -> torch.Tensor:
        return (values - values.mean()) / values.std().clamp_min(1e-6)

    def _sample_with_class_cap(
        self,
        available_scores: torch.Tensor,
        available_indices: torch.Tensor,
        remaining: int,
    ) -> List[int]:
        # Draw items one at a time and retire a class once it reaches the cap.
        # This raises per-batch class diversity without discarding the ranking,
        # so cue relevance still decides which item represents each class.
        available_labels = torch.tensor(
            [self.labels[int(index)] for index in available_indices], dtype=torch.long
        )
        active = torch.ones(len(available_indices), dtype=torch.bool)
        class_counts = Counter()
        chosen: List[int] = []
        for _ in range(remaining):
            if not bool(active.any()):
                break
            masked = available_scores.masked_fill(~active, float("-inf"))
            probabilities = torch.softmax(masked / max(self.temperature, 1e-8), dim=0)
            # A sharply peaked softmax can underflow to all-zero in float32,
            # which makes multinomial raise. Keep active candidates sampleable.
            probabilities = torch.where(
                active,
                probabilities.clamp_min(1e-12),
                torch.zeros_like(probabilities),
            )
            position = int(
                torch.multinomial(probabilities, 1, generator=self.generator).item()
            )
            chosen.append(int(available_indices[position]))
            active[position] = False
            label = int(available_labels[position])
            class_counts[label] += 1
            if class_counts[label] >= self.max_per_class:
                active &= available_labels != label
        return chosen

    @torch.no_grad()
    def select(
        self,
        condition: str,
        query_images: torch.Tensor,
        model: nn.Module,
        replay_size: int,
        device: torch.device,
        task_id: int,
        global_step: int,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        count = min(replay_size, len(self.labels))
        if count == 0:
            raise RuntimeError("Replay selection requested from an empty memory.")

        self._update_expected_counts(count)
        query_selected_similarity = float("nan")
        score_std = float("nan")
        softmax_effective_fraction = float("nan")

        if condition == "random":
            selected = self._random_indices(count)
            diagnostic_features = self.semantic_features
        else:
            if condition == "shallow":
                feature_name = "shallow"
            elif condition == "semantic":
                feature_name = "semantic"
            else:
                feature_name = self.actr_feature
            features = (
                self.shallow_features if feature_name == "shallow" else self.semantic_features
            )
            stats = (
                self.shallow_stats if feature_name == "shallow" else self.semantic_stats
            )
            if features is None:
                raise RuntimeError("Memory features must be refreshed before similarity replay.")

            was_training = model.training
            model.eval()
            _, query_shallow, query_semantic = model(
                query_images.to(device), return_features=True
            )
            model.train(was_training)
            query_raw = (query_shallow if feature_name == "shallow" else query_semantic).cpu()
            query_features = self._apply_transform(query_raw, stats)

            # Max-over-query implements cue-driven retrieval without collapsing
            # a heterogeneous current batch into a potentially meaningless centroid.
            similarities = query_features @ features.T
            scores = similarities.max(dim=0).values

            if condition == "actr":
                # Cue relevance alone concentrates replay on a small subset. The
                # base-level and overrepresentation terms restore coverage by
                # discounting traces that are already accessible or already
                # over-replayed at the class level.
                scores = (
                    self.actr_alpha * self._zscore(scores)
                    + self.actr_beta * self._zscore(self._base_level_activation(global_step))
                    - self.actr_gamma * self._zscore(self._class_overrepresentation())
                )
                if self.actr_noise > 0:
                    uniform = torch.rand(
                        len(scores), generator=self.generator, dtype=torch.float32
                    ).clamp(1e-6, 1 - 1e-6)
                    scores = scores + self.actr_noise * torch.log(uniform / (1 - uniform))

            score_std = float(scores.std())
            floor_count = min(count, int(round(count * self.floor_frac)))
            floor_indices = self._coverage_floor_indices(floor_count)
            available_mask = torch.ones(len(self.labels), dtype=torch.bool)
            if floor_indices:
                available_mask[floor_indices] = False
            remaining = count - len(floor_indices)
            available_indices = torch.where(available_mask)[0]
            if remaining:
                available_scores = scores[available_indices]
                # Absolute cosine ranges differ wildly between representations, so a
                # fixed temperature is not comparable across conditions. Z-scoring
                # makes the temperature control selectivity, not feature scale.
                if self.score_normalization == "zscore":
                    available_scores = (
                        available_scores - available_scores.mean()
                    ) / available_scores.std().clamp_min(1e-6)
                probabilities = torch.softmax(
                    available_scores / max(self.temperature, 1e-8), dim=0
                )
                softmax_effective_fraction = float(
                    torch.exp(
                        -(probabilities * torch.log(probabilities.clamp_min(1e-12))).sum()
                    )
                    / len(probabilities)
                )
                if self.max_per_class > 0:
                    similarity_indices = self._sample_with_class_cap(
                        available_scores, available_indices, remaining
                    )
                else:
                    sampled_positions = torch.multinomial(
                        probabilities.clamp_min(1e-12),
                        remaining,
                        replacement=False,
                        generator=self.generator,
                    )
                    similarity_indices = available_indices[sampled_positions].tolist()
            else:
                similarity_indices = []
            selected = floor_indices + similarity_indices
            query_selected_similarity = float(similarities[:, selected].max(dim=0).values.mean())
            diagnostic_features = features

        for index in selected:
            self.retrieval_counts[index] += 1
            self.last_retrieved_step[index] = global_step
            self.class_retrieval_counts[self.labels[index]] += 1

        selected_labels = [self.labels[index] for index in selected]
        label_counts = Counter(selected_labels)
        pair_similarity = float("nan")
        if diagnostic_features is not None and len(selected) > 1:
            chosen_features = F.normalize(diagnostic_features[selected], dim=1)
            similarity_matrix = chosen_features @ chosen_features.T
            upper = torch.triu_indices(len(selected), len(selected), offset=1)
            pair_similarity = float(similarity_matrix[upper[0], upper[1]].mean())

        self.events.append(
            ReplayEvent(
                task=task_id,
                step=global_step,
                condition=condition,
                memory_size=len(self.labels),
                selected=len(selected),
                query_selected_similarity=query_selected_similarity,
                selected_pair_similarity=pair_similarity,
                selected_class_entropy=normalized_entropy(list(label_counts.values())),
                selected_unique_classes=len(label_counts),
                score_std=score_std,
                softmax_effective_fraction=softmax_effective_fraction,
            )
        )
        images = torch.stack([self.images[index] for index in selected]).to(device)
        labels = torch.tensor(
            [self.labels[index] for index in selected], dtype=torch.long, device=device
        )
        return images, labels

    def record_gradient_diagnostics(
        self,
        current_loss: torch.Tensor,
        replay_loss: torch.Tensor,
        model: nn.Module,
    ):
        parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
        current_gradients = torch.autograd.grad(
            current_loss, parameters, retain_graph=True, allow_unused=True
        )
        replay_gradients = torch.autograd.grad(
            replay_loss, parameters, retain_graph=True, allow_unused=True
        )
        current_parts = []
        replay_parts = []
        for current, replay, parameter in zip(
            current_gradients, replay_gradients, parameters
        ):
            current_parts.append(
                current.detach().flatten()
                if current is not None
                else torch.zeros_like(parameter).flatten()
            )
            replay_parts.append(
                replay.detach().flatten()
                if replay is not None
                else torch.zeros_like(parameter).flatten()
            )
        current_vector = torch.cat(current_parts)
        replay_vector = torch.cat(replay_parts)
        current_norm = current_vector.norm()
        replay_norm = replay_vector.norm()
        cosine = F.cosine_similarity(current_vector, replay_vector, dim=0)
        self.gradient_cosines.append(float(cosine))
        self.gradient_current_norms.append(float(current_norm))
        self.gradient_replay_norms.append(float(replay_norm))

    def diagnostics(self, eligible_classes: Sequence[int]) -> Dict:
        eligible = set(eligible_classes)
        observed_by_class = Counter()
        expected_by_class = defaultdict(float)
        item_counts = []
        retrieved_items = 0
        eligible_items = 0
        task_counts = Counter()

        for index, label in enumerate(self.labels):
            if label not in eligible:
                continue
            observed = self.retrieval_counts[index]
            observed_by_class[label] += observed
            expected_by_class[label] += self.expected_retrievals[index]
            item_counts.append(observed)
            eligible_items += 1
            retrieved_items += int(observed > 0)
            task_counts[self.task_ids[index]] += observed

        classes = sorted(eligible)
        observed = np.asarray([observed_by_class[c] for c in classes], dtype=np.float64)
        expected = np.asarray([expected_by_class[c] for c in classes], dtype=np.float64)
        ratio = np.divide(
            observed,
            expected,
            out=np.zeros_like(observed),
            where=expected > 0,
        )
        nonzero_expected = expected > 0
        zero_classes = [
            class_id
            for class_id, obs, exp in zip(classes, observed, expected)
            if exp > 0 and obs == 0
        ]
        ranked_classes = sorted(
            ((class_id, int(observed_by_class[class_id])) for class_id in classes),
            key=lambda item: (item[1], item[0]),
        )
        event_values = {
            field: [
                getattr(event, field)
                for event in self.events
                if not math.isnan(getattr(event, field))
            ]
            for field in (
                "query_selected_similarity",
                "selected_pair_similarity",
                "selected_class_entropy",
                "selected_unique_classes",
                "score_std",
                "softmax_effective_fraction",
            )
        }

        def mean_or_nan(values):
            return float(np.mean(values)) if values else float("nan")

        return {
            "eligible_classes": classes,
            "zero_retrieval_classes": zero_classes,
            "zero_retrieval_class_count": len(zero_classes),
            "least_retrieved_classes": [
                {"class": class_id, "count": count}
                for class_id, count in ranked_classes[:10]
            ],
            "most_retrieved_classes": [
                {"class": class_id, "count": count}
                for class_id, count in reversed(ranked_classes[-10:])
            ],
            "class_retrieval_counts": {
                str(class_id): int(observed_by_class[class_id]) for class_id in classes
            },
            "class_expected_uniform_counts": {
                str(class_id): round(float(expected_by_class[class_id]), 4)
                for class_id in classes
            },
            "class_observed_expected_ratio": {
                str(class_id): round(float(value), 4)
                for class_id, value in zip(classes, ratio)
            },
            "retrieval_count_mean": float(observed.mean()) if len(observed) else 0.0,
            "retrieval_count_std": float(observed.std()) if len(observed) else 0.0,
            "retrieval_count_cv": float(observed.std() / observed.mean())
            if len(observed) and observed.mean() > 0
            else 0.0,
            "retrieval_count_gini": gini(observed),
            "exposure_normalized_ratio_cv": float(ratio[nonzero_expected].std())
            if nonzero_expected.any()
            else 0.0,
            "observed_vs_exposure_expected_jsd": distribution_jsd(observed, expected),
            "sample_coverage_fraction": retrieved_items / max(eligible_items, 1),
            "sample_retrieval_gini": gini(item_counts),
            "retrievals_by_source_task": {
                str(task): int(count) for task, count in sorted(task_counts.items())
            },
            "mean_query_selected_similarity": mean_or_nan(
                event_values["query_selected_similarity"]
            ),
            "mean_selected_pair_similarity": mean_or_nan(
                event_values["selected_pair_similarity"]
            ),
            "mean_batch_class_entropy": mean_or_nan(
                event_values["selected_class_entropy"]
            ),
            "mean_unique_classes_per_replay": mean_or_nan(
                event_values["selected_unique_classes"]
            ),
            "mean_score_std": mean_or_nan(event_values["score_std"]),
            "mean_softmax_effective_fraction": mean_or_nan(
                event_values["softmax_effective_fraction"]
            ),
            "raw_shallow_cosine_mean": self.raw_shallow_cosine_mean,
            "raw_semantic_cosine_mean": self.raw_semantic_cosine_mean,
            "transformed_shallow_cosine_mean": self.transformed_shallow_cosine_mean,
            "transformed_semantic_cosine_mean": self.transformed_semantic_cosine_mean,
            "gradient_cosine_mean": mean_or_nan(self.gradient_cosines),
            "gradient_cosine_std": float(np.std(self.gradient_cosines))
            if self.gradient_cosines
            else float("nan"),
            "gradient_conflict_fraction": float(
                np.mean(np.asarray(self.gradient_cosines) < 0)
            )
            if self.gradient_cosines
            else float("nan"),
            "gradient_current_norm_mean": mean_or_nan(self.gradient_current_norms),
            "gradient_replay_norm_mean": mean_or_nan(self.gradient_replay_norms),
        }


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_worker_init(seed: int):
    def initialize(worker_id: int):
        worker_seed = seed + worker_id
        random.seed(worker_seed)
        np.random.seed(worker_seed)
        torch.manual_seed(worker_seed)

    return initialize


def build_tasks(
    dataset_name: str,
    root: str,
    classes_per_task: int,
    download: bool,
):
    dataset_class = datasets.CIFAR100 if dataset_name == "cifar100" else datasets.CIFAR10
    num_classes = 100 if dataset_name == "cifar100" else 10
    if num_classes % classes_per_task:
        raise ValueError("classes_per_task must divide the dataset's class count.")

    train_base = dataset_class(root=root, train=True, transform=None, download=download)
    test_base = dataset_class(root=root, train=False, transform=None, download=download)
    mean = (0.5071, 0.4867, 0.4408) if dataset_name == "cifar100" else (0.4914, 0.4822, 0.4465)
    std = (0.2675, 0.2565, 0.2761) if dataset_name == "cifar100" else (0.2470, 0.2435, 0.2616)
    train_transform = transforms.Compose(
        [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    )
    evaluation_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean, std)]
    )

    train_targets = np.asarray(train_base.targets)
    test_targets = np.asarray(test_base.targets)
    train_tasks = []
    memory_tasks = []
    test_tasks = []
    for start in range(0, num_classes, classes_per_task):
        classes = np.arange(start, start + classes_per_task)
        train_indices = np.flatnonzero(np.isin(train_targets, classes)).tolist()
        test_indices = np.flatnonzero(np.isin(test_targets, classes)).tolist()
        train_tasks.append(TaskDataset(train_base, train_indices, train_transform))
        memory_tasks.append(TaskDataset(train_base, train_indices, evaluation_transform))
        test_tasks.append(TaskDataset(test_base, test_indices, evaluation_transform))
    return train_tasks, memory_tasks, test_tasks, num_classes


@torch.no_grad()
def evaluate(model, datasets_by_task, device, batch_size, workers):
    model.eval()
    accuracies = []
    for dataset in datasets_by_task:
        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=workers,
            pin_memory=device.type == "cuda",
        )
        correct = 0
        total = 0
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            predictions = model(images).argmax(dim=1)
            correct += int((predictions == labels).sum())
            total += labels.numel()
        accuracies.append(correct / max(total, 1))
    return accuracies


@torch.no_grad()
def feature_quality(memory: ReplayMemory) -> Dict[str, Dict[str, float]]:
    labels = torch.tensor(memory.labels)
    result = {}
    for name, features in (
        ("shallow", memory.shallow_features),
        ("semantic", memory.semantic_features),
    ):
        if features is None or len(features) < 2:
            continue
        normalized = F.normalize(features, dim=1)
        classes = labels.unique(sorted=True)
        centroids = torch.stack(
            [F.normalize(normalized[labels == class_id].mean(0), dim=0) for class_id in classes]
        )
        centroid_predictions = classes[(normalized @ centroids.T).argmax(dim=1)]
        centroid_accuracy = float((centroid_predictions == labels).float().mean())

        within_values = []
        between_values = []
        for class_id in classes:
            mask = labels == class_id
            class_features = normalized[mask]
            centroid = F.normalize(class_features.mean(0), dim=0)
            within_values.append(float((class_features @ centroid).mean()))
        if len(centroids) > 1:
            matrix = centroids @ centroids.T
            upper = torch.triu_indices(len(centroids), len(centroids), offset=1)
            between_values = matrix[upper[0], upper[1]].tolist()
        mean_within = float(np.mean(within_values))
        mean_between = float(np.mean(between_values)) if between_values else 0.0
        result[name] = {
            "nearest_centroid_accuracy": centroid_accuracy,
            "mean_within_class_cosine": mean_within,
            "mean_between_class_centroid_cosine": mean_between,
            "separation_margin": mean_within - mean_between,
        }
    return result


def backward_transfer(accuracy_matrix: Sequence[Sequence[float]]) -> float:
    if len(accuracy_matrix) <= 1:
        return 0.0
    final_row = accuracy_matrix[-1]
    differences = [
        final_row[task] - accuracy_matrix[task][task]
        for task in range(len(accuracy_matrix) - 1)
    ]
    return float(np.mean(differences))


def run_condition(args, condition: str, seed: int, device: torch.device):
    seed_everything(seed)
    if condition == "actr":
        print(
            f"  config: feature={args.actr_feature} alpha={args.actr_alpha} "
            f"beta={args.actr_beta} gamma={args.actr_gamma} decay={args.actr_decay} "
            f"noise={args.actr_noise} temp={args.temperature} floor={args.floor_frac} "
            f"max_per_class={args.max_per_class}"
        )
    else:
        print(
            f"  config: temp={args.temperature} floor={args.floor_frac} "
            f"transform={args.feature_transform} norm={args.score_normalization} "
            f"max_per_class={args.max_per_class}"
        )
    train_tasks, memory_tasks, test_tasks, num_classes = build_tasks(
        args.dataset, args.data_root, args.classes_per_task, not args.no_download
    )
    model = build_model(args.arch, num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    memory = ReplayMemory(
        per_class_capacity=args.memory_per_class,
        temperature=args.temperature,
        floor_frac=args.floor_frac,
        seed=seed + 10_000,
        feature_transform=args.feature_transform,
        score_normalization=args.score_normalization,
        actr_feature=args.actr_feature,
        actr_alpha=args.actr_alpha,
        actr_beta=args.actr_beta,
        actr_gamma=args.actr_gamma,
        actr_decay=args.actr_decay,
        actr_noise=args.actr_noise,
        max_per_class=args.max_per_class,
    )
    accuracy_matrix = []
    quality_by_task = {}
    global_step = 0

    for task_id, train_dataset in enumerate(train_tasks):
        if not args.no_reset_optimizer and task_id > 0:
            optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
        if len(memory) and (
            memory.shallow_features is None
            or task_id % max(args.refresh_every, 1) == 0
        ):
            memory.refresh_features(model, device)

        loader_generator = torch.Generator().manual_seed(seed * 1000 + task_id)
        train_loader = DataLoader(
            train_dataset,
            batch_size=args.batch_size,
            shuffle=True,
            num_workers=args.workers,
            pin_memory=device.type == "cuda",
            generator=loader_generator,
            worker_init_fn=make_worker_init(seed * 1000 + task_id),
        )
        model.train()
        for epoch in range(args.epochs):
            running_loss = 0.0
            running_current_loss = 0.0
            running_replay_loss = 0.0
            for images, labels in train_loader:
                global_step += 1
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                current_logits = model(images)
                current_loss = F.cross_entropy(current_logits, labels)
                loss = current_loss
                replay_loss_value = 0.0

                if len(memory):
                    replay_images, replay_labels = memory.select(
                        condition=condition,
                        query_images=images,
                        model=model,
                        replay_size=args.replay_batch_size,
                        device=device,
                        task_id=task_id,
                        global_step=global_step,
                    )
                    replay_logits = model(replay_images)
                    replay_loss = F.cross_entropy(replay_logits, replay_labels)
                    if (
                        args.gradient_diagnostics_every > 0
                        and global_step % args.gradient_diagnostics_every == 0
                    ):
                        memory.record_gradient_diagnostics(
                            current_loss, replay_loss, model
                        )
                    loss = current_loss + args.replay_weight * replay_loss
                    replay_loss_value = float(replay_loss.detach())

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
                optimizer.step()
                running_loss += float(loss.detach())
                running_current_loss += float(current_loss.detach())
                running_replay_loss += replay_loss_value

            batches = max(len(train_loader), 1)
            print(
                f"  [{condition}] task {task_id} epoch {epoch}: "
                f"total={running_loss / batches:.4f} "
                f"current={running_current_loss / batches:.4f} "
                f"replay={running_replay_loss / batches:.4f}"
            )

        # The current task enters memory only after its own training. Consequently,
        # the final task is correctly excluded from retrieval-opportunity analysis.
        memory.add_task_data(memory_tasks[task_id], task_id, current_step=global_step)
        memory.refresh_features(model, device)
        quality_by_task[str(task_id)] = feature_quality(memory)
        accuracies = evaluate(
            model,
            test_tasks[: task_id + 1],
            device,
            args.eval_batch_size,
            args.workers,
        )
        accuracy_matrix.append(accuracies)
        formatted = ", ".join(f"T{i}={value:.3f}" for i, value in enumerate(accuracies))
        print(
            f"[{condition}] after task {task_id}: {formatted} "
            f"| learned_current={accuracies[task_id]:.3f}"
        )

    eligible_classes = list(range(num_classes - args.classes_per_task))
    diagnostics = memory.diagnostics(eligible_classes)
    diagonal = [accuracy_matrix[task][task] for task in range(len(accuracy_matrix))]
    dead_tasks = [task for task, value in enumerate(diagonal) if value < args.plasticity_floor]
    # BWT compares final accuracy against the diagonal. If the diagonal is ~0 the
    # model never learned the task, so BWT trends toward 0 and looks like "less
    # forgetting" when it actually reflects a failure to learn.
    bwt_interpretable = len(dead_tasks) == 0
    health = {
        "diagonal_accuracy": diagonal,
        "mean_diagonal_accuracy": float(np.mean(diagonal)),
        "tasks_below_plasticity_floor": dead_tasks,
        "plasticity_floor": args.plasticity_floor,
        "bwt_interpretable": bwt_interpretable,
        "optimizer_steps_per_task": int(
            math.ceil(len(train_tasks[0]) / args.batch_size) * args.epochs
        ),
    }
    result = {
        "condition": condition,
        "seed": seed,
        "dataset": args.dataset,
        "avg_accuracy": float(np.mean(accuracy_matrix[-1])),
        "bwt": backward_transfer(accuracy_matrix),
        "accuracy_matrix": accuracy_matrix,
        "run_health": health,
        "feature_quality_by_task": quality_by_task,
        "retrieval_diagnostics": diagnostics,
        "config": vars(args),
    }
    print(
        f"[{condition} | seed {seed}] avg_acc={result['avg_accuracy']:.3f} "
        f"BWT={result['bwt']:.3f} "
        f"mean_diag={health['mean_diagonal_accuracy']:.3f}"
    )
    if not bwt_interpretable:
        print(
            f"[{condition}] WARNING: tasks {dead_tasks} never learned "
            f"(accuracy < {args.plasticity_floor}). BWT is NOT interpretable for this run."
        )
    if not args.quiet_diagnostics:
        print(f"[{condition}] RETRIEVAL DIAGNOSTICS")
        print(json.dumps(diagnostics, indent=2, allow_nan=True))
    else:
        print(
            f"[{condition}] coverage="
            f"{diagnostics['sample_coverage_fraction']:.3f} "
            f"unique_classes/batch={diagnostics['mean_unique_classes_per_replay']:.1f} "
            f"sample_gini={diagnostics['sample_retrieval_gini']:.3f}"
        )
    return result, memory.events


def paired_comparison(results: List[Dict], baseline: str = "random") -> Dict:
    by_condition = defaultdict(dict)
    for result in results:
        by_condition[result["condition"]][result["seed"]] = result

    if baseline not in by_condition:
        return {
            "status": "no_baseline",
            "note": (
                f"No '{baseline}' condition in this run, so no comparison is possible. "
                "Comparing against a baseline from a different invocation is not valid "
                "unless the seeds and settings match exactly."
            ),
        }

    comparisons = {}
    for condition, seed_map in by_condition.items():
        if condition == baseline:
            continue
        shared = sorted(set(seed_map) & set(by_condition[baseline]))
        if not shared:
            continue
        entry = {"paired_seeds": shared, "n": len(shared)}
        for metric in ("avg_accuracy", "bwt", "mean_diagonal_accuracy"):
            if metric == "mean_diagonal_accuracy":
                treatment = np.asarray(
                    [seed_map[s]["run_health"][metric] for s in shared], dtype=np.float64
                )
                control = np.asarray(
                    [by_condition[baseline][s]["run_health"][metric] for s in shared],
                    dtype=np.float64,
                )
            else:
                treatment = np.asarray(
                    [seed_map[s][metric] for s in shared], dtype=np.float64
                )
                control = np.asarray(
                    [by_condition[baseline][s][metric] for s in shared], dtype=np.float64
                )
            differences = treatment - control
            mean_difference = float(differences.mean())
            # Seeds are matched (identical init and data order per seed), so a
            # paired test is both valid and far more sensitive than comparing
            # independent means.
            if len(differences) > 1 and differences.std(ddof=1) > 0:
                standard_error = differences.std(ddof=1) / math.sqrt(len(differences))
                t_statistic = mean_difference / standard_error
                cohens_d = mean_difference / differences.std(ddof=1)
            else:
                t_statistic = float("nan")
                cohens_d = float("nan")
            entry[metric] = {
                f"{baseline}_mean": float(control.mean()),
                f"{condition}_mean": float(treatment.mean()),
                "mean_difference": mean_difference,
                "per_seed_differences": [round(float(d), 4) for d in differences],
                "seeds_favoring_treatment": int((differences > 0).sum()),
                "paired_t": float(t_statistic),
                "cohens_d": float(cohens_d),
                "significant_at_05": bool(
                    np.isfinite(t_statistic)
                    and abs(t_statistic) > t_critical(len(differences) - 1)
                ),
            }
        comparisons[condition] = entry

    for condition, entry in comparisons.items():
        bwt_gain = entry["bwt"]["mean_difference"]
        diagonal_drop = -entry["mean_diagonal_accuracy"]["mean_difference"]
        # BWT is measured against the diagonal, so learning less mechanically
        # inflates it. If the diagonal fell by as much as BWT rose, the apparent
        # "less forgetting" is really reduced plasticity.
        confounded = bwt_gain > 0 and diagonal_drop > 0.5 * bwt_gain
        entry["plasticity_confound"] = {
            "bwt_gain": bwt_gain,
            "diagonal_drop": diagonal_drop,
            "bwt_gain_explained_by_diagonal_drop": confounded,
            "note": (
                "BWT improvement is likely an artifact of learning less, not of "
                "forgetting less. Report mean_diagonal_accuracy alongside BWT."
                if confounded
                else "BWT gain is not explained by a drop in plasticity."
            ),
        }

    return {"status": "compared", "baseline": baseline, "comparisons": comparisons}


# Two-tailed t critical values at alpha=0.05, indexed by degrees of freedom.
TWO_TAILED_T_CRITICAL = {
    1: 12.706,
    2: 4.303,
    3: 3.182,
    4: 2.776,
    5: 2.571,
    6: 2.447,
    7: 2.365,
    8: 2.306,
    9: 2.262,
    10: 2.228,
    14: 2.145,
    19: 2.093,
}


def t_critical(degrees_of_freedom: int) -> float:
    if degrees_of_freedom < 1:
        return float("inf")
    # Fall back to the nearest tabulated df at or below the requested one, which
    # keeps the threshold conservative rather than anti-conservative.
    available = [df for df in TWO_TAILED_T_CRITICAL if df <= degrees_of_freedom]
    if not available:
        return float("inf")
    return TWO_TAILED_T_CRITICAL[max(available)]


def write_outputs(output_dir: Path, results: List[Dict], all_events: List[Dict]):
    output_dir.mkdir(parents=True, exist_ok=True)
    with (output_dir / "results.json").open("w", encoding="utf-8") as handle:
        json.dump(results, handle, indent=2, allow_nan=True)

    if all_events:
        with (output_dir / "replay_events.csv").open(
            "w", newline="", encoding="utf-8"
        ) as handle:
            writer = csv.DictWriter(handle, fieldnames=list(all_events[0]))
            writer.writeheader()
            writer.writerows(all_events)

    summary = []
    for condition in sorted({result["condition"] for result in results}):
        condition_results = [
            result for result in results if result["condition"] == condition
        ]
        accuracy = np.asarray([result["avg_accuracy"] for result in condition_results])
        bwt = np.asarray([result["bwt"] for result in condition_results])
        diagnostic_keys = (
            "retrieval_count_cv",
            "retrieval_count_gini",
            "exposure_normalized_ratio_cv",
            "observed_vs_exposure_expected_jsd",
            "sample_coverage_fraction",
            "sample_retrieval_gini",
            "mean_selected_pair_similarity",
            "mean_batch_class_entropy",
            "mean_unique_classes_per_replay",
            "mean_score_std",
            "mean_softmax_effective_fraction",
            "gradient_cosine_mean",
            "gradient_conflict_fraction",
        )
        row = {
            "condition": condition,
            "seeds": len(condition_results),
            "avg_accuracy_mean": float(accuracy.mean()),
            "avg_accuracy_std": float(accuracy.std()),
            "bwt_mean": float(bwt.mean()),
            "bwt_std": float(bwt.std()),
            "mean_diagonal_accuracy": float(
                np.mean(
                    [
                        result["run_health"]["mean_diagonal_accuracy"]
                        for result in condition_results
                    ]
                )
            ),
            "runs_with_interpretable_bwt": int(
                sum(
                    1
                    for result in condition_results
                    if result["run_health"]["bwt_interpretable"]
                )
            ),
        }
        for key in diagnostic_keys:
            values = np.asarray(
                [
                    result["retrieval_diagnostics"][key]
                    for result in condition_results
                ],
                dtype=np.float64,
            )
            # Random replay computes no similarity scores, so score-based metrics
            # are legitimately all-NaN for that condition.
            if values.size == 0 or np.all(np.isnan(values)):
                row[f"{key}_mean"] = float("nan")
                row[f"{key}_std"] = float("nan")
            else:
                row[f"{key}_mean"] = float(np.nanmean(values))
                row[f"{key}_std"] = float(np.nanstd(values))
        summary.append(row)

    with (output_dir / "summary.csv").open(
        "w", newline="", encoding="utf-8"
    ) as handle:
        writer = csv.DictWriter(handle, fieldnames=list(summary[0]))
        writer.writeheader()
        writer.writerows(summary)

    summary_by_condition = {row["condition"]: row for row in summary}

    print("\n===== Summary (mean +/- std over seeds) =====")
    for row in summary:
        flag = "" if row["runs_with_interpretable_bwt"] == row["seeds"] else "  [BWT UNRELIABLE]"
        print(
            f"{row['condition']:10s} "
            f"avg_acc={row['avg_accuracy_mean']:.3f} +/- {row['avg_accuracy_std']:.3f}  "
            f"BWT={row['bwt_mean']:.3f} +/- {row['bwt_std']:.3f}  "
            f"mean_diag={row['mean_diagonal_accuracy']:.3f}  "
            f"exposure-JSD={row['observed_vs_exposure_expected_jsd_mean']:.4f}  "
            f"sample-coverage={row['sample_coverage_fraction_mean']:.3f}  "
            f"grad-conflict={row['gradient_conflict_fraction_mean']:.3f}{flag}"
        )

    diagnosis = {
        "status": "insufficient_conditions",
        "interpretation": [
            "Run both random and shallow conditions to generate a direct diagnosis."
        ],
    }
    if "random" in summary_by_condition and "shallow" in summary_by_condition:
        random_row = summary_by_condition["random"]
        shallow_row = summary_by_condition["shallow"]
        findings = []

        degenerate = [
            row["condition"]
            for row in summary
            if row["runs_with_interpretable_bwt"] < row["seeds"]
        ]
        if degenerate:
            diagnosis = {
                "status": "invalid_run",
                "degenerate_conditions": degenerate,
                "interpretation": [
                    "One or more conditions failed to learn some tasks, so their "
                    "diagonal accuracy is ~0 and BWT is driven by lack of plasticity "
                    "rather than by forgetting.",
                    "Fix plasticity first (smaller --batch-size, more --epochs, or a "
                    "lower --replay-weight), then compare retrieval conditions.",
                ],
            }
            with (output_dir / "diagnosis.json").open("w", encoding="utf-8") as handle:
                json.dump(diagnosis, handle, indent=2, allow_nan=True)
            print("\n===== Diagnosis =====")
            print(json.dumps(diagnosis, indent=2))
            return

        random_jsd = random_row["observed_vs_exposure_expected_jsd_mean"]
        shallow_jsd = shallow_row["observed_vs_exposure_expected_jsd_mean"]
        random_coverage = random_row["sample_coverage_fraction_mean"]
        shallow_coverage = shallow_row["sample_coverage_fraction_mean"]
        if shallow_jsd > random_jsd * 1.25 and shallow_coverage < random_coverage - 0.05:
            findings.append(
                {
                    "mechanism": "coverage_bias",
                    "supported": True,
                    "reason": (
                        "Shallow retrieval departs more from its exposure-adjusted "
                        "uniform baseline and reaches fewer stored samples than random."
                    ),
                }
            )
        else:
            findings.append(
                {
                    "mechanism": "coverage_bias",
                    "supported": False,
                    "reason": (
                        "Shallow retrieval does not show both a materially larger "
                        "exposure-adjusted divergence and lower sample coverage."
                    ),
                }
            )

        random_pair = random_row["mean_selected_pair_similarity_mean"]
        shallow_pair = shallow_row["mean_selected_pair_similarity_mean"]
        findings.append(
            {
                "mechanism": "redundant_similarity_replay",
                "supported": bool(
                    np.isfinite(random_pair)
                    and np.isfinite(shallow_pair)
                    and shallow_pair > random_pair + 0.05
                ),
                "reason": (
                    "Selected shallow batches are more internally similar than random."
                    if np.isfinite(random_pair)
                    and np.isfinite(shallow_pair)
                    and shallow_pair > random_pair + 0.05
                    else "Shallow batches are not materially more redundant than random."
                ),
            }
        )

        random_conflict = random_row["gradient_conflict_fraction_mean"]
        shallow_conflict = shallow_row["gradient_conflict_fraction_mean"]
        findings.append(
            {
                "mechanism": "gradient_interference",
                "supported": bool(
                    np.isfinite(random_conflict)
                    and np.isfinite(shallow_conflict)
                    and shallow_conflict > random_conflict + 0.05
                ),
                "reason": (
                    "Shallow replay gradients conflict with current-task gradients more "
                    "often than random replay gradients."
                    if np.isfinite(random_conflict)
                    and np.isfinite(shallow_conflict)
                    and shallow_conflict > random_conflict + 0.05
                    else "Shallow replay does not show materially more gradient conflict."
                ),
            }
        )
        shallow_effective = shallow_row["mean_softmax_effective_fraction_mean"]
        semantic_effective = summary_by_condition.get("semantic", {}).get(
            "mean_softmax_effective_fraction_mean", float("nan")
        )
        findings.append(
            {
                "mechanism": "non_selective_retrieval",
                "supported": bool(
                    np.isfinite(shallow_effective) and shallow_effective > 0.5
                ),
                "shallow_softmax_effective_fraction": shallow_effective,
                "semantic_softmax_effective_fraction": semantic_effective,
                "reason": (
                    "The shallow retrieval distribution stays close to uniform, so "
                    "'shallow' is effectively random replay with extra compute."
                    if np.isfinite(shallow_effective) and shallow_effective > 0.5
                    else "Shallow retrieval is meaningfully peaked, so it is genuinely selective."
                ),
            }
        )
        diagnosis = {
            "status": "diagnosed",
            "thresholds": {
                "coverage_jsd_multiplier": 1.25,
                "sample_coverage_absolute_gap": 0.05,
                "pair_similarity_absolute_gap": 0.05,
                "gradient_conflict_absolute_gap": 0.05,
            },
            "findings": findings,
            "important_note": (
                "Raw class retrieval totals are age-confounded. Prefer the "
                "exposure-normalized JSD and observed/expected ratios when deciding "
                "whether similarity replay is biased."
            ),
        }

        if "actr" in summary_by_condition:
            actr_row = summary_by_condition["actr"]
            diagnosis["actr_intervention"] = {
                "coverage_recovered": bool(
                    actr_row["sample_coverage_fraction_mean"]
                    > shallow_row["sample_coverage_fraction_mean"] + 0.05
                ),
                "beats_shallow_on_bwt": bool(
                    actr_row["bwt_mean"] > shallow_row["bwt_mean"]
                ),
                "beats_random_on_bwt": bool(
                    actr_row["bwt_mean"] > random_row["bwt_mean"]
                ),
                "beats_random_on_accuracy": bool(
                    actr_row["avg_accuracy_mean"] > random_row["avg_accuracy_mean"]
                ),
                "sample_coverage": {
                    "random": random_row["sample_coverage_fraction_mean"],
                    "shallow": shallow_row["sample_coverage_fraction_mean"],
                    "actr": actr_row["sample_coverage_fraction_mean"],
                },
                "bwt": {
                    "random": random_row["bwt_mean"],
                    "shallow": shallow_row["bwt_mean"],
                    "actr": actr_row["bwt_mean"],
                },
                "note": (
                    "Recovering coverage without beating random means coverage was "
                    "necessary but not sufficient, which is itself a reportable result."
                ),
            }
    with (output_dir / "diagnosis.json").open("w", encoding="utf-8") as handle:
        json.dump(diagnosis, handle, indent=2, allow_nan=True)

    comparison = paired_comparison(results)
    with (output_dir / "paired_comparison.json").open("w", encoding="utf-8") as handle:
        json.dump(comparison, handle, indent=2, allow_nan=True)

    print("\n===== Paired comparison vs random (same seeds) =====")
    if comparison["status"] != "compared":
        print(comparison["note"])
    else:
        for condition, entry in comparison["comparisons"].items():
            for metric in ("avg_accuracy", "bwt", "mean_diagonal_accuracy"):
                stats = entry[metric]
                verdict = (
                    "SIGNIFICANT" if stats["significant_at_05"] else "not significant"
                )
                print(
                    f"{condition} vs random | {metric}: "
                    f"{stats['mean_difference']:+.4f} "
                    f"(n={entry['n']}, {stats['seeds_favoring_treatment']}/{entry['n']} seeds favor, "
                    f"t={stats['paired_t']:.2f}, d={stats['cohens_d']:.2f}) -> {verdict}"
                )
            if entry["plasticity_confound"]["bwt_gain_explained_by_diagonal_drop"]:
                print(
                    f"  WARNING [{condition}]: BWT gain "
                    f"({entry['plasticity_confound']['bwt_gain']:+.4f}) co-occurs with a "
                    f"diagonal drop ({entry['plasticity_confound']['diagonal_drop']:+.4f}). "
                    "This is reduced plasticity, not reduced forgetting."
                )

    print("\n===== Diagnosis =====")
    print(json.dumps(diagnosis, indent=2, allow_nan=True))


def parse_arguments():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--condition",
        choices=("random", "semantic", "shallow", "actr", "all", "compare"),
        default="all",
    )
    parser.add_argument("--dataset", choices=("cifar10", "cifar100"), default="cifar100")
    parser.add_argument(
        "--arch", choices=("small_cnn", "resnet18"), default="small_cnn"
    )
    parser.add_argument("--data-root", default="/kaggle/working/data")
    parser.add_argument("--output-dir", default="/kaggle/working/actr_diagnostics")
    parser.add_argument("--seeds", type=int, default=1)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--classes-per-task", type=int, default=10)
    parser.add_argument("--memory-per-class", type=int, default=40)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--eval-batch-size", type=int, default=256)
    parser.add_argument("--replay-batch-size", type=int, default=32)
    parser.add_argument("--replay-weight", type=float, default=1.0)
    parser.add_argument("--temperature", type=float, default=0.1)
    parser.add_argument("--floor-frac", type=float, default=0.0)
    parser.add_argument("--refresh-every", type=int, default=1)
    parser.add_argument("--gradient-diagnostics-every", type=int, default=25)
    parser.add_argument(
        "--feature-transform",
        choices=("none", "center", "standardize"),
        default="standardize",
    )
    parser.add_argument(
        "--score-normalization", choices=("none", "zscore"), default="zscore"
    )
    parser.add_argument("--plasticity-floor", type=float, default=0.05)
    parser.add_argument(
        "--actr-feature", choices=("shallow", "semantic"), default="shallow"
    )
    parser.add_argument("--actr-alpha", type=float, default=1.0)
    parser.add_argument("--actr-beta", type=float, default=-1.0)
    parser.add_argument("--actr-gamma", type=float, default=1.0)
    parser.add_argument("--actr-decay", type=float, default=0.5)
    parser.add_argument("--actr-noise", type=float, default=0.0)
    parser.add_argument(
        "--max-per-class",
        type=int,
        default=0,
        help="Cap items per class within a replay batch (0 disables the cap).",
    )
    parser.add_argument("--grad-clip", type=float, default=5.0)
    parser.add_argument("--lr", type=float, default=5e-4)
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument("--no-reset-optimizer", action="store_true")
    parser.add_argument(
        "--quiet-diagnostics",
        action="store_true",
        help="Print a one-line retrieval summary instead of the full JSON dump.",
    )
    parser.add_argument("--no-download", action="store_true")
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"Ignoring notebook arguments: {unknown}")
    if not 0.0 <= args.floor_frac <= 1.0:
        parser.error("--floor-frac must be between 0 and 1.")
    if not 0.0 < args.actr_decay < 1.0:
        parser.error("--actr-decay must be strictly between 0 and 1.")
    if args.max_per_class < 0:
        parser.error("--max-per-class must be 0 (disabled) or positive.")
    if args.seeds < 1 or args.epochs < 1:
        parser.error("--seeds and --epochs must be positive.")
    return args


def main():
    args = parse_arguments()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    if args.condition == "all":
        conditions = ["random", "semantic", "shallow"]
    elif args.condition == "compare":
        conditions = ["random", "shallow", "actr"]
    else:
        conditions = [args.condition]
    results = []
    all_events = []
    for condition in conditions:
        for seed in range(args.seeds):
            print(f"\n===== Running condition: {condition} | seed {seed} =====")
            result, events = run_condition(args, condition, seed, device)
            results.append(result)
            all_events.extend(
                {
                    "condition": condition,
                    "seed": seed,
                    **asdict(event),
                }
                for event in events
            )
    write_outputs(Path(args.output_dir), results, all_events)


if __name__ == "__main__":
    main()


Writing actr_diagnostic_experiment.py


In [2]:
# pure LFU / base-level, NO similarity  -> does ACT-R beat this?
!python actr_diagnostic_experiment.py --data-root /kaggle/working/data \
  --output-dir /kaggle/working/abl_lfu --condition actr \
  --seeds 5 --epochs 5 --quiet-diagnostics \
  --actr-alpha 0 --actr-beta -1 --actr-gamma 0

!python actr_diagnostic_experiment.py --data-root /kaggle/working/data \
  --output-dir /kaggle/working/abl_roundrobin --condition shallow \
  --seeds 5 --epochs 5 --quiet-diagnostics \
  --floor-frac 1.0

Using device: cuda

===== Running condition: actr | seed 0 =====
  config: feature=shallow alpha=0.0 beta=-1.0 gamma=0.0 decay=0.5 noise=0.0 temp=0.1 floor=0.0 max_per_class=0
100%|█████████████████████████████████████████| 169M/169M [21:26<00:00, 131kB/s]
  [actr] task 0 epoch 0: total=2.1336 current=2.1336 replay=0.0000
  [actr] task 0 epoch 1: total=1.5204 current=1.5204 replay=0.0000
  [actr] task 0 epoch 2: total=1.2464 current=1.2464 replay=0.0000
  [actr] task 0 epoch 3: total=1.1048 current=1.1048 replay=0.0000
  [actr] task 0 epoch 4: total=1.0038 current=1.0038 replay=0.0000
[actr] after task 0: T0=0.656 | learned_current=0.656
  [actr] task 1 epoch 0: total=4.1723 current=3.1696 replay=1.0026
  [actr] task 1 epoch 1: total=1.9379 current=1.5878 replay=0.3501
  [actr] task 1 epoch 2: total=1.4432 current=1.2785 replay=0.1647
  [actr] task 1 epoch 3: total=1.3034 current=1.1923 replay=0.1110
  [actr] task 1 epoch 4: total=1.1500 current=1.0716 replay=0.0784
[actr] after task 1